# Tiled matrix multiplication in shared memory

The naive matmul kernel (one thread per output, reading straight from global memory) performs only about 0.25 FLOP per byte loaded: every element of $A$ is fetched $N$ times and every element of $B$ is fetched $M$ times. **Tiling** stages square sub-blocks of $A$ and $B$ in shared memory so that each value fetched from global memory is reused by a whole row or column of threads.

## The algorithm
With tile size $T = 16$, block `(16, 16)` computes the tile $C[r_0 : r_0+16,\ c_0 : c_0+16]$, and grid $(\lceil N/16 \rceil, \lceil M/16 \rceil)$ with x → columns. The block walks along the shared dimension in $\lceil K/16 \rceil$ phases. In phase $p$:

1. Every thread loads **one** element of each tile: `As[ty, tx] = A[row, p*16 + tx]` and `Bs[ty, tx] = B[p*16 + ty, col]`, writing 0 where the index is outside the matrix.
2. `cuda.syncthreads()`: both tiles are complete.
3. Every thread accumulates `As[ty, k] * Bs[k, tx]` for `k = 0..15` into a register.
4. `cuda.syncthreads()` again: nobody overwrites the tiles in the next phase while a slower thread is still reading them.

After the last phase, threads inside the matrix write `C[row, col]`.

Write `tiled_matmul(A, B)`.

## Inputs and output
- `A`: `float32` `(M, K)`, `B`: `float32` `(K, N)`, $M, N, K \ge 1$, none of them necessarily a multiple of 16.
- Return `C = A @ B`: `float32` `(M, N)`. Compared with relative tolerance $10^{-4}$, absolute $10^{-5}$.
- Threads whose `(row, col)` lies outside `C` still take part in loading and in every barrier; they just do not write.

## Runtime
GPU in Colab; checked here on Numba's CUDA simulator.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
!pip -q install "numba-cuda[cu12]"

In [ ]:
import numpy as np
from numba import cuda, float32

TILE = 16

def tiled_matmul(A, B):
    """A: float32 (M, K), B: float32 (K, N). Returns float32 (M, N)."""
    M, N = A.shape[0], B.shape[1]
    d_A = cuda.to_device(np.ascontiguousarray(A))
    d_B = cuda.to_device(np.ascontiguousarray(B))
    d_C = cuda.device_array((M, N), dtype=np.float32)
    grid = ((N + TILE - 1) // TILE, (M + TILE - 1) // TILE)
    tiled_kernel[grid, (TILE, TILE)](d_A, d_B, d_C)
    return d_C.copy_to_host()

@cuda.jit
def tiled_kernel(A, B, C):
    As = cuda.shared.array((TILE, TILE), float32)
    Bs = cuda.shared.array((TILE, TILE), float32)
    tx, ty = cuda.threadIdx.x, cuda.threadIdx.y
    col, row = cuda.grid(2)
    # TODO: loop over phases: load one element of each tile (0 outside), barrier,
    #       accumulate 16 products from shared memory, barrier. Then write C.
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, 2.0], [3.0, 4.0]], "dtype": "float32"}, {"__nd__": [[5.0, 6.0], [7.0, 8.0]], "dtype": "float32"}], "e": {"__nd__": [[19.0, 22.0], [43.0, 50.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[0.0, 1.0, 2.0], [3.0, 4.0, 5.0]], "dtype": "float32"}, {"__nd__": [[1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0]], "dtype": "float32"}], "e": {"__nd__": [[3.0, 3.0, 3.0, 3.0], [12.0, 12.0, 12.0, 12.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[-0.8499407172203064]], "dtype": "float32"}, {"__nd__": [[0.3219408392906189]], "dtype": "float32"}], "e": {"__nd__": [[-0.27363061904907227]], "dtype": "float32"}}, {"a": [{"__nd__": [[1.7890428304672241, 0.8793392181396484, 0.37159281969070435, 1.4876937866210938, 0.7286655306816101, 2.0812487602233887, -1.3372681140899658, 0.12535154819488525, 0.5442212224006653, -0.6738274693489075, 0.0196447242051363, -1.0580850839614868, -1.3480662107467651, 1.2198259830474854, -2.400190830230713, -1.5565797090530396], [-0.4148247241973877, -1.0050698518753052, 0.9691764712333679, 0.6467503905296326, 0.26726439595222473, 1.5470319986343384, 1.1765365600585938, 1.149961233139038, 0.3800823390483856, 0.03514379262924194, 0.8768966197967529, 0.9787058234214783, 0.3679501414299011, 0.17431947588920593, 0.5349830985069275, 0.9070428013801575], [0.9889981746673584, -0.8052987456321716, -0.27967584133148193, 0.7485387921333313, 0.13229221105575562, -0.01968977600336075, 0.9568584561347961, -0.08517853170633316, 0.6330366134643555, -1.1527568101882935, 1.7274047136306763, -0.5925851464271545, -0.3513554334640503, 2.433537244796753, 1.4384379386901855, 0.4894554316997528], [-0.39600756764411926, -0.45785367488861084, -1.5886458158493042, -0.7149897813796997, -2.713034152984619, 0.39751723408699036, 0.1558631807565689, -1.0654304027557373, 0.077748142182827, -1.6295729875564575, -0.426994651556015, 0.40270864963531494, 0.999687671661377, 0.35438671708106995, -0.8256648182868958, -0.4918054938316345], [1.7518655061721802, -1.9476003646850586, 1.2085843086242676, -2.095126152038574, -0.9954356551170349, 0.7575695514678955, -0.304243803024292, -0.8655942678451538, 0.8038788437843323, 0.9376139044761658, -0.6693726778030396, 0.2547234892845154, 1.9258934259414673, -0.025105683133006096, 0.32560959458351135, 0.25389814376831055], [0.3140934407711029, 0.2291548252105713, -0.39332735538482666, 0.37320080399513245, -0.9764795899391174, -0.26677659153938293, -0.5847126245498657, -0.08473029732704163, 0.6126322746276855, -0.5569605231285095, -0.2170463502407074, 0.6572902798652649, -0.7355180382728577, 0.47954535484313965, -0.9015275835990906, 1.1244546175003052], [0.3302454352378845, -0.018226608633995056, -0.14549504220485687, -0.3325450122356415, 2.2947916984558105, -0.235556498169899, 0.8886603713035583, 0.7174938917160034, -0.16319575905799866, -0.6100091338157654, 1.1425960063934326, 1.2486717700958252, 0.8917248845100403, -1.0364011526107788, 0.37795281410217285, -0.1481398493051529], [0.6832348108291626, -0.2929338812828064, -0.9952728748321533, 0.24876007437705994, -0.9828354716300964, 1.4120832681655884, -0.03848360478878021, 0.4501429796218872, 1.1003589630126953, 2.311683416366577, 0.5091686844825745, -1.3615610599517822, -1.4412802457809448, -1.3208240270614624, 0.3246724605560303, -0.4123438894748688], [0.9813275933265686, -0.3425409197807312, -1.016233205795288, 1.3894137144088745, 0.1020779237151146, 0.6981303095817566, -0.5690324902534485, 0.4387633502483368, 1.6436294317245483, 0.053120605647563934, -0.7485217452049255, 1.6925138235092163, 1.0484529733657837, -0.6165080666542053, -1.2503536939620972, 0.3274178206920624], [1.483412265777588, 1.2549481391906738, -0.40305471420288086, 0.37050649523735046, 1.1179473400115967, 0.693655788898468, -0.7471545338630676, 0.06593707948923111, 1.4921085834503174, 1.547357439994812, -0.03838365897536278, 0.49077868461608887, 0.4745192229747772, 0.6989575028419495, -1.7655843496322632, 1.4455090761184692], [0.9101919531822205, -0.655621349811554, 1.1455248594284058, -1.1569808721542358, -1.3091374635696411, -1.3109029531478882, 0.8256599307060242, -0.27360019087791443, -1.313212513923645, 0.982581615447998, -0.008541498333215714, -1.5350995063781738, -1.0458391904830933, -1.4663206338882446, -0.2210962325334549, 0.9625136852264404], [1.0324127674102783, 0.003090734826400876, 0.7512480616569519, -1.132407546043396, -0.3918377161026001, -0.15599046647548676, 1.34119713306427, -0.08736111223697662, 2.080923557281494, -0.5856008529663086, -1.2316523790359497, -2.612003803253174, 0.07056395709514618, 0.7245405316352844, 0.1718449592590332, 0.5587630867958069], [0.5903984308242798, -0.5055981874465942, 0.16574451327323914, 1.1902532577514648, 0.543113648891449, 0.21620972454547882, -1.6701747179031372, -1.0030555725097656, -0.4276168942451477, -0.7732964158058167, -1.7737979888916016, -0.3036523759365082, 1.102129340171814, 1.0551642179489136, -0.11275582760572433, 0.6190850138664246], [1.5651705265045166, 0.20166616141796112, -0.43436285853385925, 0.5685554146766663, 1.0072858333587646, -0.14874903857707977, 1.508975863456726, 0.31231653690338135, 0.16261059045791626, 0.95042884349823, 0.26758188009262085, 0.4875609278678894, -0.635427713394165, -0.016120996326208115, -1.0364646911621094, 0.9669531583786011], [-2.693091869354248, -0.8702356219291687, 0.06065104529261589, -0.5661584734916687, -0.11268183588981628, -0.9519827365875244, 1.3248035907745361, -1.4268996715545654, 0.05228452757000923, 0.04726390168070793, 0.46916303038597107, -0.62961345911026, 0.23781391978263855, -1.2537143230438232, 0.09049451351165771, 0.524269700050354], [-1.2692475318908691, -0.3062484860420227, -0.7856562733650208, 0.6183202266693115, 1.066530466079712, 1.4278255701065063, 0.5109383463859558, -0.13950856029987335, -1.4407916069030762, -1.0019726753234863, 1.162032127380371, -0.34728339314460754, -0.5133857131004333, -0.44906744360923767, -0.8366147875785828, -0.32095637917518616]], "dtype": "float32"}, {"__nd__": [[0.16394168138504028, -0.2616344094276428, 0.9371179342269897, -1.5000444650650024, 0.14163155853748322, -1.0154742002487183, -1.2611626386642456, -2.3681204319000244, 1.807299256324768, -0.33687546849250793, -0.3718326985836029, -0.43338045477867126, -0.1518310010433197, 1.6622323989868164, 1.2756454944610596, 0.3618319630622864], [0.06599658727645874, -1.0165685415267944, 1.1400140523910522, -0.023156126961112022, 1.6594555377960205, 2.263594150543213, -0.4520311653614044, -0.0766783207654953, 1.3816825151443481, -0.11465580016374588, -0.49001970887184143, 0.7087109088897705, 0.8242701888084412, -0.6384328007698059, -0.5257792472839355, -0.01892034523189068], [0.7718528509140015, -1.2747259140014648, 0.8381787538528442, 0.40346044301986694, 0.3709240257740021, 1.5131515264511108, 1.5845320224761963, 0.624451756477356, 0.768513023853302, -1.51587975025177, 0.016178343445062637, 1.1183414459228516, 0.6507742404937744, 0.5723603367805481, 1.9779149293899536, 0.038694754242897034], [0.64752596616745, 0.6127846240997314, 0.567503035068512, 1.3566533327102661, 2.0120604038238525, -0.3465827405452728, 0.6165563464164734, 0.30905961990356445, 0.7020812034606934, 0.054927483201026917, -0.4187004566192627, 0.5076497793197632, -1.3655283451080322, -0.41974937915802, -0.801995038986206, 0.3449998199939728], [-1.73211669921875, -0.30878010392189026, 0.7043536901473999, -0.37112927436828613, -1.3143208026885986, 0.4121176302433014, 0.4377403259277344, -1.9104059934616089, 0.7233126759529114, -0.009239883162081242, 0.6645427942276001, -1.9505828619003296, 0.23832367360591888, 0.4318363070487976, 2.568899631500244, 0.9538344740867615], [1.2986085414886475, -0.28557732701301575, 1.3021422624588013, 0.6417636871337891, 1.6962080001831055, 0.25947436690330505, 0.13998371362686157, 0.765627920627594, -0.9850215315818787, -0.07953797280788422, 2.3809261322021484, -1.0634371042251587, -0.05228077620267868, -0.9123445749282837, 0.26799702644348145, 0.6688757538795471], [0.43029946088790894, 1.4143195152282715, 0.38021162152290344, -1.683354377746582, -1.3832756280899048, -0.10850349813699722, 0.18692347407341003, 1.2047457695007324, 0.07827777415513992, -1.0953638553619385, -0.4204428195953369, -0.1210707351565361, -0.7713541388511658, -0.16931448876857758, -0.5495817065238953, 0.15776725113391876], [0.5133041739463806, 1.2914295196533203, 0.12492672353982925, 0.7921124696731567, 0.8123092651367188, -0.05591285601258278, -0.1774132251739502, -1.9127016067504883, 2.262050151824951, -1.43229341506958, -0.5070472955703735, -0.7632193565368652, -0.27563780546188354, 0.8442263603210449, -0.47464480996131897, -1.2763558626174927], [0.4901164174079895, 0.8996813893318176, 1.0167169570922852, -0.5109129548072815, 1.1663581132888794, -0.5903369784355164, -1.344996690750122, 0.4333932101726532, 1.203902006149292, 0.08314340561628342, -0.6957683563232422, 0.057832021266222, -0.07309887558221817, 0.7646869421005249, -1.1370412111282349, 0.5531892776489258], [0.07465719431638718, 0.809863805770874, 0.05035270377993584, -0.9909310936927795, 0.247474804520607, 0.2419501692056656, 0.5242136120796204, -1.106783390045166, 3.2667839527130127, 0.08099879324436188, -0.14869539439678192, 1.080033779144287, -0.7418911457061768, 0.7967692017555237, -1.4692302942276, -0.47732266783714294], [0.18012367188930511, -1.1856563091278076, -2.2683956623077393, -0.9110128283500671, 0.7304198145866394, -1.1278409957885742, -0.5472608208656311, 0.43587562441825867, -0.35857900977134705, -0.05990143492817879, 1.0454415082931519, -1.2085771560668945, 0.6692017912864685, 0.0764402374625206, -0.6603066921234131, -0.6239384412765503], [0.5146128535270691, -0.7006332874298096, -1.4604825973510742, 0.11995869874954224, -2.161855697631836, -0.2893405854701996, -1.6390464305877686, 1.1797176599502563, -0.09502901136875153, -2.315101385116577, -0.47478607296943665, 0.3116859495639801, -0.47407564520835876, -1.5226455926895142, -0.7580224275588989, 0.6521077752113342], [1.0383292436599731, 0.20645298063755035, -0.7796690464019775, 1.1147143840789795, -0.8300212025642395, -1.0750665664672852, -0.5897802710533142, -0.8786720633506775, 1.5372314453125, -0.09156394749879837, 0.538083553314209, -0.04127312824130058, 0.5901638865470886, -0.33403876423835754, -1.211283802986145, -1.3390109539031982], [-1.0961309671401978, 0.24157005548477173, -0.4665023982524872, 0.9280747771263123, 0.9298528432846069, 0.6609188914299011, 0.6993440985679626, 0.39909952878952026, 1.4607837200164795, 0.7877650260925293, 0.3523459732532501, -1.2680206298828125, 0.6210770606994629, -0.32093819975852966, -0.45029929280281067, 0.7160486578941345], [-0.4133201241493225, -0.7149691581726074, -0.6470173597335815, -0.7943191528320312, -0.4841390550136566, -0.984304666519165, -0.2728728950023651, 0.19127905368804932, 0.5829073786735535, -0.8245996236801147, -1.1760706901550293, -0.13336731493473053, 1.6857001781463623, -0.8595542311668396, -1.3184574842453003, -2.280813217163086], [1.2229654788970947, 0.2886284589767456, 2.517054796218872, 0.5382810235023499, -0.8543614745140076, 0.11574617028236389, -0.1193971112370491, 0.6955367922782898, 0.5713809132575989, 0.6775768399238586, -1.055295467376709, 2.257370948791504, -0.33669373393058777, -0.6575366854667664, 0.4739692509174347, 0.830819308757782]], "dtype": "float32"}], "e": {"__nd__": [[-1.4419831037521362, -1.5277178287506104, 6.702376365661621, 3.8226966857910156, 16.88280487060547, 5.436326026916504, 2.317619800567627, -5.390141487121582, 0.4027785658836365, 4.3758344650268555, 8.63952922821045, -8.284321784973145, -2.8596816062927246, 5.386008262634277, 9.13744831085205, 9.654608726501465], [5.604761600494385, 1.5582878589630127, 0.9769482612609863, 1.4627430438995361, -0.6869345307350159, -2.481876850128174, 0.3351861238479614, 3.8983917236328125, 2.2386457920074463, -6.246790409088135, 2.0302867889404297, -1.261798620223999, -1.268764853477478, -2.628671884536743, -1.2980525493621826, -0.4127936363220215], [-2.306732654571533, 0.21435894072055817, -2.4075756072998047, -2.140329360961914, 3.0417914390563965, -5.361944675445557, -0.5874190330505371, 2.323850154876709, 1.5487619638442993, 1.6208863258361816, -0.33077043294906616, -6.859100341796875, 3.1097521781921387, -0.09699246287345886, -1.671661615371704, -0.23175153136253357], [3.3875019550323486, 1.4394311904907227, -5.368536472320557, 2.9961204528808594, -0.8601338267326355, -3.9224419593811035, -4.608432292938232, 8.362150192260742, -11.645859718322754, 3.406538724899292, 1.9669342041015625, 0.7664498090744019, -0.5582271218299866, -4.734571933746338, -7.508644104003906, 0.3594137728214264], [4.545303821563721, -0.4303041696548462, 0.2564079463481903, -2.9533114433288574, -6.3064141273498535, -5.709529876708984, -2.8521945476531982, -2.2652885913848877, 3.9903101921081543, -1.4914339780807495, 1.7658735513687134, 2.411625385284424, 2.233881711959839, 3.7669835090637207, 0.880264163017273, -1.938849687576294], [2.070796251296997, 0.6862987279891968, 3.060213088989258, 2.4415290355682373, 2.1300456523895264, 0.7551830410957336, -2.6454973220825195, 3.2372782230377197, -1.2166519165039062, 1.7242755889892578, -2.6225810050964355, 4.081635475158691, -2.491757869720459, -1.0221774578094482, -0.8521391749382019, 4.616383075714111], [-1.35733962059021, -1.7937428951263428, -3.7261884212493896, -3.4863109588623047, -8.735069274902344, -3.4674689769744873, -3.941835880279541, -4.492741584777832, 1.0857161283493042, -6.24276065826416, 0.8821287155151367, -6.135875701904297, 1.1769520044326782, -0.33944976329803467, 3.6116464138031006, -1.363810420036316], [2.6519031524658203, 4.2118916511535645, 3.3619723320007324, -6.182229518890381, 9.18196964263916, -2.970108985900879, -0.900718629360199, -3.0401389598846436, 3.785175085067749, 2.5647172927856445, 1.1479179859161377, 0.8668511509895325, -3.529053211212158, 5.015257835388184, -5.715966701507568, -2.072770595550537], [5.1910529136657715, 4.048659801483154, 3.288376569747925, 3.4849605560302734, 0.7893582582473755, -4.399657726287842, -7.172889232635498, -2.542837619781494, 4.258123874664307, -1.7974337339401245, -0.3141120970249176, 0.922217845916748, -6.129099369049072, -0.06910109519958496, -3.062868356704712, 4.449240207672119], [2.2481396198272705, 1.8057098388671875, 9.158069610595703, 0.5823484659194946, 4.593036651611328, 2.565950632095337, -3.8742001056671143, -6.351183891296387, 12.946905136108398, 2.704183340072632, 0.8582159280776978, 1.6551666259765625, -3.4349160194396973, 3.1184277534484863, 1.3820159435272217, 7.477345943450928], [1.4490888118743896, 0.4178510308265686, 3.100647211074829, -6.7316694259643555, -3.9161503314971924, 0.7230863571166992, 4.0354323387146, -1.2425122261047363, -0.9534764289855957, 0.8723252415657043, -4.061378002166748, 8.512996673583984, -1.2937873601913452, 5.825261116027832, 4.5547895431518555, -2.195946216583252], [0.3260141611099243, 4.941076755523682, 10.575079917907715, -3.8223235607147217, 3.5596303939819336, 1.2184969186782837, 1.4624818563461304, -1.333553671836853, 4.298418045043945, 4.170526504516602, -2.957246780395508, 1.133612036705017, 2.074925184249878, 6.528172969818115, 2.879474639892578, 0.8493425846099854], [-0.8830204606056213, -0.9627702236175537, 4.968175888061523, 7.956792831420898, 1.1121065616607666, 0.4604213237762451, 2.297173500061035, -2.4527289867401123, 0.2591550350189209, 4.8709235191345215, 0.7654443383216858, 1.072143316268921, 0.12820301949977875, -1.260629415512085, 4.895118713378906, 3.4122507572174072], [0.5931046009063721, 3.6987483501434326, 5.285333156585693, -5.1081366539001465, -2.116278648376465, -0.29618531465530396, -1.8912431001663208, -3.898575782775879, 6.97874641418457, -1.4573431015014648, -1.7587255239486694, 0.2719438076019287, -5.359499931335449, 3.1967434883117676, 2.619887590408325, 5.236514091491699], [-0.007598221302032471, 1.3056610822677612, -3.208483934402466, -1.877821445465088, -7.478894233703613, -0.7835769653320312, 3.480375289916992, 9.223599433898926, -9.692864418029785, 2.26222825050354, -0.7365932464599609, 4.533749103546143, -0.1902216374874115, -3.285062313079834, -2.555034637451172, -1.5488801002502441], [-1.1171717643737793, -1.1219404935836792, -2.359776020050049, 2.1142404079437256, 0.8236517310142517, 0.3944133222103119, 3.2637205123901367, 3.1377272605895996, -11.305027961730957, 1.903433084487915, 7.733952045440674, -6.486321449279785, -1.684625267982483, -3.7065815925598145, 3.781242847442627, 2.6975741386413574]], "dtype": "float32"}}, {"a": [{"__nd__": [[1.4393051862716675, 0.2157079130411148, 0.15320636332035065, 0.41380277276039124, 2.0750269889831543, 1.0533324480056763, -1.4813756942749023, 0.5285462737083435, -1.175817847251892, 0.04912296682596207, -0.4893733859062195, -1.5033565759658813, -0.05438409745693207, 0.2533348798751831, -0.24372054636478424, 1.456406831741333, 0.28367164731025696, -2.2113025188446045, 0.6291079521179199, 0.8943269848823547, -0.8016515374183655, 0.63413405418396, -0.27119752764701843, 0.8010045886039734, -0.629773736000061, -0.022326914593577385, 0.7408179640769958, 0.32469937205314636, 1.8563637733459473, -0.21345217525959015, 0.2693161070346832, 0.3811730742454529, 0.4485868811607361, 1.107588529586792, 1.052988052368164, -2.1919877529144287, 0.7219376564025879, 0.6092461943626404, 0.6637588739395142, 0.05503709614276886]], "dtype": "float32"}, {"__nd__": [[-0.3096170425415039], [-0.880046546459198], [-1.6116411685943604], [0.5313345193862915], [-0.9809951186180115], [0.5514960289001465], [0.14242899417877197], [-0.32820725440979004], [-1.3628801107406616], [-1.3719003200531006], [-1.578343152999878], [0.9798533916473389], [2.3249666690826416], [-2.032012701034546], [0.09424810856580734], [-0.8123989105224609], [1.3864448070526123], [-0.20486149191856384], [1.3533036708831787], [-1.0964767932891846], [0.7370263338088989], [-0.14103403687477112], [-0.8041507005691528], [1.3843907117843628], [0.6642688512802124], [0.22009214758872986], [0.49950656294822693], [-0.3883223533630371], [0.3600126802921295], [1.4164800643920898], [0.1264190971851349], [-2.323073625564575], [-1.0223876237869263], [-1.0833779573440552], [1.585784912109375], [-0.30869060754776], [-1.0299662351608276], [0.0532749742269516], [0.06749647855758667], [-0.7187204957008362]], "dtype": "float32"}], "e": {"__nd__": [[-2.833566904067993]], "dtype": "float32"}}, {"a": [{"__nd__": [[0.6616436243057251], [-0.5735931396484375], [-1.8399910926818848], [0.9705987572669983], [0.4163038730621338], [0.9482295513153076], [-0.5722586512565613], [-0.7207335233688354], [0.17192496359348297], [-0.7972572445869446], [-0.21841399371623993], [-1.4642090797424316], [1.9632359743118286], [-0.13404116034507751], [-0.8691658973693848], [-0.16399270296096802], [0.534973680973053], [-0.6136927604675293], [-0.21775154769420624], [-0.2597913146018982], [1.4286633729934692], [0.0023352012503892183], [-0.3034577965736389], [0.5060416460037231], [0.9794664978981018], [-0.8828831911087036], [0.8029918670654297], [0.2895701825618744], [0.15358683466911316], [-1.373121738433838], [-1.7935502529144287], [-1.4012664556503296], [-1.3703194856643677]], "dtype": "float32"}, {"__nd__": [[-2.404571771621704, 0.04863593727350235, 1.6726540327072144, -0.34574437141418457, 1.2723066806793213, 0.7024322152137756, 0.5880951285362244, -0.739011287689209, -0.061333004385232925, -0.8362312912940979, -1.399953842163086, 0.14666584134101868, -1.4750890731811523, 0.8427942395210266, 0.4974752962589264, -0.273827463388443, -0.14054442942142487]], "dtype": "float32"}], "e": {"__nd__": [[-1.5909695625305176, 0.032179657369852066, 1.1067008972167969, -0.2287595570087433, 0.8418136239051819, 0.46475979685783386, 0.38910940289497375, -0.4889621138572693, -0.040580593049526215, -0.5532870888710022, -0.9262705445289612, 0.09704051911830902, -0.9759832620620728, 0.557629406452179, 0.32915136218070984, -0.18117620050907135, -0.09299032390117645], [1.3792458772659302, -0.027897240594029427, -0.9594228863716125, 0.19831660389900208, -0.7297863960266113, -0.4029102921485901, -0.33732733130455017, 0.4238918125629425, 0.035180188715457916, 0.47965651750564575, 0.8030039072036743, -0.08412651717662811, 0.8461009860038757, -0.4834209978580475, -0.28534841537475586, 0.15706555545330048, 0.0806153193116188], [4.42439079284668, -0.08948969095945358, -3.0776684284210205, 0.6361665725708008, -2.3410329818725586, -1.2924690246582031, -1.0820897817611694, 1.3597742319107056, 0.11285217851400375, 1.5386581420898438, 2.5759027004241943, -0.2698638439178467, 2.7141506671905518, -1.5507339239120483, -0.9153501391410828, 0.5038400888442993, 0.25860050320625305], [-2.333874464035034, 0.047205980867147446, 1.6234759092330933, -0.3355790674686432, 1.2348992824554443, 0.6817798614501953, 0.5708044171333313, -0.7172834277153015, -0.05952973663806915, -0.8116450309753418, -1.3587934970855713, 0.14235368371009827, -1.4317196607589722, 0.8180150389671326, 0.4828489124774933, -0.2657766044139862, -0.13641224801540375], [-1.0010325908660889, 0.020247329026460648, 0.6963323354721069, -0.1439347267150879, 0.5296661853790283, 0.2924252450466156, 0.2448262870311737, -0.3076532483100891, -0.025533167645335197, -0.3481263220310211, -0.5828062295913696, 0.06105755642056465, -0.61408531665802, 0.35085850954055786, 0.20710089802742004, -0.11399543285369873, -0.05850918963551521], [-2.280086040496826, 0.046118032187223434, 1.5860599279403687, -0.32784503698349, 1.2064387798309326, 0.6660670042037964, 0.5576491951942444, -0.700752317905426, -0.05815776810050011, -0.7929392457008362, -1.3274775743484497, 0.13907288014888763, -1.3987230062484741, 0.7991623878479004, 0.47172078490257263, -0.2596513032913208, -0.13326838612556458], [1.3760370016098022, -0.027832336723804474, -0.957190752029419, 0.19785520434379578, -0.7280884981155396, -0.4019729197025299, -0.3365425169467926, 0.4229055941104889, 0.03509834408760071, 0.4785405993461609, 0.8011357188224792, -0.08393079787492752, 0.8441324830055237, -0.48229628801345825, -0.28468453884124756, 0.15670013427734375, 0.08042776584625244], [1.7330554723739624, -0.035053551197052, -1.2055377960205078, 0.249189555644989, -0.9169940948486328, -0.5062664747238159, -0.4238598644733429, 0.532630205154419, 0.04420475289225578, 0.6026999354362488, 1.0089936256408691, -0.10570698976516724, 1.0631461143493652, -0.6074300408363342, -0.3585471212863922, 0.19735662639141083, 0.10129508376121521], [-0.4134059250354767, 0.00836173165589571, 0.287570983171463, -0.05944208800792694, 0.2187412828207016, 0.12076563388109207, 0.10110823065042496, -0.12705448269844055, -0.010544674471020699, -0.1437690407037735, -0.24068701267242432, 0.025215519592165947, -0.25360462069511414, 0.14489737153053284, 0.0855284258723259, -0.04707777500152588, -0.02416309528052807], [1.9170622825622559, -0.038775354623794556, -1.3335355520248413, 0.27564719319343567, -1.0143556594848633, -0.5600191950798035, -0.4688630998134613, 0.5891820788383484, 0.048898182809352875, 0.6666914820671082, 1.1161233186721802, -0.11693040281534195, 1.176025390625, -0.671923816204071, -0.39661577343940735, 0.21831092238426208, 0.11205006390810013], [0.525192141532898, -0.010622769594192505, -0.3653310537338257, 0.07551541179418564, -0.27788957953453064, -0.15342102944850922, -0.12844820320606232, 0.1614104062318802, 0.013395986519753933, 0.18264462053775787, 0.30576950311660767, -0.03203387185931206, 0.3221800923347473, -0.18407805263996124, -0.10865556448698044, 0.05980775132775307, 0.030696870759129524], [3.5207958221435547, -0.07121317833662033, -2.44911527633667, 0.506242036819458, -1.8629230260849, -1.0285075902938843, -0.8610942363739014, 1.0820670127868652, 0.08980434387922287, 1.2244174480438232, 2.0498251914978027, -0.21474945545196533, 2.159838914871216, -1.2340270280838013, -0.7284078598022461, 0.4009406566619873, 0.20578643679618835], [-4.7207417488098145, 0.09548382461071014, 3.2838146686553955, -0.678777813911438, 2.497838258743286, 1.379040241241455, 1.1545695066452026, -1.4508535861968994, -0.12041115760803223, -1.6417193412780762, -2.7484397888183594, 0.2879396677017212, -2.8959479331970215, 1.6546039581298828, 0.9766613841056824, -0.5375879406929016, -0.27592188119888306], [0.3223115801811218, -0.006519217509776354, -0.22420448064804077, 0.046343978494405746, -0.17054146528244019, -0.09415482729673386, -0.07882895320653915, 0.09905792772769928, 0.008221146650612354, 0.11208941042423248, 0.18765144050121307, -0.019659260287880898, 0.1977226436138153, -0.11296911537647247, -0.06668216735124588, 0.03670414909720421, 0.01883873902261257], [2.0899717807769775, -0.042272698134183884, -1.4538137912750244, 0.3005092144012451, -1.10584557056427, -0.6105301380157471, -0.5111522078514099, 0.6423234343528748, 0.053308553993701935, 0.7268237471580505, 1.216792106628418, -0.12747694551944733, 1.2820971012115479, -0.732528030872345, -0.432388573884964, 0.2380014955997467, 0.12215642631053925], [0.39433223009109497, -0.007975938729941845, -0.27430304884910583, 0.05669955536723137, -0.20864900946617126, -0.11519375443458557, -0.096443310379982, 0.12119245529174805, 0.010058165527880192, 0.1371358335018158, 0.22958222031593323, -0.024052128195762634, 0.24190384149551392, -0.13821209967136383, -0.08158231526613235, 0.044905707240104675, 0.023048261180520058], [-1.2863825559616089, 0.02601894550025463, 0.8948258757591248, -0.1849641352891922, 0.6806505918502808, 0.3757827579975128, 0.3146154284477234, -0.3953515887260437, -0.032811544835567474, -0.4473617374897003, -0.7489384412765503, 0.07846236228942871, -0.7891338467597961, 0.4508727490901947, 0.26613619923591614, -0.14649048447608948, -0.0751875713467598], [1.4756683111190796, -0.0298475231975317, -1.0264956951141357, 0.21218082308769226, -0.7808054089546204, -0.4310775697231293, -0.36090973019599915, 0.4535258710384369, 0.03763962164521217, 0.5131890773773193, 0.8591415286064148, -0.09000776708126068, 0.9052515029907227, -0.5172167420387268, -0.30529698729515076, 0.16804593801498413, 0.08625110238790512], [0.5235992074012756, -0.010590550489723682, -0.36422300338745117, 0.0752863734960556, -0.27704674005508423, -0.15295569598674774, -0.12805862724781036, 0.1609208583831787, 0.013355356641113758, 0.1820906549692154, 0.30484211444854736, -0.03193671256303787, 0.32120293378829956, -0.18351975083351135, -0.1083260178565979, 0.05962635576725006, 0.030603766441345215], [0.6246868371963501, -0.012635193765163422, -0.4345409870147705, 0.08982138335704803, -0.3305342197418213, -0.18248578906059265, -0.15278200805187225, 0.19198870658874512, 0.01593378186225891, 0.21724562346935272, 0.3636958599090576, -0.03810251131653786, 0.3832153379917145, -0.21895062923431396, -0.1292397677898407, 0.07113799452781677, 0.03651222214102745], [-3.435323715209961, 0.06948438286781311, 2.3896596431732178, -0.49395233392715454, 1.8176980018615723, 1.0035392045974731, 0.8401899933815002, -1.0557984113693237, -0.0876242145895958, -1.1946929693222046, -2.0000627040863037, 0.2095361202955246, -2.107405662536621, 1.2040692567825317, 0.7107247114181519, -0.3912072777748108, -0.200790673494339], [-0.005615158937871456, 0.00011357470066286623, 0.0039059838745743036, -0.0008073826902545989, 0.002971092239022255, 0.0016403206391260028, 0.0013733204687014222, -0.0017257400322705507, -0.00014322491188067943, -0.0019527683034539223, -0.003269173903390765, 0.00034249425516463816, -0.003444629954174161, 0.0019680941477417946, 0.0011617048876360059, -0.0006394422380253673, -0.00032819953048601747], [0.7296860218048096, -0.014758954755961895, -0.5075799226760864, 0.10491882264614105, -0.3860913813114166, -0.2131585329771042, -0.17846205830574036, 0.22425873577594757, 0.018611978739500046, 0.2537609040737152, 0.4248269200325012, -0.04450689256191254, 0.4476272761821747, -0.25575247406959534, -0.15096275508403778, 0.08309508115053177, 0.04264930263161659], [-1.2168134450912476, 0.024611810222268105, 0.846432626247406, -0.17496104538440704, 0.6438401937484741, 0.35545995831489563, 0.2976006269454956, -0.37397047877311707, -0.03103705495595932, -0.4231678545475006, -0.7084349393844604, 0.07421902567148209, -0.746456503868103, 0.426488995552063, 0.25174322724342346, -0.13856810331344604, -0.07112133502960205], [-2.3551974296569824, 0.047637272626161575, 1.6383086442947388, -0.3386450409889221, 1.2461817264556885, 0.6880088448524475, 0.5760194659233093, -0.7238367795944214, -0.060073621571063995, -0.8190605640411377, -1.3712078332901, 0.14365427196025848, -1.4448003768920898, 0.8254887461662292, 0.4872603714466095, -0.26820483803749084, -0.13765856623649597], [2.1229560375213623, -0.04293985292315483, -1.4767581224441528, 0.3052518963813782, -1.123298168182373, -0.6201655864715576, -0.5192192792892456, 0.6524606347084045, 0.05414987727999687, 0.7382945418357849, 1.2359957695007324, -0.12948881089687347, 1.3023313283920288, -0.744088888168335, -0.4392125904560089, 0.24175766110420227, 0.12408431619405746], [-1.9308515787124634, 0.0390542633831501, 1.3431276082992554, -0.27762991189956665, 1.0216518640518188, 0.5640473365783691, 0.47223562002182007, -0.5934200286865234, -0.04924990236759186, -0.6714869141578674, -1.124151587486267, 0.11777147650718689, -1.1844844818115234, 0.6767569184303284, 0.39946863055229187, -0.21988122165203094, -0.11285603046417236], [-0.6962922811508179, 0.014083516784012318, 0.4843507409095764, -0.1001172587275505, 0.36842209100723267, 0.20340342819690704, 0.17029480636119843, -0.21399563550949097, -0.01776020973920822, -0.24214765429496765, -0.40538489818573, 0.0424700528383255, -0.42714181542396545, 0.2440480887889862, 0.14405401051044464, -0.07929226756095886, -0.040697477757930756], [-0.36931055784225464, 0.007469839882105589, 0.25689762830734253, -0.053101781755685806, 0.19540955126285553, 0.10788433998823166, 0.09032367169857025, -0.11350240558385849, -0.009419942274689674, -0.12843412160873413, -0.2150144726037979, 0.02252594195306301, -0.2265542596578598, 0.12944209575653076, 0.07640565931797028, -0.04205629229545593, -0.021585773676633835], [3.301769733428955, -0.06678306311368942, -2.296757698059082, 0.4747491180896759, -1.7470319271087646, -0.9645249247550964, -0.8075262308120728, 1.0147525072097778, 0.08421768248081207, 1.1482473611831665, 1.922307014465332, -0.20139005780220032, 2.0254769325256348, -1.1572591066360474, -0.6830941438674927, 0.37599843740463257, 0.19298461079597473], [4.31272029876709, -0.08723099529743195, -2.9999890327453613, 0.6201099157333374, -2.2819459438323975, -1.2598475217819214, -1.0547782182693481, 1.3254538774490356, 0.11000382900238037, 1.4998228549957275, 2.510887622833252, -0.2630525529384613, 2.645646333694458, -1.5115938186645508, -0.8922469615936279, 0.4911233186721802, 0.2520734965801239], [3.36944580078125, -0.06815190613269806, -2.3438339233398438, 0.48447999358177185, -1.7828407287597656, -0.9842947125434875, -0.8240779638290405, 1.0355517864227295, 0.08594388514757156, 1.1717828512191772, 1.9617083072662354, -0.20551791787147522, 2.06699275970459, -1.1809792518615723, -0.6970954537391663, 0.3837052285671234, 0.19694019854068756], [3.2950315475463867, -0.0666467696428299, -2.2920703887939453, 0.47378024458885193, -1.7434666156768799, -0.9625565409660339, -0.805878221988678, 1.0126816034317017, 0.08404581248760223, 1.1459040641784668, 1.918384075164795, -0.2009790539741516, 2.021343231201172, -1.1548973321914673, -0.6817001104354858, 0.3752311170101166, 0.19259077310562134]], "dtype": "float32"}}, {"a": [{"__nd__": [[-1.249143362045288, 1.3864648342132568, 1.5575013160705566, -1.5350253582000732, -0.12025515735149384, -0.631278395652771, -0.7626879811286926, 1.3463048934936523, -0.46980810165405273, 0.3091701567173004, -0.3539285957813263, -0.04074394702911377, 1.9184051752090454, -0.10652761906385422, -0.5001365542411804, 0.0804072916507721, -0.0016956881154328585, -0.2690334618091583, 0.7391549348831177, 1.494371771812439, 2.1804635524749756, 1.029153823852539, 1.2638380527496338, -0.6678385138511658, 0.3044050335884094, 2.579052209854126, -1.7521017789840698, -0.30035844445228577, 0.3365589678287506, -1.3102123737335205, -1.3625051975250244, 0.5590873956680298, 1.7408554553985596, -0.4950840473175049, 1.7679154872894287, 0.17865389585494995, -0.6907303929328918], [0.2870519459247589, -1.991064190864563, 0.28878340125083923, 0.059283655136823654, -0.6473793387413025, 1.5908905267715454, -1.16514253616333, 0.14992958307266235, 0.3109428286552429, -1.078850507736206, 0.6931899189949036, 0.6338115334510803, 0.028431151062250137, -0.035141315311193466, 0.06815551221370697, 0.7132683396339417, 0.7500962615013123, -1.0333094596862793, 1.4909391403198242, -0.41319429874420166, 0.4867914617061615, -0.9059522151947021, 2.3636868000030518, 0.5986505150794983, 0.016170701012015343, 0.6402618288993835, -0.34035739302635193, 1.6303861141204834, -1.0398505926132202, -0.7267575263977051, -0.3188627064228058, 0.8893001079559326, -0.11471333354711533, -0.7236098647117615, 0.5714967250823975, 0.13550733029842377, -0.29000523686408997], [-0.3421729803085327, -1.433215856552124, -1.1694828271865845, 1.8697670698165894, -1.195660948753357, -0.23578619956970215, 0.06306257843971252, 0.8827603459358215, -0.9751105308532715, -0.6480603814125061, -1.034399151802063, -0.8476278781890869, -0.2557566463947296, -0.6976123452186584, -0.35610610246658325, 0.16111372411251068, 0.0785754919052124, 0.17436432838439941, 2.1978368759155273, -0.3287617564201355, -0.43385574221611023, -0.09730363637208939, -2.302438259124756, -1.1010717153549194, -0.6183634996414185, 0.03662435710430145, -0.8106314539909363, 1.3213928937911987, -0.23513244092464447, -1.7580069303512573, -1.0246694087982178, 0.7698619961738586, 0.2646069824695587, -0.40825021266937256, -0.3601579964160919, -0.12244436144828796, 0.729735255241394], [0.12535610795021057, 0.5561830997467041, 0.1785457879304886, -0.9302701950073242, 0.6681833267211914, -0.8419563174247742, 1.2510459423065186, 1.0379191637039185, -0.34233328700065613, -0.0788799449801445, 0.2634779214859009, 3.2086703777313232, 0.41442251205444336, 0.42347678542137146, -1.7596644163131714, 0.7797736525535583, 0.9547021985054016, -0.7975969910621643, -0.13559161126613617, -1.7351101636886597, 0.8714841604232788, 1.1553287506103516, -0.2971850335597992, -1.0662462711334229, -1.724220871925354, -0.449076384305954, -1.7607719898223877, -0.029500547796487808, -2.598599672317505, -0.2943534255027771, -0.5226547122001648, 1.184491753578186, -1.2497830390930176, 1.2983227968215942, -0.3616882562637329, 0.39026111364364624, 0.03449484333395958], [0.4415722191333771, -0.03668029606342316, 0.24084100127220154, -1.6161285638809204, 1.6760927438735962, -1.1091654300689697, 1.1380482912063599, -0.6414288878440857, 0.18370020389556885, 0.8982983231544495, -1.465858817100525, -0.5795390605926514, 0.6521354913711548, -0.6570749282836914, -2.1090738773345947, -0.9541490077972412, -0.8810287714004517, 1.0128810405731201, 1.3122284412384033, -1.9575026035308838, -1.038789987564087, 1.0009418725967407, -1.7115119695663452, -1.936537265777588, 0.7586038708686829, -0.7709392309188843, 0.04210721701383591, -0.04356762394309044, -0.31715571880340576, 0.6255019307136536, -0.9640300869941711, 0.707813024520874, 0.26430824398994446, -0.4953363239765167, 0.739973783493042, 2.0185346603393555, 0.39960458874702454], [1.5595645904541016, 1.6187598705291748, -0.07426338642835617, 0.21154919266700745, 1.6580768823623657, 1.2356985807418823, -0.006145513150840998, -2.0290160179138184, -0.19367428123950958, -0.13646744191646576, -1.874213457107544, -0.7290323972702026, 0.14578668773174286, 1.4098713397979736, 0.29439863562583923, 1.682746410369873, 0.308089017868042, -0.5604760646820068, -0.06413189321756363, -0.4128934144973755, 1.4751826524734497, -0.49360159039497375, 1.0209181308746338, 0.3318013846874237, -0.8718373775482178, 0.8568539619445801, -1.0112839937210083, -0.02728966623544693, -0.11003953963518143, 2.778522253036499, -0.27525049448013306, 0.028604580089449883, 0.25267043709754944, -0.1921064853668213, -1.2365200519561768, 0.016064591705799103, -0.09801001101732254], [-0.6751662492752075, 1.2458211183547974, -0.39068013429641724, -0.30406591296195984, -0.27017346024513245, -0.9987879991531372, -0.5535424947738647, 0.6235530972480774, -0.26837795972824097, -0.35557231307029724, 0.41402509808540344, 0.13297665119171143, 0.5745187997817993, -0.3241819441318512, -0.04212848097085953, 0.9195706248283386, 0.126670703291893, -0.4733211398124695, -0.8314712643623352, -0.06718345731496811, 0.6314348578453064, 0.5154050588607788, -0.003568109590560198, -0.29038482904434204, 1.030746579170227, 1.146702766418457, -0.19437795877456665, -2.2944836616516113, -1.378235101699829, -1.184241533279419, -0.952409565448761, -0.18643450736999512, 0.31696128845214844, 0.12626589834690094, -0.27498698234558105, -1.6894514560699463, 1.384412169456482], [-1.281720757484436, -0.007135729305446148, 0.8371917605400085, -1.0725151300430298, -0.8686923980712891, 1.5389866828918457, -0.8533881902694702, 0.47000810503959656, -0.2802385985851288, -0.5040450692176819, -0.12391887605190277, -0.9785252809524536, -1.0057810544967651, 0.12330267578363419, -0.09007872641086578, 2.032615900039673, 2.9996607303619385, -0.06056956201791763, -0.4902915060520172, 0.6778901219367981, -0.39383354783058167, 0.3597414791584015, -1.4072767496109009, 1.5651042461395264, 0.14865854382514954, -2.2213375568389893, 1.2420669794082642, 0.9369343519210815, -0.45740771293640137, -0.8589756488800049, -1.3924587965011597, -0.26098787784576416, 0.6651279330253601, -0.34357157349586487, -0.6438571810722351, 0.3048096299171448, 0.39460328221321106], [-0.5752949714660645, 0.2740379273891449, -2.033566951751709, 0.3127802908420563, 3.001997470855713, 1.212977647781372, 1.5060349702835083, 1.0168200731277466, 0.581000030040741, 0.3067168593406677, -2.1356568336486816, 0.46647584438323975, -0.6529149413108826, 0.6503286957740784, 0.1016879603266716, 0.6700311899185181, 1.9069148302078247, -0.4157576262950897, -0.3423141837120056, 0.5503010749816895, -1.935341238975525, 2.2327466011047363, 0.9394667148590088, 0.7244943976402283, 1.1710630655288696, 0.7291367650032043, 0.8445981740951538, -1.021562099456787, 0.42913827300071716, -1.3657705783843994, -0.12510772049427032, 0.8573450446128845, 0.7175241708755493, -1.4023841619491577, 0.7866395711898804, -0.5287216901779175, 1.1277523040771484], [-1.105575680732727, 1.8512192964553833, -0.15679797530174255, -0.5178839564323425, 0.2309720814228058, -0.42517217993736267, -0.15489055216312408, -1.5976958274841309, -0.10397736728191376, 0.6329456567764282, -1.0314908027648926, -0.28766775131225586, 1.3993430137634277, 0.37977784872055054, -1.491407036781311, -0.43044543266296387, -0.3196876347064972, -0.047127656638622284, 1.5223264694213867, -0.8430125117301941, -1.1400549411773682, 0.8768761157989502, -1.6308377981185913, 0.26935237646102905, 0.6249772906303406, -0.10660835355520248, -0.42585045099258423, 0.07441144436597824, 0.37998417019844055, 1.216493010520935, -2.2910187244415283, 0.005825773347169161, 0.5633667707443237, 0.5574789643287659, 0.11787765473127365, 2.0108091831207275, 0.917820155620575], [-0.4133795201778412, -0.9262291789054871, -0.2935240864753723, -0.2418680191040039, 0.05965636298060417, 0.6460497379302979, 1.6183027029037476, -0.028219306841492653, -0.3777739107608795, 1.6835299730300903, 1.050227403640747, -1.0933327674865723, -0.6931524872779846, 0.4318079948425293, 1.0296363830566406, 0.11504550278186798, -0.9536433219909668, -1.102808952331543, 0.7636988759040833, -0.09190214425325394, 0.29760193824768066, 0.6707137823104858, 1.2592467069625854, 1.537795901298523, 0.6752349138259888, 1.1614712476730347, -1.442550539970398, -2.460768938064575, -1.203094244003296, -0.3099099099636078, -0.09529716521501541, 0.9803205132484436, 0.46033644676208496, 0.039410609751939774, 0.856511116027832, -1.7120025157928467, 0.5662954449653625], [1.0462185144424438, -0.3179423213005066, -0.47317641973495483, 0.25570085644721985, -0.9777848124504089, 0.3670669198036194, -0.9426488280296326, -0.19269827008247375, 0.8709204792976379, 0.5318325161933899, 0.07146637886762619, 0.4572938084602356, -2.790585517883301, -0.31492847204208374, 0.4813557267189026, -0.8155454397201538, 1.2389284372329712, -0.2675003111362457, 0.7258123159408569, 0.8356037139892578, 1.6163688898086548, -1.2614860534667969, -0.7022428512573242, 0.8090056777000427, 0.257311075925827, -0.11108759790658951, -1.6241480112075806, -0.3471240699291229, 0.7485638856887817, 0.40926140546798706, -0.283202588558197, 0.8620163202285767, 0.2776765823364258, -3.084627389907837, 0.2717599868774414, 0.14679276943206787, 0.4201306700706482], [0.9879507422447205, -0.843071460723877, -0.5383954048156738, 0.9336435198783875, 0.45197558403015137, -1.1398388147354126, 0.36596745252609253, -0.6508204936981201, 1.2774202823638916, 1.051245093345642, -0.7102986574172974, 0.527148962020874, -1.0913618803024292, 1.33149254322052, -0.15090304613113403, 0.430849552154541, 1.2083252668380737, -0.11553888767957687, 0.7698143124580383, -0.36024436354637146, 1.9209129810333252, -0.0651976689696312, -0.943240225315094, 1.1080294847488403, -0.1543721854686737, 0.7548221945762634, 0.6341050267219543, 0.13531941175460815, 0.1845429390668869, 0.17221792042255402, -0.9360170960426331, -1.465679407119751, 1.4771405458450317, 0.4014008045196533, -0.9969277381896973, 1.0923945903778076, -1.3937816619873047], [0.9776761531829834, 0.0581263042986393, 0.0783671960234642, 1.4388608932495117, -1.365017294883728, 0.9396739602088928, 0.7490217685699463, 0.3384007215499878, 0.6991943717002869, 0.05185524746775627, 1.3848739862442017, 0.5873803496360779, -1.127777099609375, -0.4576147496700287, 0.26250767707824707, -1.684719443321228, 1.319298505783081, -1.0417231321334839, 2.7474257946014404, 0.4342908561229706, 0.23166324198246002, 1.2521055936813354, -0.8384727835655212, 0.6586491465568542, -1.2642626762390137, 0.308167040348053, -0.055591415613889694, -0.5868768692016602, -0.15473274886608124, 0.08325271308422089, 1.5397613048553467, 1.942628026008606, 2.8202481269836426, 0.5505018830299377, -1.9720745086669922, -1.4974710941314697, -0.7298610210418701], [-0.3194061517715454, 0.2915869355201721, 1.1071504354476929, -0.7288244962692261, -0.45122236013412476, 0.995521068572998, 1.7341314554214478, -0.5763592720031738, -0.6715118288993835, -0.528118908405304, 0.21406735479831696, -0.4869407117366791, 0.4103084206581116, -0.053654175251722336, 0.04007983207702637, -1.3655372858047485, 0.1240016296505928, -0.11893566697835922, -0.6383205056190491, -1.221197485923767, 0.4200476408004761, -0.2777344286441803, 0.4337327778339386, 1.6353886127471924, -1.1717056035995483, 0.048349425196647644, -0.08404122292995453, 1.710408329963684, 0.15471519529819489, -0.8562072515487671, -0.6256473064422607, 1.2565068006515503, 2.575093984603882, 0.5017510056495667, 0.5485460162162781, 0.9783571362495422, 1.0617135763168335], [-0.7155612111091614, -0.5456958413124084, 0.5741896033287048, 1.2359877824783325, -1.010845422744751, -0.009636381641030312, -0.14084511995315552, 1.5754530429840088, 0.6774392127990723, -0.1361870914697647, 0.16563767194747925, -0.3381301164627075, 1.5955612659454346, -0.3677208721637726, 0.3715198338031769, -1.0955206155776978, -0.0768110528588295, -1.2698429822921753, -0.04699414595961571, 1.4775782823562622, -1.6743742227554321, 2.5757925510406494, -2.65767765045166, 1.7015374898910522, 0.8877031207084656, -0.9511576890945435, -0.3471750020980835, 1.9024590253829956, -0.44302454590797424, -1.2331416606903076, 0.6684482097625732, 1.8564808368682861, 0.2506594657897949, -1.3609486818313599, -0.8364236950874329, 0.6723834872245789, -0.08916323632001877], [-0.5893425345420837, 0.04126964882016182, 0.41630297899246216, -1.622368574142456, -1.0057936906814575, 1.1322025060653687, -0.48473724722862244, -1.3451350927352905, -1.0954031944274902, 0.8065809011459351, 0.5236005783081055, 1.7143880128860474, 1.1881511211395264, 0.6888318657875061, 1.4619053602218628, -0.35873401165008545, 0.5645006895065308, -1.0028167963027954, 0.08832293748855591, -1.5908300876617432, 1.1645220518112183, 1.3093597888946533, 0.8304781317710876, -0.3690938353538513, 1.575237512588501, -0.35229364037513733, 0.3343455493450165, 0.15983949601650238, 2.05971097946167, 0.3632194399833679, 1.1741479635238647, -0.7668637037277222, -0.08113471418619156, 1.7619833946228027, 1.0360064506530762, -0.2669912874698639, -0.4503484070301056], [0.6926666498184204, 2.3003954887390137, -0.026392169296741486, -1.085078477859497, -1.128860354423523, -1.3353674411773682, 0.9261583685874939, -0.7217307090759277, 0.3458363115787506, 2.193772554397583, 1.5325822830200195, -0.1292942464351654, 1.067788004875183, 0.7859490513801575, -0.8789788484573364, -0.37807977199554443, 1.4989432096481323, 0.5072832703590393, 2.2579593658447266, -0.2549755275249481, -1.4313000440597534, -2.231471061706543, 0.31454524397850037, 0.45538318157196045, -0.14387273788452148, 0.19202624261379242, -0.42015746235847473, -1.1689953804016113, 0.17610934376716614, 1.8826913833618164, 1.8295838832855225, -1.748116374015808, -1.2473276853561401, -0.2924667000770569, 0.6186904311180115, 0.14022503793239594, -0.8309634923934937], [-0.5502707362174988, 1.7769157886505127, 0.6296822428703308, -1.2755571603775024, 1.615081787109375, 0.5269462466239929, 0.6434381604194641, -0.14438538253307343, 0.24542127549648285, -1.3713003396987915, -0.4622924327850342, 0.7267752289772034, -0.9035587310791016, -0.41588667035102844, 0.26233381032943726, -0.4279780089855194, 0.19320057332515717, -1.8480764627456665, 1.0903615951538086, 1.1965627670288086, 2.030982732772827, -0.7512194514274597, 0.41160714626312256, 0.2430250346660614, -0.48376187682151794, -0.12580835819244385, -1.3850840330123901, -1.3514164686203003, -0.014389577321708202, 1.07095205783844, -0.5718356966972351, -1.9524612426757812, 0.48971179127693176, -0.6996635794639587, -2.0256118774414062, 0.8309434652328491, -0.7203055024147034], [1.3177138566970825, 0.12341027706861496, 1.015089988708496, -1.2044181823730469, -0.0387839712202549, 0.6552708148956299, 0.1136617586016655, 0.11534211039543152, -1.9692076444625854, -1.0241056680679321, 0.43385186791419983, -0.20742198824882507, -0.29410308599472046, 1.4511785507202148, -0.856670081615448, 0.23892469704151154, -0.3365667760372162, 1.487099289894104, 1.2002419233322144, -0.37797313928604126, -0.23980708420276642, 0.11594068259000778, 0.6526628732681274, -1.6502258777618408, 1.8933535814285278, 0.7317666411399841, 0.7234911918640137, 0.22595398128032684, 0.6824948787689209, -0.7872582674026489, -0.9581859707832336, 0.2479042410850525, -0.10163338482379913, 2.2963058948516846, -1.0970830917358398, 0.1408650279045105, 1.0219844579696655]], "dtype": "float32"}, {"__nd__": [[1.2645602226257324, 1.1480120420455933, 1.089289903640747, -0.7070832848548889, -1.3004045486450195, -0.381103515625, 0.45978525280952454, -0.6585033535957336, -0.24698691070079803, -1.6493470668792725, 0.72837233543396, -1.9553622007369995, -0.6275736093521118, 0.08214612305164337, -0.19775128364562988, 1.5013024806976318, 0.5202142596244812, -0.9082051515579224, 0.12710916996002197, 1.8933992385864258, 1.261661410331726, -0.12296511977910995, 0.5953481197357178, 0.48032036423683167, 1.1764148473739624, 0.9368293285369873, 1.5540276765823364, -0.6483160257339478, -0.1702885776758194, 0.9316307306289673, -0.130025714635849, -0.8479329943656921, -0.1093049868941307, 0.2694140672683716, -1.4293893575668335, 1.6872432231903076, 0.7083818316459656, 1.3644380569458008, -0.6857261657714844, -1.785672664642334, -0.4387011229991913, 0.5981029272079468, 1.5914885997772217, 0.5040165185928345, 0.05004998669028282, -0.1396123617887497, 0.41789937019348145, -0.18337538838386536, 0.9892582297325134, -0.44643235206604004], [0.220261350274086, -0.529640257358551, -0.89341801404953, -0.854446291923523, -1.5227231979370117, 0.8321229219436646, 1.3464245796203613, -1.343045949935913, -0.47856539487838745, 0.6716835498809814, -1.0904918909072876, -0.40195995569229126, 0.5824376344680786, 0.33923813700675964, -1.507087230682373, 1.699859380722046, 0.9626574516296387, 0.8569311499595642, -0.10566814988851547, 0.07231952995061874, 0.5684998035430908, 0.7276915907859802, 0.3188548982143402, 0.7028483748435974, -0.5775349736213684, -1.6148266792297363, -1.187673568725586, 0.7397963404655457, -0.6158575415611267, 0.03421718254685402, -0.873441219329834, 1.5210238695144653, 0.018161501735448837, 0.005328693427145481, 1.6254569292068481, 0.6375282406806946, -0.8297415971755981, -0.14209751784801483, 0.5391647219657898, -0.9679165482521057, -0.5187839865684509, 0.011033749207854271, 0.7549916505813599, -0.3730923533439636, -1.0136079788208008, -0.5793966054916382, -0.7147062420845032, 0.06591462343931198, -2.168314218521118, 1.511496901512146], [-0.5582626461982727, 0.3797326982021332, -1.556175947189331, -0.686828076839447, 1.0584343671798706, 0.4665988087654114, -1.034747838973999, 0.33080241084098816, -2.184577465057373, -2.4023375511169434, -0.19889765977859497, 0.03560533747076988, -1.7878788709640503, -1.0250877141952515, 0.9384136199951172, 0.46353596448898315, 0.7484400868415833, 0.4283333122730255, -0.7897458672523499, 1.559450387954712, -1.2777808904647827, 0.39373326301574707, -0.392904669046402, -2.053642749786377, -0.2231009155511856, -0.6431012749671936, 1.8350756168365479, -0.8729358911514282, 0.11836715787649155, -0.7786174416542053, 2.6200523376464844, 0.3490086793899536, 2.1421267986297607, 2.410931348800659, 0.7412024736404419, 0.24625329673290253, 0.5117915868759155, -1.1725623607635498, 0.5626295804977417, 0.6280818581581116, 2.5666232109069824, -0.5560736060142517, -0.472011923789978, -0.6911661624908447, -0.8372477293014526, -0.5163918137550354, -0.14404256641864777, 1.5830408334732056, -0.4559612274169922, -0.3383301794528961], [1.6988730430603027, 0.06366371363401413, 1.337294578552246, -0.9776114225387573, -0.06232842430472374, 0.6099709868431091, 1.1078646183013916, -0.35659173130989075, 0.9722387194633484, 0.3896883726119995, -0.6461375951766968, 0.28369492292404175, 0.20874305069446564, -0.5237234234809875, -0.10355953872203827, -1.7758182287216187, 1.128983974456787, 1.8363581895828247, 0.5461604595184326, -1.343428373336792, -0.16854813694953918, 0.4372425675392151, 0.46255287528038025, 0.845501184463501, -1.942229151725769, -0.3402252495288849, 0.38116323947906494, 1.324306845664978, 0.48674139380455017, -0.210441455245018, 0.06518097966909409, -0.5594789981842041, 1.4821830987930298, 0.9776222109794617, -0.7524546384811401, 0.4705246388912201, -0.5266633629798889, -0.349508672952652, 0.13032734394073486, 0.4456884562969208, 0.0093560004606843, 0.08892691135406494, 1.659904956817627, -0.16974835097789764, -1.2856721878051758, 0.2119104117155075, 0.5470382571220398, 0.5810170769691467, 0.9811555743217468, 1.6910489797592163], [-0.004783032927662134, -1.2059775590896606, -1.2523783445358276, -0.18939387798309326, 0.8570515513420105, -0.35370489954948425, 0.4890599250793457, 0.4533108174800873, -1.814461350440979, -1.5757837295532227, 1.8708546161651611, -1.5620462894439697, -1.4354251623153687, -0.7054746747016907, -0.6361232995986938, -1.6810030937194824, -0.36928442120552063, -0.9848297834396362, 1.3925957679748535, -1.509105920791626, -0.5336877107620239, -0.012398703023791313, 0.19683466851711273, -0.462501585483551, 1.8566538095474243, -0.9185366630554199, 0.6533066034317017, -0.6843893527984619, 1.1506212949752808, 1.1424535512924194, 1.5447810888290405, -0.38932058215141296, -1.7532007694244385, 1.4308488368988037, -0.43124210834503174, 0.8705706596374512, -0.8915296196937561, 1.0761247873306274, -0.6725930571556091, -0.053979311138391495, 0.6942532658576965, -1.408097743988037, 0.6960569024085999, -0.07215608656406403, -1.0765550136566162, -2.050204038619995, -0.9720120429992676, 2.3013272285461426, -0.48034417629241943, -0.8356047868728638], [-0.5368346571922302, -0.5925765037536621, 1.921505093574524, -0.09141748398542404, -0.8529096841812134, 0.08762780576944351, -0.5178886651992798, 1.8717453479766846, -0.6823317408561707, -0.9525488018989563, 0.8232854008674622, 0.5434777736663818, -0.65842604637146, -0.5775011777877808, 1.6250251531600952, 1.977691411972046, -0.9918935298919678, 0.07591095566749573, 1.1936860084533691, 0.6554006934165955, 1.1783416271209717, 1.4998364448547363, -1.3718795776367188, -1.3034380674362183, -0.734732985496521, -0.05931644141674042, -1.203064203262329, -0.6650708913803101, -0.7264554500579834, 0.4757269620895386, 0.1008576974272728, 0.16101546585559845, -0.11254727095365524, -1.0642503499984741, 1.5929034948349, 0.284936785697937, -1.2058873176574707, 0.7234839200973511, 0.7327496409416199, -0.038749296218156815, 0.7475981116294861, -1.357555866241455, -0.7583580017089844, -0.38042712211608887, 0.26459458470344543, 0.4433271288871765, -0.38358116149902344, 1.5406839847564697, -1.1075975894927979, -0.7729676365852356], [-0.007740907836705446, -0.6308556199073792, -0.23087017238140106, -0.6457707285881042, -0.06957513093948364, 0.9258058071136475, -0.30790960788726807, -0.2806311845779419, -2.123372793197632, -0.9638135433197021, 0.5497223138809204, -0.3125702738761902, -0.05161762610077858, -1.5566834211349487, 0.013744886964559555, -1.3662422895431519, -2.377211570739746, 1.2190979719161987, 0.07903856039047241, -0.08471040427684784, -0.8686246275901794, -0.9355689287185669, 0.2796752154827118, -1.5432504415512085, -0.5887665152549744, 1.7150119543075562, -0.13851983845233917, 0.15650370717048645, -0.0050267064943909645, -0.9750561118125916, 1.0659459829330444, 1.3541030883789062, 1.3360460996627808, -0.1861182153224945, 0.11240163445472717, -0.23690000176429749, 0.052543312311172485, -0.08125567436218262, -0.9394270777702332, 1.568725347518921, -0.13950282335281372, -0.21829426288604736, -0.9808777570724487, -1.9150152206420898, 0.46753042936325073, -0.7874442934989929, -0.7108036279678345, -0.35395967960357666, -0.9560460448265076, 0.5564242005348206], [0.19909131526947021, 0.6674235463142395, 0.6909879446029663, 0.8447456955909729, 0.42426246404647827, 0.40978768467903137, -1.3455561399459839, 1.0574755668640137, 0.6339831352233887, -1.044692039489746, 1.3319212198257446, -0.9070123434066772, 0.6255741715431213, -0.5417338609695435, -1.059451937675476, -1.4459563493728638, 1.8079017400741577, 1.4892908334732056, -0.2031591534614563, -0.3977430760860443, -1.1080807447433472, -1.6647874116897583, -0.6646973490715027, 0.21539168059825897, -0.06792453676462173, 0.012369859032332897, -0.7255412936210632, -0.12084514647722244, 0.5170561671257019, -0.1594506949186325, 0.24872982501983643, -0.09152692556381226, 0.32319897413253784, -0.44937920570373535, 0.6389808654785156, -0.10929305851459503, -0.26424914598464966, -1.003653645515442, -0.23478175699710846, -0.551044225692749, 0.8135408163070679, 0.7233843803405762, -1.5543113946914673, 0.5775173902511597, -0.8230866193771362, -0.016612203791737556, -0.45683160424232483, -0.5388100743293762, 0.7630089521408081, -1.2222061157226562], [-0.435001939535141, -0.9906783103942871, 0.17841078341007233, -0.5612765550613403, 1.2070348262786865, -0.1458492875099182, -0.7796670198440552, -0.998496413230896, 0.08826714754104614, -0.6672530770301819, -0.46013975143432617, 1.5345556735992432, 0.6952883005142212, -0.546180248260498, 0.9176040291786194, -0.19536158442497253, 2.3745665550231934, -1.3529223203659058, 1.398808240890503, -1.1153143644332886, -0.9076778292655945, 1.7798349857330322, 0.627372682094574, 1.643419623374939, 0.6226535439491272, -0.90618896484375, -0.4378754198551178, -0.4601186513900757, 1.11629056930542, 0.5453159213066101, 0.5229141712188721, 0.03273117542266846, -1.5336923599243164, -0.5193619132041931, 2.2122249603271484, 1.4983012676239014, 0.5016546249389648, 1.3666772842407227, -0.06354735046625137, 0.38827240467071533, -0.6069867610931396, 0.6493679881095886, -1.1724731922149658, 0.6171168088912964, -1.311450719833374, 0.7389886379241943, 0.08117274194955826, 0.8120885491371155, -0.524132251739502, 0.10020778328180313], [-0.35411539673805237, 0.6900312304496765, -0.5733237862586975, -1.2864031791687012, 0.5380593538284302, -0.2226782739162445, 0.17494451999664307, 0.20388631522655487, -0.6629428863525391, 1.6304517984390259, 0.7456673979759216, 0.468588650226593, 1.0992385149002075, -0.4897550344467163, 0.5114569067955017, 0.1607087254524231, -0.5831444263458252, 0.851082444190979, 0.6904820799827576, -2.5857176780700684, -1.2862567901611328, 0.9996694326400757, 0.08881863951683044, -0.8678407073020935, -0.12061329931020737, -0.5823057293891907, -1.6252341270446777, -1.96379816532135, 0.08739656955003738, 1.4050374031066895, 1.999518871307373, -1.1576794385910034, -0.002897367812693119, -0.8433405756950378, -0.1495736539363861, 1.1216613054275513, 0.35777637362480164, -0.558384358882904, 0.5787261724472046, 1.1179828643798828, 0.34921249747276306, 0.7671771049499512, -0.9660554528236389, 1.10946786403656, 0.9343395233154297, 0.9630346894264221, 1.1328150033950806, 1.4027094841003418, 2.269205331802368, 1.4778680801391602], [0.3256719410419464, -0.28243696689605713, -1.3976365327835083, -0.823079526424408, -0.8268972039222717, 1.2119749784469604, 0.21735069155693054, -0.251101016998291, 1.4029474258422852, 0.0700446218252182, 0.04600995033979416, 1.4163376092910767, -0.24723853170871735, 0.4177952706813812, -2.7586517333984375, 2.037703514099121, -0.2583867609500885, -0.2576827108860016, 0.5686384439468384, -0.41363635659217834, -0.11735305190086365, -0.9259362816810608, -2.3070123195648193, 0.7642122507095337, 0.4020560383796692, 1.3537030220031738, -0.7087494730949402, 0.8719728589057922, 1.2881370782852173, 1.4061992168426514, -0.46073514223098755, 1.1130532026290894, 1.036354422569275, -0.0018726319540292025, 2.312147617340088, -0.20559373497962952, -0.23321695625782013, 0.38690850138664246, 0.7366601228713989, 0.472453236579895, -1.5783991813659668, -0.04891880229115486, -0.2513831853866577, -0.8397697806358337, 0.21699295938014984, 0.1102939322590828, -0.2514530122280121, -1.4220088720321655, 1.3038737773895264, 1.7613277435302734], [1.8721214532852173, 0.47706177830696106, 0.2212570309638977, 0.41473132371902466, 0.012514335103332996, -0.6864796876907349, 0.4125426709651947, 1.1048171520233154, -0.303370863199234, 1.4361019134521484, 0.05460594594478607, -0.20969602465629578, -1.338902235031128, 0.7137719988822937, -0.12637576460838318, 0.11676070094108582, 0.31531527638435364, 0.6812131404876709, 0.9767925143241882, 2.7422902584075928, 1.785828948020935, -1.5518206357955933, -0.06821833550930023, 0.710247278213501, 0.4804232120513916, -0.5767115950584412, 0.9757390022277832, -0.8324673175811768, 1.1501684188842773, -0.3283328711986542, 0.4633027911186218, -0.8289011120796204, 0.4920460283756256, 0.5792191028594971, 0.5042728185653687, -1.3237793445587158, 0.7377601265907288, 1.4820207357406616, 1.0264201164245605, -0.5094977021217346, -0.5869675874710083, 0.6085849404335022, 0.5724745988845825, -0.7358440160751343, -0.09947732090950012, -1.5700194835662842, 0.2571950852870941, -1.212960958480835, -1.4196326732635498, 0.057463858276605606], [-1.1347843408584595, -0.8783043026924133, 0.3462214767932892, -1.3331314325332642, 0.33088818192481995, -0.5054306983947754, 0.24025411903858185, -0.9333522319793701, 0.22214406728744507, 2.1121490001678467, -0.23079794645309448, -0.6335833668708801, -2.693505048751831, 1.5347070693969727, 1.201977252960205, 0.9341517686843872, 0.23105214536190033, -1.168864369392395, -1.1758676767349243, 0.23218168318271637, 1.0444475412368774, 1.0377583503723145, 0.11218462139368057, 0.8337341547012329, 3.0070273876190186, 0.32529184222221375, -0.3230976164340973, 0.7080478668212891, 0.38579678535461426, -0.08018037676811218, -0.5381625890731812, 1.523825764656067, 0.571584939956665, 1.8683582544326782, 0.4559931457042694, -0.025877108797430992, -0.4834684729576111, -0.2241656333208084, -0.5914570093154907, -0.8860446214675903, 2.371731996536255, -0.7177395224571228, 0.31301236152648926, -0.07215725630521774, -0.7739130258560181, 0.5370174050331116, -0.6607946753501892, -0.01552191935479641, -0.6849875450134277, -0.235868439078331], [0.7074315547943115, 0.1864742785692215, -0.08452802896499634, -1.7247675657272339, -0.2221619188785553, 0.3188636898994446, 0.0460275337100029, 1.2075858116149902, -0.7788944244384766, 0.8487532734870911, 0.9848702549934387, 1.5810022354125977, 1.0109769105911255, -0.4600547254085541, 0.31880125403404236, -0.8974292278289795, -0.7683585286140442, 0.11637184768915176, -0.3006962537765503, -1.7464796304702759, -0.13358336687088013, 0.12794789671897888, -0.3465403914451599, -1.1490375995635986, 0.3653581142425537, -1.6644213199615479, 0.71102374792099, 0.4587205946445465, -0.3687838912010193, 1.180892825126648, 2.427128314971924, -1.4350556135177612, 3.0121631622314453, -0.9160420298576355, 0.18705150485038757, -0.9658995866775513, 1.4027493000030518, 1.335868239402771, 0.739508867263794, -0.08468367904424667, -1.531703233718872, -0.5128681063652039, 0.5736358165740967, -1.0104750394821167, -1.3509776592254639, -0.25517019629478455, -1.0832263231277466, 0.7197386622428894, -2.5480337142944336, -2.0688316822052], [-0.24383080005645752, 0.19595977663993835, 0.5086770057678223, 0.8093763589859009, -1.3194687366485596, -0.08729375898838043, -0.17898127436637878, -2.187352180480957, 1.3686552047729492, -0.38423851132392883, -1.1545541286468506, 0.9664652347564697, 1.7165663242340088, -0.027502046898007393, -1.9292361736297607, 0.002891498152166605, 0.4271262288093567, 0.38725560903549194, -0.6690965890884399, 2.0406312942504883, 0.16791072487831116, -1.5523147583007812, 2.239041328430176, 0.024214768782258034, 0.7059259414672852, 0.752334713935852, 0.23593419790267944, 2.1415770053863525, 2.405034065246582, -1.2726308107376099, -0.30002138018608093, 0.0854167565703392, -0.4520319104194641, -0.330121785402298, -1.523206353187561, 0.23559248447418213, 0.7740203142166138, 0.5300425887107849, 0.5925338268280029, 0.4434700310230255, 0.7399482727050781, -0.40624961256980896, 0.27955108880996704, 0.9591022729873657, 0.5230389833450317, 0.9566441774368286, 0.9974808096885681, 2.059363842010498, -0.8447833061218262, 0.40081122517585754], [0.33625900745391846, -0.2500150799751282, -1.369166374206543, -0.45149683952331543, -0.13992534577846527, 1.0539839267730713, 0.9572040438652039, -0.006316834595054388, 1.651762843132019, 1.1141623258590698, 0.8046355247497559, -1.9342378377914429, 0.5152010917663574, -2.111771821975708, 0.9693052172660828, -0.7362279891967773, -0.4704338610172272, 1.3099172115325928, -0.09809327870607376, 1.3463445901870728, 1.7055503129959106, -0.4286310374736786, 0.08077171444892883, 1.6075137853622437, 0.5604603886604309, 1.1365090608596802, -0.3143640458583832, -0.6265687942504883, 1.198891282081604, 0.12237443029880524, 1.032938838005066, -1.432288646697998, -0.18614435195922852, -0.03976251929998398, -2.282787322998047, -0.03277045860886574, -0.6669425964355469, -2.336409568786621, 2.4007551670074463, -2.067124366760254, -0.8446318507194519, 0.005849225912243128, -1.118632435798645, 0.3052087724208832, 1.2943077087402344, 0.8609392046928406, -0.9440411329269409, -0.6419950723648071, -0.9307560324668884, 1.0310683250427246], [-0.9892227053642273, 0.9034432172775269, -1.4151952266693115, 0.6166813373565674, 1.1462326049804688, -1.1220208406448364, 0.7261725664138794, 0.09091460704803467, -0.4420568346977234, 0.5565149784088135, 1.5454083681106567, -0.006032893434166908, 1.4903517961502075, -0.2599658966064453, 1.3753327131271362, -1.3266923427581787, -0.6419159770011902, -0.5802133679389954, 0.9137206077575684, 0.3234652578830719, 0.3936518430709839, 0.10496483743190765, -0.6144333481788635, 0.18716321885585785, -1.2896777391433716, -1.0046343803405762, 0.35872209072113037, -0.6127650737762451, -0.5947835445404053, -1.2752318382263184, -0.349999338388443, -0.6452031135559082, -0.02789265103638172, 0.19813339412212372, -0.7899273633956909, -1.1677618026733398, 0.8429938554763794, 1.9081370830535889, -0.9071237444877625, -0.04924788326025009, 0.23021183907985687, 0.47120440006256104, -0.4919814169406891, -1.3296833038330078, 0.8518815636634827, -1.1783850193023682, -1.8480274677276611, 0.1134871393442154, 0.09589114040136337, 0.6757953763008118], [0.9516214728355408, 0.4888175129890442, -1.804687738418579, 0.5229776501655579, 0.5718135833740234, 0.5459193587303162, 0.7595729231834412, 0.5755571722984314, -0.5922728776931763, -0.5069499015808105, 0.11503284424543381, 0.09770950675010681, 1.2358077764511108, -0.6754658818244934, 0.2916754484176636, 0.45971137285232544, -0.18337340652942657, -0.08725789189338684, -1.1426292657852173, -0.19392025470733643, -0.3994760811328888, 2.1967556476593018, 2.2666914463043213, 0.3092086613178253, -0.18445754051208496, -0.3953980505466461, -0.6399937868118286, -0.9022952914237976, -0.16550341248512268, -0.938649594783783, 1.5290806293487549, 2.032240390777588, 0.4489549696445465, 1.631435513496399, -1.6949427127838135, 1.623600721359253, -1.5088673830032349, 0.7265759110450745, -0.6743375062942505, -0.8166723251342773, -1.5219194889068604, -0.5627012252807617, -1.5520886182785034, 2.9688074588775635, 0.9587752819061279, -2.3874669075012207, 0.4304404854774475, -1.2041242122650146, 0.6440552473068237, 0.027495695278048515], [0.5860118865966797, -1.3358958959579468, 0.5461649894714355, -0.13761277496814728, -0.17824193835258484, 1.576758861541748, -0.33270856738090515, -0.17333149909973145, -0.20706653594970703, 0.6601279973983765, -1.8042278289794922, -1.422723650932312, -2.3888449668884277, -0.4051752984523773, -0.43335577845573425, 0.8393136262893677, 0.2202054113149643, 2.5396432876586914, -0.3870890736579895, -1.4716758728027344, -1.5009727478027344, 0.819591760635376, -0.01734856516122818, -0.028817109763622284, 0.0783914178609848, 0.9797849655151367, -1.7644822597503662, -0.1764647513628006, 0.04106985405087471, -1.2491974830627441, 0.15362222492694855, -1.7588139772415161, -0.03624362498521805, 0.5997241139411926, -1.5880168676376343, -0.2900104224681854, -0.6025481820106506, 0.6086460947990417, -0.38267919421195984, -0.25107064843177795, -0.09355485439300537, -0.8374363780021667, -3.0453076362609863, -1.194606900215149, 0.30855080485343933, 1.4969264268875122, 1.212843418121338, 0.7428429126739502, -0.43196216225624084, 0.18667064607143402], [0.5933001637458801, -0.7676787376403809, -0.2952705919742584, 0.20952409505844116, -0.11602799594402313, 0.1597328931093216, -0.0188607145100832, 1.6914198398590088, 1.8444113731384277, 1.0965296030044556, 1.2774794101715088, -0.2210708111524582, -1.9133131504058838, 0.9851023554801941, 0.5758922696113586, -1.521328091621399, 0.8148145079612732, 0.15329474210739136, 0.5997889041900635, 0.9974714517593384, 0.789165735244751, 1.3444148302078247, 0.6857297420501709, 0.8957391381263733, -1.5453096628189087, -1.3991563320159912, -0.818630039691925, 0.3411581218242645, -0.759716272354126, 1.008109450340271, -1.427865982055664, -0.30671802163124084, -0.7111170887947083, -1.337664008140564, -1.3692240715026855, 0.40330517292022705, 0.14458595216274261, -1.9208717346191406, 0.32330143451690674, -0.24856609106063843, -0.08391207456588745, -1.1507651805877686, -0.5818400979042053, -0.7314302325248718, 0.21134278178215027, 0.8184105753898621, -0.3921002745628357, 1.5669127702713013, 0.23296387493610382, -0.5065225958824158], [1.2171152830123901, 0.9453398585319519, 1.1684210300445557, -0.2385861575603485, 1.1145997047424316, -0.2356678694486618, -2.5408122539520264, 0.34305423498153687, -0.5328109860420227, -0.8768041133880615, -0.4674392342567444, -1.6481660604476929, 0.555151641368866, 0.4796181321144104, -0.49675649404525757, 0.1329992562532425, -0.2588334083557129, -0.7102725505828857, 1.130462884902954, 0.6682116389274597, 0.254055917263031, -0.6080504059791565, 0.26382848620414734, -0.27004188299179077, -0.08048474043607712, -0.043865565210580826, -0.20042942464351654, 0.3626028001308441, 0.47848081588745117, 0.013241644017398357, 0.25317615270614624, -1.5118708610534668, -0.773615300655365, 1.1280415058135986, 1.6558181047439575, 0.3337720036506653, -1.6735087633132935, 0.38885846734046936, -0.3081958293914795, 0.5847465395927429, -0.5024855136871338, -0.6564361453056335, -0.2695750296115875, 1.7802406549453735, 0.8331274390220642, 1.0901542901992798, -1.3123058080673218, 0.8180710673332214, -0.7091953158378601, -0.17060615122318268], [0.13026058673858643, -0.3239325284957886, 0.8197145462036133, 0.4459439814090729, -0.7347607016563416, -1.159143090248108, 0.8420801758766174, -0.07619834691286087, 0.0030574221163988113, 1.1254435777664185, 1.7743948698043823, 0.08179683238267899, 1.3957858085632324, 0.6096082329750061, 0.799565315246582, 1.0448179244995117, 0.9764506220817566, 0.2881815731525421, 0.392909973859787, -0.08440208435058594, -0.9332201480865479, -0.5394158959388733, 0.548929750919342, -0.13916440308094025, -0.17164960503578186, -0.47128888964653015, -1.2421634197235107, -0.09385206550359726, 2.167567253112793, -0.8422312140464783, -0.4145737886428833, -0.13229338824748993, -2.064345121383667, 1.0585087537765503, 1.6317052841186523, -1.5373114347457886, -0.06506829708814621, 0.6581884026527405, 0.8450308442115784, 0.6118930578231812, -0.3393046259880066, 1.9228134155273438, 1.839762806892395, 1.094246745109558, -0.4389977753162384, -0.5769278407096863, -1.7831315994262695, 1.9475561380386353, -1.3656933307647705, 1.7046438455581665], [0.31161242723464966, -0.45300692319869995, -1.7119615077972412, 1.1600499153137207, -1.4202760457992554, 0.7339221239089966, -0.020517686381936073, 0.3725070059299469, -0.005991172976791859, 0.6826841831207275, -0.4697280526161194, -1.00529146194458, -0.20384614169597626, -1.4410768747329712, -0.5718653798103333, -0.8223726153373718, -1.267394781112671, -1.0860460996627808, 0.12836270034313202, 1.3478702306747437, 0.840717077255249, -0.23572808504104614, 1.3589059114456177, 1.8438160419464111, -0.45966416597366333, 0.4957677721977234, -0.8940050601959229, -0.5956977605819702, 0.20965850353240967, -1.9385889768600464, 0.32367950677871704, 0.19139134883880615, -0.4192635715007782, -1.5117260217666626, 0.41738399863243103, 0.7425286173820496, -0.3798225522041321, -0.19206613302230835, 1.2005711793899536, 1.7149306535720825, -0.6695691347122192, -0.8992399573326111, 0.4257996082305908, -1.3460841178894043, 1.77567458152771, -0.4496210217475891, 0.5000278353691101, 0.7236065864562988, -0.36642980575561523, 0.8287714719772339], [0.9750484228134155, 0.9968124032020569, -1.0347027778625488, -0.21646326780319214, -0.34600332379341125, -0.3323284089565277, -0.5070212483406067, 1.148950457572937, -1.2932038307189941, 0.1276731789112091, -2.357815742492676, -1.8559610843658447, 2.9200010299682617, 0.8604548573493958, 0.35685670375823975, -1.2522348165512085, 0.006367255002260208, -1.3115373849868774, 2.3011634349823, 0.49351364374160767, 0.18168328702449799, 0.058764152228832245, 0.020850950852036476, 0.10362407565116882, -0.768466055393219, 1.3524965047836304, -0.024490362033247948, 1.094936490058899, -1.2598495483398438, 0.4959775507450104, 1.6626648902893066, 0.07538005709648132, 1.4108136892318726, 0.8142896890640259, 0.05100078135728836, 0.03716157376766205, 1.4316946268081665, 0.8348436951637268, 0.8299329876899719, 0.9009752869606018, 0.559562087059021, 0.2600940465927124, -0.08686991035938263, 0.5345415472984314, 0.9291606545448303, -0.4966270327568054, 1.1658657789230347, 0.7817546725273132, -0.30824729800224304, -1.314348816871643], [-1.8191488981246948, -0.3788280785083771, 0.6520504951477051, -0.9234529733657837, -0.8681440353393555, -1.605859398841858, 0.49324899911880493, -0.16805289685726166, 1.6871061325073242, -1.0735033750534058, -0.26537856459617615, -0.06748759001493454, 3.014894723892212, -0.16721321642398834, -1.452223777770996, -0.2614857256412506, 0.8614343404769897, -0.06297643482685089, 1.317683219909668, 0.6308490633964539, -0.19905999302864075, -1.5232155323028564, 0.6209851503372192, 1.55419921875, 0.13860958814620972, -0.07728852331638336, -2.2485530376434326, -0.7727380990982056, -2.5083281993865967, -1.1998578310012817, 0.7411932945251465, 0.11084204912185669, -1.2432869672775269, 0.5323168635368347, 0.6230180263519287, 0.6265855431556702, 0.7536498308181763, 0.6135632395744324, -2.65333890914917, -0.7497868537902832, -1.7966915369033813, -0.7925037145614624, 0.9528340697288513, -0.1007474809885025, 0.4039134979248047, 1.1179279088974, -0.7599010467529297, -0.242445707321167, -0.5864659547805786, 0.6673561930656433], [0.47540467977523804, 0.4665287435054779, 0.6104158163070679, 0.3397122919559479, 0.5131535530090332, 1.6043305397033691, 0.5262947082519531, 0.33969947695732117, 0.6453188061714172, -0.24305564165115356, 0.8863562941551208, -1.9012658596038818, 0.22669297456741333, 1.1486878395080566, 1.4584088325500488, -0.019109725952148438, 0.5934128761291504, 0.30509883165359497, -1.6544369459152222, 0.528037965297699, -1.2073084115982056, -0.35767310857772827, -0.622617244720459, -0.7149064540863037, -1.4277434349060059, -0.3932921588420868, -0.5239421129226685, -1.0255895853042603, 1.5012283325195312, -1.128313422203064, 2.207954168319702, 0.32465100288391113, 1.9957914352416992, -2.1441080570220947, 0.1415441334247589, 0.3244612216949463, 1.5127766132354736, 1.1570172309875488, -0.3825685381889343, -1.0887813568115234, -0.9000867605209351, 0.6334351301193237, -0.5973252654075623, -1.9132314920425415, 0.9386897087097168, 1.3053864240646362, -0.46040087938308716, -1.7093379497528076, 0.46767330169677734, 0.4171428084373474], [-0.7644041180610657, 0.40336915850639343, -0.12163962423801422, 0.06369738280773163, 0.8253206014633179, -0.1941417157649994, 0.935703456401825, -0.7976208329200745, 0.3678155243396759, 0.6267343759536743, -0.574927568435669, -0.13785342872142792, -1.652496099472046, -0.11274100840091705, 1.46453857421875, -0.2254623919725418, -2.6840672492980957, 0.5878673791885376, 0.08076059818267822, 0.5030307769775391, 1.7524654865264893, 0.6973145008087158, 0.4380243718624115, -1.779984474182129, 0.09612441062927246, 0.19638098776340485, -0.2193244844675064, -0.419921875, -0.5455467104911804, -0.7823826670646667, -1.312395453453064, 0.9204605221748352, -1.0140960216522217, 1.0421431064605713, -1.8320614099502563, -0.7100114822387695, 0.39706429839134216, -0.1893424093723297, 0.1363590508699417, 0.27617812156677246, 2.0849051475524902, -1.0479981899261475, 0.7168412208557129, -0.06331167370080948, -0.863084077835083, 0.7023965120315552, -1.0336060523986816, -1.5562409162521362, 0.061688754707574844, -0.16569118201732635], [-1.196328043937683, -0.438267320394516, -0.9374651908874512, 1.0159670114517212, -0.3449999988079071, 2.478349447250366, 0.8680616021156311, 1.7121034860610962, -0.5387057065963745, 0.14049692451953888, 0.6608187556266785, -1.3746695518493652, 0.014883262105286121, -1.9386056661605835, -0.8610669374465942, 0.5907076001167297, -1.7544583082199097, 1.4720264673233032, -0.10004088282585144, 0.21443480253219604, 0.26002243161201477, 1.261051893234253, -2.194399833679199, 1.1238353252410889, 1.7906662225723267, 0.3655408024787903, -0.9647160768508911, -1.1012297868728638, -0.43048420548439026, 1.3183099031448364, -0.30778685212135315, -0.6869386434555054, -1.0719835758209229, 0.9135949015617371, 0.01741785556077957, -0.9785722494125366, -1.0652810335159302, 1.6958585977554321, -1.4002721309661865, 0.37251147627830505, -0.3085152506828308, 0.7621947526931763, 1.8628525733947754, 1.2774274349212646, -0.2622454762458801, -0.2705998122692108, 0.282366007566452, -2.783641815185547, 1.1630650758743286, 0.3654652237892151], [2.602886438369751, 0.3035064935684204, -0.26301807165145874, -1.3255558013916016, 0.711492121219635, -0.0733538419008255, -0.0948953628540039, 0.37481337785720825, -0.7347272038459778, -2.321058750152588, 0.11476484686136246, 0.1991972029209137, -0.30728259682655334, -1.8654825687408447, 0.11024750024080276, -0.4110621511936188, -2.181002616882324, 0.2766070067882538, -2.0344133377075195, -0.8812565803527832, 0.697045624256134, -0.33179640769958496, -0.9705292582511902, 1.1180181503295898, -0.4534396231174469, 0.8063833713531494, 0.9621663689613342, -0.9678592681884766, 0.606805145740509, 1.0899543762207031, -0.9152448773384094, 0.557654082775116, -0.7471842765808105, 0.2421523928642273, -0.3505365550518036, 0.25371551513671875, -0.6867471933364868, 0.3231123685836792, 1.6857993602752686, 1.7344082593917847, -0.20364466309547424, 0.724907636642456, -1.3057096004486084, -1.7209537029266357, -0.8441126942634583, 0.6363702416419983, -0.22721143066883087, 1.0570424795150757, 0.5410723686218262, -0.27086326479911804], [-0.369803786277771, 1.7278977632522583, -0.8916212916374207, 0.3834211826324463, -1.155185580253601, -0.4529455602169037, -2.0808265209198, 0.08323553204536438, 0.6054298281669617, -0.552621066570282, 1.5436772108078003, 0.2832805812358856, -0.726475715637207, -1.1025365591049194, 1.3144530057907104, 0.293451189994812, 1.545610785484314, -0.10176218301057816, 1.2889370918273926, 0.3936477601528168, 1.2444895505905151, -0.09249287843704224, -0.8773882389068604, 0.0497799851000309, 2.110281467437744, 0.8075231313705444, 0.23801854252815247, 0.019801484420895576, 2.346491813659668, 0.1556384414434433, 0.3075758218765259, -0.15091608464717865, -0.6182922124862671, 0.24112015962600708, 0.7208699584007263, 0.06005702167749405, 0.5635924935340881, 0.28317010402679443, -0.5895947813987732, -2.626077890396118, 1.482499122619629, -1.8910844326019287, -1.0593783855438232, -0.6884863972663879, 0.132127583026886, 0.900721549987793, 0.25086426734924316, -0.36600881814956665, -0.45643362402915955, -0.22024677693843842], [0.9049060940742493, 1.1831549406051636, -0.21005964279174805, 0.8327051401138306, 0.5467255115509033, 0.0518997460603714, 0.34892091155052185, 1.6149717569351196, -0.4785013198852539, -1.4695838689804077, 2.204955816268921, -0.8575074672698975, -0.6759755611419678, -0.44906309247016907, 1.3260319232940674, -1.4706052541732788, 0.0834115669131279, 0.9553254246711731, 1.553328037261963, -0.5610603094100952, -0.37655603885650635, -0.050004370510578156, -1.2696384191513062, -0.15296882390975952, -0.18641318380832672, 0.04352623596787453, -0.12171632796525955, 0.03508785367012024, 0.7224028706550598, 0.7080456018447876, 0.6460744738578796, -1.3467696905136108, -0.0004552202590275556, -0.41709670424461365, -0.3681618869304657, 0.5880953669548035, 1.45542311668396, -0.7086297869682312, 0.06433461606502533, -1.5980901718139648, 1.4864261150360107, 1.3774172067642212, 0.9484905004501343, 1.0095537900924683, -0.46185681223869324, -1.178817629814148, -0.38786837458610535, 0.4958593547344208, 0.920204758644104, -0.456038236618042], [-0.19067949056625366, 0.5431271195411682, 0.4968881607055664, -1.1210910081863403, -0.6858059763908386, -1.4786604642868042, -0.20314301550388336, -0.946769654750824, 0.796835720539093, 0.3716874122619629, 0.38971710205078125, 0.7053523659706116, 0.16003401577472687, 1.2334225177764893, -1.0954082012176514, -1.1870815753936768, 1.0918959379196167, -0.07413363456726074, 1.6497970819473267, -1.051274061203003, 1.1215617656707764, -0.739675760269165, 0.3525974154472351, -0.16626587510108948, 1.128659963607788, -0.13255484402179718, 0.9543246030807495, 0.13836902379989624, 0.45274391770362854, 1.5726757049560547, -0.6399295926094055, -1.2502444982528687, 0.09616884589195251, -0.2654489278793335, 0.3618348240852356, -0.13790947198867798, -0.2836529612541199, -0.047791484743356705, 1.6616661548614502, 1.8211699724197388, 2.848409652709961, -0.20748727023601532, 0.4701235890388489, -0.24023433029651642, 0.7135583758354187, 0.32849618792533875, -0.7670748233795166, -0.24357563257217407, -0.17656917870044708, 0.9007766246795654], [-0.7451869249343872, -0.3271649181842804, 1.595462679862976, -1.6907520294189453, 0.38934090733528137, 0.38963502645492554, -0.37625494599342346, 0.8721588850021362, 0.7813990116119385, -1.1500152349472046, -0.16610412299633026, -0.4061080515384674, -0.2525133788585663, -1.425921082496643, -1.6746413707733154, -1.0075923204421997, 0.07782712578773499, 0.03772682696580887, -1.6244173049926758, -0.9954590797424316, 0.008000515401363373, -0.507759153842926, 1.2962003946304321, -0.6680095195770264, 0.9359002113342285, 1.211479663848877, 0.9972441792488098, 0.7384680509567261, -2.3912651538848877, 2.1437816619873047, -0.9084041714668274, 1.6775262355804443, 1.0092390775680542, -0.29668229818344116, -0.032639291137456894, 0.37333089113235474, -1.3453110456466675, -0.836829662322998, 0.1159457266330719, 1.819782018661499, -1.3380868434906006, -0.2928721010684967, 0.5461346507072449, -1.389777660369873, 2.125490665435791, -0.7597867250442505, -0.08024245500564575, -0.4691866338253021, -0.38312697410583496, 0.1401437371969223], [1.1740611791610718, -0.2799369692802429, 0.6968728303909302, -0.8568754196166992, 0.759416937828064, -0.2938353717327118, 0.12544210255146027, -1.8960130214691162, 0.9324209094047546, -1.0335925817489624, -1.3177753686904907, 0.7401193976402283, -0.0959627702832222, 0.1806173175573349, 2.395698308944702, -1.878165364265442, 0.021109556779265404, 1.1934547424316406, -0.7442479133605957, -1.9375308752059937, 1.0466631650924683, 0.4312545359134674, -0.3059682846069336, 0.6600979566574097, 0.6193650364875793, 0.42968106269836426, 1.1025339365005493, 1.078926682472229, 0.2374013513326645, 2.4998779296875, -0.07960686832666397, -0.5453230738639832, 0.36012595891952515, -2.736332893371582, -0.7246825695037842, -0.666635274887085, -0.10365992784500122, -1.294977068901062, 0.7454490661621094, -0.6912915110588074, -0.11774113774299622, -0.46362143754959106, 0.21744635701179504, -0.782783567905426, 0.5628504753112793, -1.3209953308105469, 1.0431036949157715, -0.8992328643798828, -0.19958436489105225, -0.6734799742698669], [2.569257974624634, 0.7752550840377808, -1.4360154867172241, -0.5851970314979553, -0.7127944231033325, 0.15088705718517303, -0.9102725386619568, -0.8252868056297302, -0.7719985842704773, -0.6264172792434692, 0.7102393507957458, -0.25841549038887024, -0.2892570197582245, 0.7010968327522278, -1.229552984237671, 0.7283388376235962, 0.8705869317054749, -1.2299926280975342, 1.0472993850708008, -0.07637593895196915, 0.20638306438922882, 1.8000622987747192, -0.7757465243339539, 0.4116893410682678, -0.24841295182704926, 0.9673092365264893, -0.990385890007019, 2.1365466117858887, 0.7339971661567688, 1.1546785831451416, -0.5525583028793335, -1.7279834747314453, -1.290213942527771, -1.547139048576355, 0.9021161794662476, 1.3663859367370605, 0.24497824907302856, 1.4477659463882446, -0.3396914601325989, 1.0235013961791992, 0.4032747447490692, 0.03929394483566284, 1.224328875541687, 1.3541204929351807, 0.2653360962867737, -1.2292454242706299, -0.23785677552223206, -0.6686041355133057, 0.09385491162538528, 0.6818994283676147], [0.7717533111572266, -0.43732163310050964, 0.28679290413856506, 0.3372661769390106, -1.181162714958191, 1.0772316455841064, -0.2106904834508896, 0.757020890712738, -0.7847585678100586, -0.041997380554676056, 0.9393075704574585, -0.7238654494285583, 0.5290493965148926, 0.6754623055458069, -0.9002118706703186, 1.300455927848816, 0.17639866471290588, 1.0774896144866943, -3.011341094970703, 0.014081486500799656, -1.3774702548980713, -0.16365882754325867, 0.45844167470932007, 0.575640857219696, -0.8252514600753784, -0.00039088455378077924, 1.92697012424469, 1.0936315059661865, -1.785441517829895, 0.9842174649238586, -0.0879441574215889, 0.053840525448322296, 0.018921835348010063, 0.69056636095047, -1.5758564472198486, 0.8656759858131409, 1.8578743934631348, 1.747946858406067, 0.5161390900611877, -0.6967349648475647, -0.46635472774505615, 1.3021961450576782, -0.5915070176124573, 0.1467781811952591, -1.4883171319961548, 0.09600479155778885, -0.3334795832633972, 0.21125051379203796, 1.0760196447372437, 0.5720446705818176], [0.2754766047000885, -1.5385953187942505, 0.18671488761901855, 2.5993263721466064, -0.5601039528846741, -1.9405534267425537, -1.2193961143493652, 2.3049778938293457, 0.3399839997291565, -1.6328045129776, -0.24007117748260498, -0.11461807042360306, 1.2271214723587036, -0.25313717126846313, -0.278226763010025, 1.3735406398773193, -0.803699791431427, 1.2333778142929077, 0.5492779016494751, 1.547725796699524, 0.017427071928977966, -0.16493302583694458, 0.6965530514717102, 0.12034318596124649, -0.8964018225669861, -0.7680503726005554, -0.10523893684148788, 1.1341075897216797, -0.26217883825302124, -1.7750686407089233, -0.4867137670516968, -1.2134604454040527, 1.6656849384307861, 0.010368947871029377, -0.7639133930206299, 0.6494563221931458, -0.8009252548217773, 1.2630118131637573, -0.09696100652217865, -0.888423502445221, -0.10892818868160248, -0.3522150218486786, 0.034388430416584015, 0.6836258172988892, 1.845322847366333, 1.058450698852539, -0.2398218810558319, -0.722018837928772, 0.0343250147998333, -1.0840409994125366]], "dtype": "float32"}], "e": {"__nd__": [[2.261154890060425, -3.650606155395508, -0.4024428725242615, -7.036079406738281, 2.7237722873687744, 6.217710018157959, -7.028805732727051, 1.32918119430542, 0.4705851674079895, 4.616306304931641, 1.9381282329559326, -12.264623641967773, -5.9055585861206055, 8.652664184570312, -8.682901382446289, 3.2615532875061035, 11.367788314819336, -4.862647533416748, -12.869488716125488, 2.1970248222351074, -8.224491119384766, -0.13297992944717407, 2.910421133041382, 2.580338478088379, -1.7511204481124878, -5.116264820098877, -8.408616065979004, -0.17661911249160767, -0.7185983657836914, -0.9206975698471069, 2.820956230163574, 1.9684374332427979, 1.3779067993164062, -4.197653770446777, 14.24251937866211, 2.697134017944336, -6.852694034576416, -4.14479398727417, 2.3924691677093506, 9.743391990661621, -3.179847002029419, 0.2812860310077667, -5.615426540374756, -5.912916660308838, 7.362284183502197, 3.7561423778533936, -9.241299629211426, 5.08798885345459, -7.6852946281433105, 5.579455375671387], [-2.0073471069335938, -1.1489073038101196, 0.5992023944854736, 5.928345680236816, -4.26691198348999, 7.571145057678223, -3.670905828475952, 7.52782678604126, 3.1657636165618896, 3.943457841873169, -3.2049946784973145, -8.322200775146484, -4.448667049407959, -1.584213137626648, -3.647451877593994, 2.6242363452911377, -1.703607439994812, -2.889037609100342, 6.085562705993652, 11.036849975585938, 4.252233505249023, -0.3650437295436859, -7.120357990264893, 6.298932075500488, 1.0153586864471436, 6.756668567657471, -3.223586082458496, -2.357896327972412, -0.9520705938339233, -4.459623336791992, 1.5479687452316284, -11.704610824584961, 1.0650060176849365, -3.625316858291626, 2.1328489780426025, -2.660788059234619, 0.10711146891117096, 8.23820686340332, 2.3853609561920166, 5.0744805335998535, 0.47943946719169617, -2.401214599609375, -0.605089545249939, -3.876863956451416, 8.198773384094238, 5.642117977142334, 2.6574301719665527, -4.910896301269531, 1.41569185256958, -0.05052033066749573], [-1.4445858001708984, -6.2759857177734375, 14.720248222351074, 3.619077205657959, 3.723039150238037, 5.115290641784668, 3.3941597938537598, 1.572856068611145, 6.41395902633667, 4.156681060791016, -4.85046911239624, -1.1706507205963135, -1.0981175899505615, 0.760757565498352, -4.477478504180908, -0.6717203855514526, 0.1605541706085205, 16.339879989624023, -10.01199722290039, -9.308815956115723, -8.543502807617188, 0.7458290457725525, -0.9690027832984924, -1.3058699369430542, -5.9853081703186035, 2.4212050437927246, -1.6238104104995728, 2.2774581909179688, -6.389348983764648, -3.0801548957824707, -9.387107849121094, -6.3701958656311035, 4.22280216217041, 3.1418702602386475, -13.45005989074707, -6.42544412612915, -9.290459632873535, -1.7583634853363037, -6.2501397132873535, 2.320460796356201, -2.4600110054016113, 6.308475971221924, -4.490957736968994, 3.752941370010376, 0.5393681526184082, 7.247650146484375, 4.143335342407227, -9.80021858215332, 9.604491233825684, 1.4010599851608276], [2.091139793395996, -0.10678577423095703, -1.2975784540176392, 2.9102988243103027, 1.774092197418213, -2.8234729766845703, -0.6285768747329712, -1.9515224695205688, -9.056562423706055, 11.985102653503418, 7.858948230743408, -2.583737373352051, -5.247984409332275, 7.660311222076416, 2.1612539291381836, 1.3544930219650269, 6.510441303253174, 3.866481065750122, 6.011491298675537, 2.1290223598480225, 0.44493624567985535, -10.352588653564453, -6.641417980194092, -1.7605122327804565, 8.50181770324707, -5.543174743652344, 8.396408081054688, -2.6084771156311035, 11.382390022277832, 1.028824806213379, 6.809564590454102, -10.397222518920898, 5.925470352172852, 1.697887659072876, 11.260029792785645, -13.035304069519043, -0.039978884160518646, 3.748101234436035, 4.116901397705078, -6.690070629119873, 0.9471146464347839, 7.946979999542236, 2.5597386360168457, -0.702322244644165, -1.9607880115509033, -14.409908294677734, -6.797440052032471, -8.78395938873291, -9.904723167419434, 2.652951955795288], [-7.963309288024902, -6.556532859802246, 0.055757686495780945, -3.8164031505584717, 1.4528049230575562, -4.3365864753723145, -0.23257359862327576, -5.06874418258667, -14.40036392211914, -4.190131664276123, 7.302218914031982, -0.674845814704895, -7.64225959777832, 0.05397418141365051, -2.217545747756958, 12.396156311035156, 1.1546745300292969, 1.603252649307251, -10.210926055908203, -14.231036186218262, -13.211339950561523, 5.595506191253662, 3.9259135723114014, -7.728691577911377, 13.33268928527832, -0.8857914209365845, 2.4152729511260986, -7.544816017150879, -6.836830139160156, 1.7453758716583252, 1.677269458770752, 3.251737594604492, -10.168097496032715, 13.803973197937012, -2.246061325073242, 4.72009801864624, -2.5666842460632324, 9.17416000366211, -13.376973152160645, -2.6168196201324463, 3.3039677143096924, 2.0921244621276855, -3.3445022106170654, 5.6269965171813965, -7.580056667327881, -5.115139961242676, -2.9099810123443604, -3.6924428939819336, 2.4866111278533936, 2.3773152828216553], [0.01506652869284153, 2.6647374629974365, -0.524480938911438, -3.8764560222625732, -6.416330337524414, 1.7600525617599487, -3.037616729736328, 0.9928067922592163, -6.341042995452881, -1.9756923913955688, 4.6717352867126465, -11.700996398925781, -1.5184428691864014, -8.73925495147705, 10.37513256072998, 0.10651402175426483, -1.2981268167495728, -4.585823059082031, 3.47892165184021, 4.792634010314941, 10.192801475524902, 2.7770516872406006, 3.7384631633758545, -1.3723727464675903, 8.30074691772461, -2.5225512981414795, 6.933320999145508, -3.352029800415039, 5.9695048332214355, 1.3625692129135132, 10.453132629394531, -5.924108028411865, 1.971325159072876, 0.3137621283531189, -2.8221404552459717, 4.855781078338623, -2.334331512451172, 3.6698083877563477, 4.071254253387451, -12.660937309265137, 0.7423890829086304, -11.172295570373535, 0.5918235778808594, -6.628360271453857, 3.5639634132385254, 3.7240865230560303, -3.0190327167510986, 11.713971138000488, -14.734416961669922, -3.7468044757843018], [-4.827801704406738, -3.932671546936035, 3.4258382320404053, 1.1689481735229492, 1.258786678314209, -9.651508331298828, 1.4945197105407715, -6.775325298309326, 12.39857006072998, 6.34487247467041, -6.4491400718688965, 0.29125022888183594, 9.203031539916992, 10.402504920959473, -3.5264875888824463, -0.005160093307495117, 8.677967071533203, -4.122471332550049, 2.5926594734191895, 6.67093563079834, 0.9335841536521912, -9.795537948608398, 8.17616081237793, 1.7601299285888672, -5.0035624504089355, -5.958174705505371, -6.78761625289917, 4.379019737243652, 0.9614945650100708, -11.741010665893555, -1.461416482925415, 3.78318452835083, 5.267578601837158, -4.728855609893799, 6.7141194343566895, -1.6012370586395264, -3.061516761779785, -8.831315994262695, 0.2855954170227051, -4.164731502532959, -7.082239151000977, -1.415954828262329, 2.175248146057129, -1.29591965675354, 8.657771110534668, 4.187953472137451, -6.295943260192871, -2.529412269592285, -8.701791763305664, 3.0470938682556152], [-14.215886116027832, -1.603550910949707, -5.862980842590332, 3.532292604446411, 3.346930980682373, -3.664266347885132, 2.0271191596984863, 8.208558082580566, 1.3237931728363037, 4.366771221160889, -1.2182435989379883, 2.96333909034729, 13.646690368652344, -6.099194049835205, 4.47422456741333, -0.5077240467071533, -8.456609725952148, 1.5891778469085693, 3.7261433601379395, 6.108380317687988, 5.699367046356201, 3.72943377494812, -3.0597474575042725, -3.522535800933838, -6.6338582038879395, -2.5193095207214355, -0.9963325262069702, -0.855387270450592, -16.90863037109375, -0.5384758710861206, -6.285401344299316, 0.7913872003555298, -2.2219760417938232, 7.56331205368042, -6.630546569824219, -10.593926429748535, -3.157240629196167, -3.374584913253784, 4.1975483894348145, 2.4201650619506836, 1.5704915523529053, -0.9038712978363037, -4.891163349151611, 3.4376776218414307, 4.009454727172852, -0.21422079205513, -8.253074645996094, 0.7256981134414673, -4.525629043579102, -3.352199077606201], [-1.598252296447754, -7.824702262878418, 1.1373460292816162, 3.046635866165161, -1.0837204456329346, -11.825244903564453, 9.994304656982422, 7.981339454650879, -3.829050064086914, 2.371593475341797, 15.014032363891602, -5.884578704833984, 13.50890064239502, -4.346311092376709, 1.8579405546188354, -16.749011993408203, -5.003335475921631, -1.3388278484344482, 13.084545135498047, -3.2287449836730957, -0.7108107805252075, -3.6593992710113525, 10.32674789428711, -1.1056253910064697, -9.005901336669922, -8.013598442077637, -9.754271507263184, -4.504759788513184, -1.6841861009597778, -8.997629165649414, 2.570298433303833, -3.1633474826812744, -9.52995491027832, -5.877056121826172, -4.322810173034668, -0.4083394408226013, 1.6354633569717407, 9.944424629211426, 0.8019739985466003, 9.13569164276123, -1.5274031162261963, 1.941239595413208, 5.507490634918213, -6.959415435791016, 3.1331734657287598, -7.2781572341918945, -14.341987609863281, 13.90364933013916, -11.75068473815918, 0.8939839601516724], [-5.025618553161621, -9.560876846313477, 0.8063521385192871, -7.8467631340026855, -4.105457782745361, -2.0234169960021973, 1.692435622215271, -4.03404426574707, -5.591779708862305, 7.955014228820801, -6.979480266571045, 2.218924045562744, 0.22365474700927734, 4.5977654457092285, 1.2861406803131104, 13.511752128601074, 2.6896138191223145, 4.587258338928223, -12.08486270904541, -12.045648574829102, -5.163834571838379, 8.247151374816895, 1.5069613456726074, -0.3484437167644501, 5.818225383758545, -3.8541665077209473, -0.5304750204086304, 2.4787774085998535, -10.37559700012207, 3.5114221572875977, -1.6349354982376099, 5.3343281745910645, 0.795300304889679, 7.764270305633545, 0.6608999371528625, -1.9392147064208984, -1.2320997714996338, 8.02051830291748, -3.2716710567474365, -3.7873361110687256, -2.205796957015991, -1.5952072143554688, -4.4630231857299805, -5.19070291519165, -5.0140581130981445, 3.331705093383789, 0.1437198370695114, -1.4975401163101196, -7.692101955413818, 2.971689224243164], [0.8512153029441833, -0.6676429510116577, 3.444345474243164, -4.80357027053833, -5.685731887817383, -4.841037750244141, -8.093756675720215, -3.011406421661377, 2.7852542400360107, -0.20031529664993286, -4.591418266296387, 1.5444903373718262, 12.290672302246094, 3.9636142253875732, -5.69044828414917, -5.645937442779541, 3.830601692199707, -4.315006732940674, 13.821179389953613, -5.51078987121582, -8.15771198272705, -9.023499488830566, 4.810422897338867, -7.394648551940918, -6.618258953094482, 8.005388259887695, -11.14912223815918, 6.106332302093506, 6.106439590454102, -3.8428902626037598, 12.033102035522461, -5.927784442901611, 9.204106330871582, -16.07283592224121, 9.434456825256348, 3.818814754486084, 2.805291175842285, -5.688044548034668, 4.965987682342529, 12.317447662353516, -2.231358528137207, -4.59545373916626, -5.088649749755859, -6.016239643096924, 17.16423988342285, 7.086965084075928, 3.746335506439209, 13.453076362609863, -4.1957621574401855, 3.152743101119995], [6.01765775680542, 8.152993202209473, 3.4615137577056885, 3.332087755203247, -3.1881051063537598, -3.594848155975342, -10.583325386047363, 9.86958122253418, -0.07244108617305756, -6.2381134033203125, 0.1675383597612381, -0.18541276454925537, 13.10905647277832, 0.9305286407470703, -14.227560043334961, 4.498722076416016, 8.394745826721191, -5.03816032409668, 12.020146369934082, 6.568737983703613, -3.9667255878448486, -3.4332528114318848, 0.4990525245666504, 2.106640577316284, -12.635493278503418, -2.2366621494293213, -0.18701094388961792, -2.088242769241333, -4.714911460876465, -1.6878008842468262, -1.517789602279663, -11.63026237487793, -1.7672927379608154, 0.0869111493229866, 3.2882602214813232, 7.573172092437744, 2.52356219291687, 12.208724021911621, -1.6972180604934692, 5.556222915649414, -7.82373571395874, 3.3512351512908936, -8.009572982788086, 3.4612619876861572, 6.1639533042907715, 9.934592247009277, 2.87969970703125, 7.470869064331055, 6.48692512512207, 0.8576120138168335], [4.775899887084961, 4.889916896820068, 5.430905342102051, -9.281595230102539, 9.485791206359863, 5.039634704589844, -0.4645174741744995, -0.2770254611968994, -4.6415324211120605, 1.1249401569366455, -2.5523593425750732, -6.876777172088623, 8.016420364379883, -5.059091091156006, 5.334068298339844, -9.723565101623535, -2.6093854904174805, 1.130320429801941, -3.795760154724121, -8.403716087341309, -4.924613952636719, -0.7756626009941101, 4.5601348876953125, -2.4974775314331055, -1.813337802886963, 0.8229774236679077, 7.597262382507324, -4.328843593597412, -4.4691619873046875, 6.287083148956299, 10.458785057067871, -4.871192455291748, 2.5536229610443115, 3.606872081756592, -7.900387763977051, -0.9254164099693298, 5.798563480377197, 8.532038688659668, -2.5727381706237793, 1.5225470066070557, -13.721000671386719, 3.7519915103912354, -4.161400318145752, -2.9962210655212402, -1.0183767080307007, 1.5018078088760376, 0.3253450393676758, 2.399083137512207, 0.09120020270347595, -1.0742690563201904], [1.0660145282745361, 3.197920322418213, 17.883729934692383, -7.421102046966553, 2.597968101501465, 0.49143552780151367, -1.3805506229400635, 0.016585230827331543, 4.695522308349609, -2.3254923820495605, -4.089560508728027, 3.539032459259033, -6.362508296966553, 0.7622116804122925, 0.2813299298286438, -7.085189342498779, 7.461635112762451, 10.812314987182617, 8.585813522338867, -10.6859712600708, -3.101346969604492, -5.774396896362305, -3.029064178466797, -8.882550239562988, -4.912604331970215, 5.174124717712402, 2.0319461822509766, 1.3207380771636963, 1.8192507028579712, 6.4679975509643555, -6.232133865356445, -1.667457103729248, 9.487272262573242, -3.579735517501831, 6.063080310821533, -7.673703193664551, -1.018385648727417, -2.2669031620025635, 5.358778476715088, 10.674448013305664, 4.552321434020996, 4.101410865783691, -4.264393329620361, -14.984753608703613, 5.97676944732666, 2.3497376441955566, 2.6918158531188965, 5.881505012512207, 0.12181645631790161, 5.131303310394287], [-2.108153820037842, -2.346153497695923, 1.3656076192855835, -2.68680739402771, -2.0640110969543457, 4.419513702392578, -5.806515693664551, 5.6707258224487305, -13.451275825500488, -9.811824798583984, -7.208895206451416, -2.77884840965271, 2.7745752334594727, -2.8754184246063232, -7.547811031341553, 3.383194923400879, -12.766468048095703, -1.218726634979248, -7.062258243560791, 0.3880702257156372, -0.8676305413246155, 1.0124839544296265, -1.6933250427246094, -9.779070854187012, 1.6115405559539795, 8.790749549865723, 9.019326210021973, 7.94417667388916, -13.001967430114746, 7.857577800750732, -1.3310966491699219, 8.905037879943848, 12.871066093444824, 1.6161051988601685, 6.447353839874268, -1.4511964321136475, -5.291348457336426, 7.792812347412109, 2.0178680419921875, 19.03035545349121, 4.684450626373291, 0.3662113845348358, 5.230911731719971, -4.9685235023498535, 9.063907623291016, -7.533228874206543, 0.03565555810928345, -6.476274013519287, -2.1653223037719727, -2.1316118240356445], [-11.394244194030762, -3.972081184387207, 10.967628479003906, -1.909619927406311, -1.6460208892822266, -4.606103897094727, 4.515273094177246, 6.195788860321045, 5.069462299346924, 8.019579887390137, 3.547013759613037, 4.5565409660339355, 6.222745895385742, 10.036641120910645, -4.967777729034424, -1.742055058479309, 13.551161766052246, 6.994420051574707, 5.721399307250977, -4.480441570281982, -9.122445106506348, -1.903132677078247, -5.07283353805542, 0.3958776593208313, 0.906254768371582, -3.1719603538513184, -3.8536155223846436, 5.39783239364624, -5.937368392944336, 5.825084686279297, -7.841100692749023, 1.0890138149261475, -0.8039581775665283, 13.353259086608887, 6.052134990692139, -8.9668607711792, 6.025898456573486, -0.07111925631761551, -2.4612557888031006, 7.266134262084961, 13.872754096984863, 13.358677864074707, 9.662528038024902, 6.424844741821289, -12.513195037841797, 5.142625331878662, -3.8861541748046875, 5.538681983947754, 4.153125762939453, 2.7449328899383545], [4.1395792961120605, 4.8101725578308105, 1.2002506256103516, -5.180458068847656, -2.509269952774048, -6.576900959014893, -1.5002453327178955, -4.532526016235352, -2.328477144241333, -1.4630147218704224, -1.4860286712646484, 4.510312557220459, 1.5392760038375854, -1.0704636573791504, 4.229350566864014, 8.372526168823242, -10.931347846984863, -0.9674825072288513, -0.9289159774780273, 5.38333797454834, 9.225378036499023, -6.227643966674805, -7.581149578094482, 1.0961213111877441, 7.604185104370117, 5.490321636199951, -3.3867523670196533, -0.18557199835777283, 6.362129211425781, 1.0435242652893066, -0.5064194202423096, -4.189006328582764, -6.030038833618164, -3.089369297027588, 9.81730842590332, -7.842825889587402, 3.327678680419922, 6.753159046173096, 4.968521595001221, 3.3900861740112305, 0.9697992205619812, 0.8458002805709839, 6.236732482910156, -3.919466972351074, 3.105327606201172, -0.31085604429244995, -2.7274417877197266, 4.046875476837158, -9.653430938720703, 2.6644439697265625], [3.4465651512145996, 6.05029296875, -18.812484741210938, -6.620939254760742, -2.1868653297424316, 8.728994369506836, 2.3307061195373535, -5.422526836395264, -7.9259490966796875, 7.45914888381958, -1.3994593620300293, 0.11867114156484604, -6.734806060791016, -1.0319324731826782, 4.341692924499512, 7.637970924377441, 0.21171331405639648, 1.3463658094406128, 0.17561551928520203, -8.430182456970215, -4.866356372833252, 9.668936729431152, -8.525678634643555, 2.5845956802368164, -0.014703154563903809, 2.27067232131958, -8.105239868164062, -3.3171873092651367, 0.2382875233888626, -1.648394227027893, 9.234910011291504, 2.04099702835083, 9.610815048217773, -4.251633644104004, -0.9363327622413635, 7.032249450683594, 12.978718757629395, 2.882883310317993, -5.015189170837402, -13.68148422241211, -1.9153506755828857, -0.48127084970474243, -16.549213409423828, -9.348773002624512, -2.3063158988952637, -0.24530029296875, 7.175629615783691, -3.5119004249572754, 4.545103073120117, 3.923675775527954], [-1.9275496006011963, -5.42234992980957, 2.601060152053833, 2.738093376159668, 0.0026386380195617676, 3.825749397277832, -10.439098358154297, 4.396426200866699, -7.556771278381348, -5.271965503692627, -3.4841222763061523, -7.610334396362305, -8.771048545837402, -1.249603509902954, -3.2151734828948975, 0.5174362063407898, 3.441317558288574, -2.434812307357788, -0.7989877462387085, 9.099384307861328, -3.35105562210083, -8.561976432800293, 1.6469907760620117, -2.697610855102539, -3.401594638824463, -4.105889320373535, 4.600705146789551, -0.05274885892868042, -1.9314213991165161, -5.595766544342041, 0.9323887228965759, 3.9605464935302734, -2.3684377670288086, 2.8648781776428223, 7.669261455535889, -1.1815392971038818, -1.238093376159668, -2.0568530559539795, -0.5001747012138367, -4.0402021408081055, -5.301968574523926, -8.182170867919922, -12.40285873413086, -14.669760704040527, -3.061676502227783, 3.2152459621429443, -2.0294301509857178, 17.53603744506836, -14.523509979248047, -6.581144332885742], [-0.5054806470870972, -5.780704975128174, 2.5298233032226562, -1.9252877235412598, -2.629570245742798, 1.387647271156311, 5.034944534301758, 1.6569010019302368, 2.100275754928589, -9.814574241638184, -1.3561185598373413, -2.1884891986846924, -3.091503381729126, -5.058077812194824, 1.864598274230957, 6.134519100189209, -11.672086715698242, 9.974175453186035, -14.905200004577637, 2.2372705936431885, 3.105393409729004, -3.245274782180786, 0.985740065574646, -2.42698073387146, 1.662833571434021, -0.6605724692344666, 1.2242770195007324, -7.548105239868164, -9.667256355285645, -3.6286933422088623, 4.104727268218994, 2.4628705978393555, 8.570575714111328, -1.6732861995697021, -9.003722190856934, -3.0118188858032227, -5.294132232666016, 0.40592068433761597, -2.301516056060791, -7.0571489334106445, -10.088805198669434, -7.520532608032227, -2.1677072048187256, -8.81910514831543, 4.654827117919922, -1.4922709465026855, -3.5118894577026367, -10.078269004821777, -5.155965805053711, -5.3006815910339355]], "dtype": "float32"}}, {"a": [{"__nd__": [[-1.746380090713501, -0.2089056670665741, -0.4400106966495514, 1.2884273529052734, -0.015026125125586987, 1.0310425758361816, 2.0873916149139404, 1.030765414237976, 0.1286032795906067, 0.7893506288528442, -0.7998166680335999, -0.011119124479591846, -0.003350673010572791, 0.29989171028137207, -0.27260822057724, 0.027029821649193764, 1.5752683877944946, -0.9893953204154968, -0.14949114620685577, 0.42030245065689087, 0.5405539274215698, -0.7410818338394165, -0.772459864616394, 0.2893558144569397, 0.32584401965141296, -1.8905339241027832, 1.7257603406906128, -0.3142034411430359, 0.8437734842300415, 0.7361533641815186, -2.025681972503662, 0.24966351687908173, -0.5411805510520935, 1.386242151260376, 0.9042420387268066, 1.4448610544204712, 0.1920994073152542, -1.0671272277832031, -0.10055353492498398, 0.012354274280369282, 0.5973161458969116, 0.7221139073371887, 1.0225578546524048, 0.6730152368545532, 0.05790335685014725, 0.9189769625663757, -0.5598537921905518, -0.23841887712478638], [-0.47027185559272766, 0.8484802842140198, -0.5654975175857544, 0.49384427070617676, -0.11790359020233154, -0.14898079633712769, 0.7805946469306946, -0.6534267067909241, -0.04251738637685776, 0.2543303966522217, 0.07679883390665054, 0.6467585563659668, -0.11788786947727203, 0.16680572926998138, 2.7396764755249023, -0.27220937609672546, -0.9443727731704712, 0.36104637384414673, -1.1933114528656006, -0.04460880532860756, 0.0789412409067154, 0.5876556634902954, 0.40017083287239075, -0.26228028535842896, -0.24059398472309113, 1.1913622617721558, -0.5458018779754639, 1.0052897930145264, -0.33041971921920776, 0.1391570121049881, 0.7158512473106384, 0.8256233334541321, 0.7586010694503784, -0.6964051723480225, 0.6054577231407166, -0.9820114970207214, 1.1372888088226318, -1.238551378250122, 0.36008477210998535, 0.7109385132789612, 0.05133131146430969, 0.8781425952911377, -0.5109930038452148, 2.4006571769714355, 0.7573519349098206, -1.4055063724517822, 1.4228179454803467, -1.562523365020752], [-1.02340567111969, -1.2637912034988403, 1.2875101566314697, -1.1043146848678589, 0.29330089688301086, -0.8967726826667786, 1.5487253665924072, -0.6841149926185608, 0.31861793994903564, -0.8763634562492371, 0.82012939453125, 1.0066249370574951, -1.147998571395874, -0.08897913247346878, 1.5730141401290894, -0.7140316963195801, -0.3811781406402588, 0.4096173048019409, -1.6268837451934814, -1.0474344491958618, 0.2302701473236084, -0.00934061873704195, -0.583576500415802, 0.8792913556098938, -0.771436333656311, 2.6030375957489014, 1.366551399230957, 1.101009726524353, -0.9147170782089233, -0.035126566886901855, -2.558504104614258, 0.5395760536193848, -0.18303653597831726, -1.0878288745880127, 0.3676592707633972, -0.10567354410886765, 0.026139097288250923, 0.29798686504364014, -0.8728751540184021, 0.6651589274406433, 0.11861798912286758, 2.3406496047973633, -0.6416298747062683, -0.71499103307724, -0.5282906293869019, 0.26872071623802185, -1.043703317642212, -0.7529801726341248], [-1.0512232780456543, -0.735197126865387, -0.11873216181993484, 0.5530580282211304, -2.357268810272217, -0.3827626407146454, 0.06612750142812729, 0.05322887748479843, -1.794205904006958, -1.0040287971496582, -0.8006788492202759, 1.3952113389968872, 1.0094537734985352, -0.19694973528385162, -0.2132488340139389, -1.11405348777771, -0.25315061211586, -0.15704946219921112, -0.18537062406539917, -0.67781662940979, 0.152736634016037, -2.008991003036499, 1.1536803245544434, -1.1624740362167358, -0.11786562204360962, -0.44368284940719604, -0.14946357905864716, 0.5788874626159668, 1.057074785232544, -1.3678058385849, 0.8660799264907837, 1.3777498006820679, -1.369943380355835, 0.9464929699897766, -0.8457232117652893, -0.41794583201408386, 0.2747495472431183, -1.8038326501846313, -1.4084454774856567, 0.028408262878656387, -0.19784043729305267, -0.36149004101753235, 0.22233043611049652, -2.0195837020874023, -0.19248676300048828, 0.1857520341873169, -1.202987551689148, 1.0002562999725342], [-0.3863840699195862, -0.2043466866016388, -1.2394739389419556, -1.378739833831787, 0.05915774777531624, 1.861011028289795, 0.03840361908078194, 0.680342972278595, -1.2148323059082031, -1.7659482955932617, -0.5313579440116882, 0.874686598777771, -0.03492017462849617, -0.10346350818872452, 0.09612850844860077, -0.29578515887260437, -0.5816799998283386, 1.1312222480773926, -0.843525230884552, -0.012902913615107536, 1.472634196281433, 0.18727482855319977, -1.3752496242523193, 0.12211894989013672, 0.23149274289608002, -0.5793580412864685, -1.4243412017822266, 1.3094890117645264, 0.5121847987174988, 0.1835029274225235, 0.09210575371980667, 0.9542244076728821, -0.832221269607544, -0.22378313541412354, 0.2888813316822052, 1.1131755113601685, 0.22408351302146912, 0.2603955864906311, -1.021438717842102, -0.37901920080184937, 0.8093445897102356, 0.9242526292800903, 1.8147149085998535, -0.7486894130706787, -0.2895752489566803, 1.0565861463546753, 1.1411445140838623, 1.3380783796310425], [0.7960910797119141, 0.20407401025295258, -0.43079379200935364, -0.6031455397605896, -0.071995809674263, -0.5932394862174988, 0.014257168397307396, 0.2764448821544647, -0.21106822788715363, 1.060246467590332, 0.6190458536148071, 0.8397354483604431, -0.8016805648803711, -1.21145761013031, -1.0699827671051025, -0.5919691324234009, 0.3786931037902832, 0.8437108397483826, 0.6261184215545654, -1.24109947681427, -0.9060524702072144, -0.9690135717391968, 0.4802410304546356, 0.2364252209663391, 1.5511596202850342, -0.028325539082288742, -0.5545212626457214, -0.26148948073387146, 0.46268513798713684, -0.11734449863433838, 0.2480500042438507, -0.25994086265563965, -1.795501708984375, -0.29753637313842773, 0.3543534278869629, 1.5198968648910522, -0.30901041626930237, -0.4514319598674774, 0.06367853283882141, -0.6765781044960022, 1.3868454694747925, 0.5980395078659058, -1.931681752204895, -0.5590532422065735, -0.5499759316444397, 0.6929145455360413, 0.1522648185491562, -0.5927154421806335], [-0.5299177169799805, -2.0809476375579834, 0.7745916843414307, -1.8541626930236816, -0.11773978918790817, 0.583724319934845, -0.18974623084068298, -1.4363701343536377, -0.8217371106147766, -0.2157207578420639, 0.3863690197467804, 0.6900519728660583, -1.371292233467102, -2.062898635864258, 0.15915608406066895, -0.5866742730140686, 1.403903841972351, -0.15754210948944092, 0.5249263644218445, -1.057490348815918, -1.1582989692687988, 1.0214570760726929, 1.665810227394104, -0.5014817714691162, 0.39085274934768677, -0.8332372903823853, 0.8969417810440063, 0.2827502191066742, 1.0271648168563843, 0.3944423496723175, 0.8950156569480896, -0.3275032937526703, 0.1591130793094635, -0.49556422233581543, 0.3687269985675812, -0.16856710612773895, -0.16030390560626984, -0.39112192392349243, -0.6998331546783447, 0.19982506334781647, -0.40966516733169556, -1.028012752532959, 0.17733575403690338, -0.2812531590461731, -0.26639845967292786, 0.8072568774223328, 1.63486909866333, -0.46688413619995117], [-1.7368392944335938, 0.4560997784137726, 0.291691392660141, -0.07801707834005356, -1.3920234441757202, -0.3267575204372406, -1.0191799402236938, 0.15675859153270721, -1.926442265510559, -0.48534759879112244, 0.5087734460830688, -0.5297176837921143, -0.13989576697349548, 1.6265069246292114, -3.102766752243042, 1.5884310007095337, -1.152420163154602, 0.22716055810451508, 1.0486834049224854, -0.3696390986442566, -0.06449876725673676, -0.27613621950149536, 0.8529143929481506, -0.5982843041419983, 0.7202693819999695, -0.47967371344566345, 2.545558452606201, 0.9942295551300049, -0.648729145526886, 0.9905902147293091, 1.011814832687378, 0.7369252443313599, -0.9847242832183838, -0.12995091080665588, 0.5276728868484497, 0.4098457992076874, -1.539851427078247, -0.15266574919223785, -1.9521403312683105, -0.48208072781562805, 1.5303821563720703, 1.8983619213104248, -0.23989106714725494, -1.442370891571045, 0.11371562629938126, -0.6220051646232605, 0.46020734310150146, 0.03259972855448723], [0.17920713126659393, -0.6920582056045532, -0.01375857088714838, -0.8910428285598755, 0.29265615344047546, 1.610408067703247, -0.4643416404724121, 2.208022356033325, 2.316286087036133, 0.6729205250740051, -1.4435973167419434, 0.3287232518196106, 1.5883640050888062, -2.5174806118011475, 0.5469470620155334, -1.0891003608703613, -0.49018460512161255, -0.4309346079826355, -0.021984292194247246, -1.4356509447097778, 0.30193793773651123, -3.0275659561157227, -1.2141554355621338, 0.4576743543148041, 0.1351543515920639, -2.1904070377349854, 0.8772131204605103, -0.7234810590744019, 0.6699211001396179, -0.5219968557357788, -0.6999762654304504, 1.7069674730300903, 1.2457557916641235, 0.20811037719249725, 1.3863463401794434, -0.7130988240242004, -1.135819673538208, -1.510110855102539, -2.0740909576416016, 0.6122463345527649, 0.05188702791929245, -0.5094501972198486, 0.12145428359508514, 2.0153768062591553, -0.30012863874435425, -1.5752233266830444, -0.1300215721130371, -0.8439679741859436], [0.8205783367156982, 0.9935522675514221, 0.3899112343788147, -1.2882744073867798, -0.6862913966178894, 0.26218992471694946, 2.273422956466675, -0.8303574919700623, -0.07064016163349152, 1.1794601678848267, 0.9279952049255371, 1.2751290798187256, -0.38761159777641296, 2.1340692043304443, 0.38323089480400085, -1.6120917797088623, 0.11219047009944916, 0.01835182122886181, 0.7437961101531982, 0.9274523854255676, 1.5165326595306396, 0.873278796672821, 1.1413246393203735, 1.0254875421524048, -1.3469735383987427, -0.9604366421699524, -0.7644945979118347, 1.801971673965454, 1.121229648590088, -0.26570001244544983, 0.4800000786781311, 1.5018432140350342, -2.6005260944366455, -0.3720564842224121, 0.16267560422420502, -0.7427534461021423, -0.7423965334892273, 0.6417749524116516, -0.5732476711273193, -0.810120165348053, 1.7564334869384766, 1.1502491235733032, -1.6703662872314453, -1.0269986391067505, 0.6397996544837952, -0.7113092541694641, 1.5102782249450684, 0.3993477523326874], [-1.7488278150558472, -1.5641673803329468, 0.016908226534724236, -0.9745091795921326, -0.1143784150481224, 0.028563743457198143, 0.059555888175964355, -0.7176026701927185, -2.198859930038452, 0.2704658508300781, -0.21191570162773132, -0.615328311920166, 2.0498735904693604, 0.028600910678505898, -0.15245407819747925, 0.882993221282959, -0.9946930408477783, 0.5033929944038391, -1.6865034103393555, -1.1135247945785522, -0.2945598065853119, -0.32898515462875366, 0.0076652392745018005, -1.976439118385315, -1.5850472450256348, -1.0502811670303345, -2.034473419189453, 2.163424015045166, -0.6075407862663269, 1.8011060953140259, -0.10341761261224747, -0.5553499460220337, -0.23726189136505127, -0.4179951548576355, 0.6561327576637268, 1.409958004951477, 0.773398756980896, -1.2067400217056274, -0.6980657577514648, 2.1171209812164307, 1.0196564197540283, 1.0145373344421387, -0.9609907865524292, -0.3782676160335541, 0.8780348896980286, -0.36440807580947876, 1.3552814722061157, -1.72480309009552], [-0.7416036128997803, -0.06628262996673584, 1.3941316604614258, 0.2145293802022934, 1.2656587362289429, 0.3445669114589691, 0.5668801069259644, 1.7053278684616089, 0.7062257528305054, -0.3710746765136719, -1.744094729423523, -0.01664111576974392, 0.03798838332295418, 1.1985594034194946, -1.2139534950256348, -1.1751247644424438, -1.5037468671798706, -0.5553100109100342, 0.9045631289482117, 0.45041295886039734, -0.43239912390708923, -1.0132611989974976, -0.19535045325756073, -0.86729496717453, -0.5297405123710632, 1.1090598106384277, 1.433346152305603, -0.20500026643276215, -0.21705232560634613, 0.307784765958786, -0.7685860991477966, -1.5970827341079712, -0.40841612219810486, -0.1732778400182724, -0.44590654969215393, 0.34305423498153687, -0.15149755775928497, -2.216586112976074, 0.5607106685638428, 1.702014446258545, 0.1306115984916687, -1.0050874948501587, 0.9579099416732788, 0.9840791821479797, -2.2280564308166504, 0.5367321372032166, -1.1502965688705444, -0.3716437518596649], [0.087391197681427, -0.453491747379303, 1.0132980346679688, -0.27564936876296997, -0.5849916934967041, -1.1907902956008911, -0.3466067910194397, -0.056568488478660583, -0.5028638243675232, -0.7799944281578064, -0.4873890280723572, -1.2815030813217163, -0.06376128643751144, -0.028749478980898857, 0.2822123169898987, 1.4324443340301514, -1.3009834289550781, 1.56820547580719, -0.8349868059158325, -0.5734810829162598, 0.7280118465423584, 0.0639645904302597, -0.49896806478500366, -0.4759376049041748, -1.4653276205062866, -0.9821169972419739, -0.9022391438484192, 0.9532374143600464, -0.34723952412605286, 0.5042223930358887, -0.04566918686032295, -0.4703005850315094, 0.0389193519949913, 0.024694833904504776, -0.1673574000597, 0.1731552928686142, -0.31460633873939514, -0.07984618097543716, -1.4481202363967896, -1.9750841856002808, 0.2275897115468979, 0.08390659838914871, -0.8141686320304871, -2.1688120365142822, -0.7902674674987793, -1.2567391395568848, -2.177978515625, -0.9468443989753723], [1.4783623218536377, 0.9398244619369507, -0.7758578658103943, -0.6133963465690613, -1.59995436668396, 1.039568543434143, -1.8476152420043945, -0.49376195669174194, -0.0704384371638298, -0.7439624667167664, 2.0668118000030518, -0.2046358734369278, 0.7955878973007202, -0.5435616374015808, 1.0037835836410522, -0.04543603956699371, -0.2589609622955322, 1.2508617639541626, 1.5009486675262451, 1.0520695447921753, -0.45177537202835083, 0.04251383990049362, -0.3377729654312134, -0.06608015298843384, 0.5321251153945923, -0.07460961490869522, 0.8126443028450012, -0.10308021306991577, 0.8750208020210266, -0.2762209475040436, 0.7364643812179565, -0.004525717347860336, 0.4554658532142639, 0.5130754113197327, -0.018907606601715088, 0.2145656794309616, 1.7514009475708008, 0.4632539749145508, 1.6001107692718506, -2.6666290760040283, 0.884489119052887, -0.7465969324111938, -0.9436154365539551, 0.14026282727718353, 0.4266873896121979, -0.19281639158725739, 1.3266485929489136, 0.7415223121643066], [-0.19815298914909363, 0.3207099735736847, -1.0092368125915527, -0.7149240970611572, -1.8400644063949585, 0.8417984843254089, -0.9947930574417114, -0.38373231887817383, 2.2019596099853516, -2.47684645652771, 0.7648683190345764, -0.6143351793289185, 1.1145331859588623, 0.2718574106693268, 0.21366682648658752, -0.03305491805076599, 0.6003572344779968, -0.8123219013214111, -0.7929093241691589, -0.6309654116630554, -0.7920814752578735, 1.358738899230957, -0.4201905131340027, 0.9417427182197571, -0.07510882616043091, 0.9506660103797913, 0.28008997440338135, 0.6653990745544434, -1.284298300743103, -0.8195086121559143, 1.595658302307129, -1.2668039798736572, 0.7535181045532227, 0.9078179597854614, 0.7893692255020142, 1.9472651481628418, -0.25375911593437195, -0.8384325504302979, -0.8697323203086853, -0.004354923032224178, -0.0907050371170044, -0.1121411919593811, -0.3080066740512848, -0.4807485342025757, 0.8420906066894531, -1.5661542415618896, 0.5357738733291626, 0.33191168308258057], [-0.9566012620925903, 1.7455288171768188, -1.1878479719161987, 0.2361716628074646, 1.383468747138977, 1.0923800468444824, -0.2600953280925751, -0.7900083661079407, -0.8204383254051208, 1.0843510627746582, -0.3902915120124817, -1.4215809106826782, 0.7923387885093689, -1.3435273170471191, -0.9343307018280029, 0.8047654628753662, -1.5057884454727173, 0.7423068881034851, 1.0809204578399658, -0.5692366361618042, 0.00487436493858695, -0.5298929214477539, 0.7776638865470886, -0.9288402199745178, 1.084806203842163, -0.9683738946914673, 0.8264214992523193, 0.005308754742145538, -0.777126133441925, -0.027347415685653687, 0.7144597172737122, -0.15734535455703735, -1.7724719047546387, -0.9729019403457642, -1.8330388069152832, -0.8239615559577942, -2.5021214485168457, 0.5068232417106628, 2.6534976959228516, -0.8601467609405518, 1.4524426460266113, 1.044198989868164, 0.22204595804214478, -0.07303354144096375, -0.2674098312854767, -2.358076572418213, -2.057904005050659, 0.13370822370052338], [0.07912169396877289, 1.2011595964431763, 2.243821859359741, -2.0364997386932373, 0.1896296739578247, 2.118351697921753, -0.1988900601863861, 1.867856502532959, -0.9774162769317627, 0.5755080580711365, -1.2889745235443115, 0.689559817314148, 0.37478867173194885, -0.9834287166595459, -0.3961819112300873, -0.40209493041038513, 0.230145663022995, 2.051706075668335, -0.27963659167289734, 1.5360100269317627, 0.533357560634613, -1.431361436843872, -0.38471904397010803, -1.772265076637268, 0.39992985129356384, 0.8579006791114807, -0.4317663908004761, 0.5033159852027893, 0.11693542450666428, 0.07482819259166718, -0.5774467587471008, -0.31842052936553955, -1.3447879552841187, 0.019683191552758217, 1.0057276487350464, -0.4135761559009552, -1.197189450263977, 0.34089916944503784, -2.0419163703918457, -0.47017157077789307, 0.9685278534889221, -0.9308550357818604, 0.12967291474342346, -0.19955138862133026, -0.4882926344871521, -0.40655452013015747, 1.2314940690994263, -1.061019778251648], [1.8007471561431885, -0.06200175732374191, -0.4984928369522095, 0.1966203898191452, 2.130784034729004, -1.2472199201583862, -0.8925608396530151, 0.8556375503540039, 0.0870969370007515, -1.5127642154693604, -0.849291980266571, -0.6775492429733276, 0.7044861316680908, -0.7011201977729797, 3.080477237701416, 1.2011748552322388, -0.5471930503845215, -0.816186249256134, 2.5162832736968994, 0.10296161472797394, 0.7475177049636841, -0.6630802750587463, -0.9886635541915894, 1.0579791069030762, -0.6283922791481018, 1.422073245048523, 1.5596799850463867, 0.5812035202980042, 0.7931308746337891, 0.6913754940032959, 0.0024227870162576437, 0.24279245734214783, 0.13446679711341858, 0.20764729380607605, 0.5594627261161804, 0.92823326587677, -0.9016844034194946, -0.4252804219722748, -0.811810314655304, -0.41660770773887634, 0.27088186144828796, 1.0709229707717896, -0.3966912031173706, 0.4851992428302765, 0.06371309608221054, 0.42731764912605286, 0.6195098757743835, -1.4836870431900024], [-0.8802759647369385, 0.9370142817497253, -0.08480934053659439, -0.5300102233886719, -1.1880613565444946, -2.280313730239868, 0.1462668478488922, 1.2351282835006714, -1.7214680910110474, -0.7835667133331299, 1.457394003868103, 0.6576526761054993, 0.3183972239494324, 1.0261911153793335, 0.691537082195282, -1.1389763355255127, -1.6128112077713013, -0.5302248597145081, 0.12999805808067322, 0.6304345726966858, -0.21565893292427063, -0.9217380881309509, 0.19986695051193237, -0.6637445092201233, 0.44725924730300903, -0.6553597450256348, -0.09288188815116882, -1.5783759355545044, -0.33469146490097046, -1.233197808265686, 1.6285086870193481, 0.9084498882293701, 1.2815748453140259, 1.9993757009506226, 0.3006613254547119, 2.0687613487243652, 0.07214941829442978, -0.08799310773611069, -0.5503888130187988, -0.47143980860710144, -1.8311954736709595, 1.7643322944641113, 0.6438138484954834, -1.5044211149215698, 0.6045989394187927, -1.1033838987350464, 1.2461957931518555, -0.8776715397834778], [0.369935005903244, -1.3224756717681885, 0.8350120186805725, -0.4060375690460205, 1.2973885536193848, -0.35920998454093933, -0.6931217312812805, -0.10890548676252365, -1.0408248901367188, -0.8847781419754028, -1.2535990476608276, 0.2471255362033844, 2.0211398601531982, 1.5906152725219727, 0.49788904190063477, 0.22219817340373993, -1.9871107339859009, -0.8664984107017517, -2.464951992034912, 0.3266626298427582, -0.5111989974975586, 1.2127134799957275, 0.12796157598495483, 0.5215876698493958, -0.6833959221839905, -0.7820907235145569, -0.6663548946380615, -1.2461689710617065, -0.9524210691452026, -0.04107816144824028, -1.5845036506652832, 0.5932377576828003, -0.8243902325630188, 0.6687378883361816, 1.176674723625183, 0.09197518229484558, 0.1893896609544754, 0.7380198836326599, 0.6725926995277405, -1.6815154552459717, 0.1649201363325119, -1.4462798833847046, 1.1535040140151978, -1.2017306089401245, 0.171670600771904, -0.6310123205184937, 2.321261405944824, -0.14319416880607605], [-0.7857236862182617, -1.1751289367675781, 0.5261887311935425, -0.40140122175216675, 0.27100440859794617, 1.3181167840957642, -0.018032675608992577, -0.04184535890817642, -0.16115862131118774, -0.20656728744506836, -0.47417184710502625, 0.10274152457714081, 0.18395690619945526, -0.2656375765800476, -0.23626480996608734, 0.022989530116319656, -0.3374978303909302, 0.02169874683022499, -0.17585913836956024, 0.8545751571655273, 0.028479604050517082, 0.09086333215236664, -0.3732910752296448, -1.7482266426086426, -0.4467046558856964, 0.9311919808387756, -0.04181622341275215, 0.11787020415067673, -0.3600921034812927, -1.212788701057434, -0.04852432385087013, 1.5021601915359497, 1.8857345581054688, -0.0028861903119832277, -1.7911590337753296, -0.4020659923553467, 1.084356427192688, -0.43498969078063965, 1.4728031158447266, -0.29622921347618103, 0.010886656120419502, -2.181199073791504, 1.3287842273712158, 0.35437387228012085, 0.4490916132926941, 0.18152368068695068, -1.398382544517517, 0.12683048844337463], [-0.12202023714780807, 1.0841588973999023, -0.2602361738681793, 0.02487877942621708, -0.5719757080078125, 0.04602885991334915, -0.4292609989643097, 0.8057587742805481, 0.3070814311504364, -0.7267754673957825, 1.699957251548767, -0.34814703464508057, 1.3255765438079834, 1.7551767826080322, 0.1354956179857254, 2.4334840774536133, 0.2451678216457367, 0.5947879552841187, 0.5413603782653809, -2.9790396690368652, -1.8799817562103271, -1.6746851205825806, 1.5180332660675049, 0.22262243926525116, -0.6622584462165833, 0.4145333468914032, 0.6703577637672424, 2.4081549644470215, -0.4255653917789459, -0.24255745112895966, -0.3914399743080139, -0.5785060524940491, 0.9183811545372009, 1.0608834028244019, -1.0260084867477417, -0.13899309933185577, -0.9264750480651855, -2.2291271686553955, 1.086205005645752, 0.6213974356651306, -0.4337083697319031, -0.6661126613616943, -1.5459949970245361, -0.5214345455169678, -0.009801083244383335, -2.5232338905334473, 0.2578009366989136, -0.5770134925842285], [-0.28259041905403137, -0.9971698522567749, 0.1661371886730194, 0.46177875995635986, -2.553889751434326, 0.10111775249242783, -0.6539037823677063, -0.6024709939956665, 1.0507346391677856, -2.203806161880493, 0.7997071146965027, 0.5657213926315308, 0.1369444578886032, 0.8815867900848389, -0.18531136214733124, 0.8823239207267761, 0.6421823501586914, 0.35910919308662415, 2.471323251724243, 1.0745978355407715, -0.4000762701034546, -0.7453504204750061, -0.1881272941827774, -1.1121864318847656, 0.06708762794733047, -1.2568703889846802, -1.5990577936172485, 0.8305184841156006, -0.48702070116996765, 0.3997724652290344, -2.008957624435425, 0.3759802579879761, -0.42467620968818665, -0.35061153769493103, 0.5649270415306091, 0.9158793687820435, 0.7111758589744568, -0.5408341288566589, -1.582308292388916, 0.38669469952583313, 0.9293519854545593, -0.5755167007446289, 0.4893859624862671, 1.5704604387283325, -1.0905944108963013, -0.10324843972921371, 0.21223919093608856, 0.3015318810939789], [1.6317797899246216, 2.3265044689178467, 0.11136980354785919, -2.3445749282836914, -0.13989077508449554, 0.6681056618690491, 0.23881737887859344, 0.18963661789894104, -0.18249715864658356, -0.041773878037929535, -0.9546219706535339, -1.4976752996444702, -0.3500552177429199, 0.8687249422073364, 1.186989188194275, -0.4549017548561096, 0.03900457173585892, -0.7690303921699524, -0.28263533115386963, 0.6871933341026306, 1.6892949342727661, -1.2621936798095703, -0.5491392612457275, 0.7516872882843018, -0.1172967329621315, 0.679097592830658, -0.69901442527771, 0.24025499820709229, 0.5779102444648743, -0.059811655431985855, -1.1792832612991333, 0.3629530072212219, -1.3113869428634644, -0.5701441168785095, -1.0015543699264526, 0.39031705260276794, -1.5915099382400513, 0.25780928134918213, 1.557137131690979, -1.8880174160003662, 0.8008405566215515, -0.11392682790756226, 1.6948589086532593, -0.14110314846038818, 0.102780282497406, -2.216548442840576, -0.3008084297180176, -0.37211838364601135], [-0.48382270336151123, 1.1081483364105225, -1.967312216758728, 1.091548204421997, 0.4515873193740845, -1.038404107093811, -0.5591520667076111, -0.5654308795928955, -0.6905324459075928, -0.4995451271533966, -0.31960898637771606, -1.2282854318618774, 1.3689656257629395, -0.05867781862616539, 0.048743996769189835, 0.8806535005569458, -0.5222865343093872, 2.1785945892333984, 2.3137974739074707, 0.8824970126152039, -0.5576287508010864, -0.9800035953521729, -0.29430511593818665, -1.7792199850082397, -0.8417106866836548, -0.08024723827838898, -0.7758183479309082, -0.29440420866012573, -0.13639071583747864, 0.5216881036758423, -0.7677299976348877, -0.9200358986854553, 0.6311514377593994, 0.6745845675468445, 1.3885996341705322, -1.053331971168518, 1.8640437126159668, 0.6999543309211731, 0.5543522238731384, -0.4122242331504822, -0.29974430799484253, 0.7250187993049622, 0.06138971820473671, -0.9323459267616272, -0.4823096692562103, -0.20157133042812347, -0.282100111246109, -0.02904505655169487], [0.6592336893081665, -0.5829105973243713, 2.2254459857940674, -0.2144477814435959, -0.17494304478168488, 0.3012142479419708, -0.7185835242271423, 0.3629279136657715, -1.0071383714675903, -1.7990520000457764, 0.3958626985549927, -1.398402214050293, 0.46067115664482117, 0.7584511041641235, -1.9619389772415161, -0.41745156049728394, -0.883426308631897, -0.16699133813381195, -0.7262819409370422, -0.6625444293022156, 1.388340950012207, 1.2116687297821045, -1.3183605670928955, 0.058073900640010834, 0.1634056568145752, -1.4028855562210083, -0.8852651715278625, 0.09284212440252304, -0.03057517483830452, -0.6032484769821167, -0.5984526872634888, -0.17904873192310333, -0.3754122853279114, -0.9813917279243469, -0.05943939834833145, -1.331350564956665, 0.745834469795227, -0.13997752964496613, 0.9618678092956543, -0.6370105147361755, -1.070970058441162, -0.501410186290741, -0.9028007388114929, 2.3154027462005615, -1.250464916229248, 0.17891277372837067, -1.1710368394851685, 0.6772178411483765], [-0.4821774959564209, 0.35575437545776367, -1.846130609512329, 0.3107205629348755, -0.8535779714584351, -1.4480180740356445, -0.7062206268310547, 1.0513982772827148, -0.03914641588926315, -0.5945762991905212, -1.5053313970565796, -2.133838653564453, 0.630724310874939, 1.4300768375396729, -0.3735368549823761, -0.5428338646888733, -0.8457878232002258, -0.9205107092857361, -0.18610349297523499, -3.110278844833374, -0.03470229730010033, -0.6073634624481201, 0.030247507616877556, 0.8648706674575806, 1.9116843938827515, 0.08994142711162567, -1.9918543100357056, 1.6001477241516113, -0.14901170134544373, 1.1886111497879028, -0.9502624273300171, -0.3366253674030304, 0.9926969408988953, 0.557137668132782, 0.29220953583717346, -0.1152876764535904, 0.2376539260149002, 0.3223787546157837, 0.49036234617233276, -1.617243766784668, -1.003042221069336, -0.7896530032157898, -0.42132067680358887, 0.4500076174736023, 0.16554145514965057, -1.647139072418213, -0.7305516600608826, -1.0503904819488525], [-1.7898938655853271, 1.4841234683990479, 0.13327454030513763, -0.2499583661556244, 0.6158924102783203, -0.08410315960645676, 0.24170039594173431, -0.3470480740070343, -1.2846211194992065, 1.1132293939590454, 1.0195072889328003, -1.2046648263931274, -1.4555528163909912, 0.002628449583426118, -1.063852310180664, -1.0713086128234863, 1.597724199295044, -1.833773136138916, 0.6388543844223022, 0.9512966275215149, 0.3966754078865051, -0.5508360862731934, 2.4132823944091797, 0.17064562439918518, -0.672479510307312, -0.8341355919837952, 0.07641752809286118, 1.7898112535476685, -1.560462236404419, -0.47494184970855713, 0.1519962102174759, 0.11040659993886948, 0.2623614966869354, -0.6225024461746216, -0.28298425674438477, 0.24984335899353027, 1.4317548274993896, -1.0732840299606323, 0.5501698851585388, 1.4881497621536255, 1.54335355758667, 0.7267269492149353, -0.35186994075775146, 0.5368716716766357, 0.16029182076454163, 1.0650904178619385, 0.8967076539993286, -0.2565634250640869], [-0.78138667345047, -0.5646750926971436, -0.6777650117874146, -0.5386002063751221, 1.070249080657959, 0.5741996765136719, -0.08160699903964996, -2.2296130657196045, 0.7759155631065369, 0.4566532075405121, 2.173204183578491, 0.7583757042884827, 0.16964338719844818, 0.33023568987846375, -0.8468887209892273, 1.2025253772735596, -0.9526947736740112, 0.1472238153219223, 0.31556108593940735, 2.1052169799804688, 0.3476592004299164, 0.07694943249225616, -1.3197070360183716, 0.2950213551521301, 2.5638513565063477, -0.7641448974609375, -0.5022497773170471, -0.6236503720283508, 1.136989712715149, 1.8173388242721558, -0.5409977436065674, -0.9372501969337463, 0.5107886791229248, -1.341511845588684, -1.1890405416488647, -0.6844701766967773, 1.6042652130126953, -1.4377144575119019, 1.107378602027893, 1.0772132873535156, 0.6334829330444336, 0.3837866187095642, 1.378307580947876, -0.28506818413734436, -0.3913474977016449, 0.018667997792363167, -0.5787061452865601, -0.14084310829639435], [-0.9388062357902527, 1.1929670572280884, -0.4103442132472992, 0.7277792692184448, 0.37144121527671814, -1.180332899093628, -0.4204675257205963, 0.45286646485328674, -0.5574801564216614, -1.1582131385803223, 1.116264820098877, -0.9827442765235901, -0.38503482937812805, -0.7143837809562683, 0.7411742806434631, -0.08532734960317612, -0.23661978542804718, -0.36451202630996704, 1.263661503791809, 0.39847859740257263, -0.3129329979419708, -0.3310185968875885, 0.19646614789962769, 0.22215525805950165, 0.34894001483917236, 0.5853389501571655, -0.881708562374115, -1.7836869955062866, 1.1921592950820923, -0.7375122308731079, -0.029612882062792778, 0.018887721002101898, -1.2869372367858887, 0.3350036144256592, 0.7731116414070129, -1.139211893081665, 0.771064043045044, 0.01585235632956028, 0.6770585179328918, -0.3312377333641052, -0.6043355464935303, 0.34874141216278076, -0.7175315618515015, 0.1837865561246872, -0.3334499001502991, -0.497018963098526, -0.4581386148929596, 1.1937339305877686], [-0.1475975513458252, -0.5073489546775818, -0.2222835272550583, -0.046787064522504807, 0.20238570868968964, 0.8012226819992065, -0.7887625694274902, -0.7291619181632996, 0.5366185307502747, -0.29481780529022217, -0.46963685750961304, -0.626106858253479, 0.004791982006281614, -3.17002272605896, -0.8381258249282837, 0.8662643432617188, -0.06822773069143295, 0.12126777321100235, -1.2938488721847534, -2.0163605213165283, -0.6633201837539673, 1.4001823663711548, -0.5482015013694763, -0.6897605061531067, -0.4262969195842743, -0.3976907730102539, -0.18580816686153412, 0.7703490853309631, -0.5468032956123352, -1.3165781497955322, 1.0231088399887085, -0.8318039774894714, 1.0499927997589111, -0.06913899630308151, -0.9169610142707825, 0.18862874805927277, 0.24986328184604645, -0.7331387400627136, 0.3979184329509735, 0.4947761297225952, -0.5149025321006775, -1.0722267627716064, -0.1801455169916153, -0.7286790013313293, -0.6365666389465332, -1.1331775188446045, -0.5099487900733948, -0.21559157967567444], [-1.6674766540527344, -0.05969646945595741, -1.097583532333374, 0.7104862332344055, 1.19892156124115, 0.03625760227441788, -0.16492792963981628, -0.27652108669281006, 0.6509904861450195, 0.5120078325271606, 0.16354681551456451, 0.14226728677749634, 0.3581089377403259, -1.7219619750976562, 1.0490986108779907, -0.3902641832828522, -2.0873398780822754, -0.27948057651519775, -0.9963297247886658, -0.05697764828801155, 0.930594801902771, -1.1512947082519531, 0.17901106178760529, 2.316168785095215, 1.4353312253952026, 1.4882359504699707, 1.4268449544906616, 0.5088478326797485, -0.24599342048168182, 0.8550191521644592, -0.4230539798736572, 1.2905735969543457, -0.7856351733207703, 0.007839047349989414, -1.3060383796691895, 0.7003212571144104, -0.2645052373409271, 1.405397653579712, 1.120520830154419, 0.1572338193655014, 1.2411260604858398, 0.6764851212501526, -0.20309647917747498, -0.4722106456756592, -0.23474381864070892, 1.3413361310958862, 1.5031929016113281, 0.9647682905197144], [2.6763813495635986, 0.22726522386074066, 1.3199853897094727, 0.5579659342765808, 0.3263304829597473, 0.5702598094940186, 0.33290600776672363, -1.183565378189087, -0.30473795533180237, -0.6422874927520752, 0.24292314052581787, -0.4254455864429474, 0.11903221160173416, 0.13371294736862183, 0.26593688130378723, -2.0574541091918945, -0.2583794891834259, 1.2442032098770142, 0.6743271946907043, -1.5547354221343994, -1.1037425994873047, -0.09227114915847778, -1.194429874420166, 0.44608017802238464, 0.9914326667785645, -1.457990288734436, -0.7402060627937317, 0.6690405607223511, 1.4593333005905151, 1.2455782890319824, 0.11132284998893738, -1.2211990356445312, 0.4840752184391022, 1.9864460229873657, -0.3483560383319855, 0.017730843275785446, -0.15085895359516144, -0.408640056848526, -0.4738440215587616, 0.37736040353775024, -0.6128852963447571, 0.5109583139419556, -1.0527403354644775, 0.9663093090057373, -0.9819652438163757, 2.4648594856262207, -0.4526936113834381, 0.05240819230675697], [0.06434764713048935, 0.5882725715637207, 0.020309725776314735, -0.6339250802993774, -0.9419956803321838, 1.0877997875213623, -0.7164368629455566, 1.0316424369812012, 0.22706468403339386, 1.276058316230774, -0.5985484719276428, 0.038619473576545715, 0.9488453269004822, 0.21656158566474915, 1.200340747833252, 0.445221483707428, -0.660415530204773, -0.4164339601993561, -0.12758037447929382, 1.052957534790039, -0.5251949429512024, 0.9737565517425537, -1.352246642112732, -0.06110025942325592, -0.7812129855155945, 0.7438442707061768, 0.00961562804877758, 0.7401317358016968, -0.30644968152046204, -2.060896158218384, -0.634552538394928, 0.08925583213567734, 0.23780511319637299, -1.345211148262024, 0.12195176631212234, -0.33727842569351196, -0.06604614853858948, 0.4706260859966278, 0.55198734998703, -0.7353076338768005, -0.8965147137641907, -1.04900324344635, 0.1657605916261673, 0.8393901586532593, 0.02020563930273056, -0.9127650260925293, -0.08850046992301941, 1.6605607271194458], [0.012969598174095154, -0.17495489120483398, -0.09014901518821716, 1.0456446409225464, 0.7926206588745117, -0.3046579658985138, -0.19553907215595245, -1.4474965333938599, -0.9184353351593018, -0.18792614340782166, -1.7095180749893188, 1.0089216232299805, 0.543097734451294, -0.46964696049690247, 0.3131139278411865, 0.6007949113845825, -0.5490915775299072, -0.34293362498283386, -0.11141005158424377, 0.6349584460258484, 0.9573575854301453, -0.057353463023900986, 1.3190152645111084, -1.453645944595337, 1.1279175281524658, 1.3958637714385986, -0.7464175224304199, 0.08116016536951065, -1.0066179037094116, -0.7687801122665405, 1.977758765220642, -0.7322400808334351, -1.3379307985305786, -0.08712282031774521, 0.26557251811027527, 0.6159955263137817, 0.7575456500053406, 0.7026175260543823, -0.789152979850769, 0.39735522866249084, 0.23365703225135803, -0.07244180142879486, 0.5230313539505005, 0.44497406482696533, -1.7161163091659546, 0.19702556729316711, -0.43456095457077026, 1.176454782485962], [0.31806299090385437, 2.1446731090545654, 1.3219432830810547, -0.4844789505004883, 0.18829235434532166, -0.24069486558437347, 0.4513861835002899, 0.1775672733783722, -0.6972098350524902, 0.049125317484140396, -0.980828583240509, 1.1828956604003906, 0.7601694464683533, 1.4263126850128174, 0.6454064846038818, -0.27618762850761414, 0.4578463137149811, 1.4135265350341797, 0.23818551003932953, 1.438398838043213, -1.643531084060669, 1.0372174978256226, -1.4765863418579102, -0.2648536264896393, 0.6199542284011841, -0.47589531540870667, 2.019669532775879, 0.2835919260978699, -0.5153506994247437, -0.09971647709608078, 0.07969491928815842, 1.7454874515533447, -0.064224474132061, -0.3968546986579895, -0.4796628952026367, 0.7397157549858093, 0.08717644959688187, -0.21179090440273285, 0.04821062833070755, 0.5144385695457458, -0.5088745951652527, -1.037854790687561, -1.0787378549575806, 0.27786770462989807, 0.30361947417259216, -1.7614890336990356, 0.054288990795612335, -0.28214892745018005], [1.5366458892822266, 0.19585217535495758, 0.7218961119651794, 0.4611709415912628, 1.1216706037521362, -1.2771273851394653, -0.32334277033805847, -0.5453163981437683, 0.8402338027954102, 0.21048897504806519, -1.1631306409835815, 0.7015267610549927, 0.6766374111175537, -0.17672500014305115, -0.04728507623076439, -0.7888217568397522, -0.3937358260154724, 0.13417932391166687, -2.0056164264678955, -0.5961311459541321, -0.07915204763412476, 0.43025076389312744, 1.459903597831726, -1.9370182752609253, -0.43311285972595215, 0.7215791940689087, 1.4217203855514526, -0.6756963133811951, 0.057957977056503296, -1.5311270952224731, 0.6698507070541382, 1.5389647483825684, -2.926442861557007, 0.5136327743530273, -0.03910655900835991, -0.9585457444190979, 0.37557584047317505, 0.5366184711456299, 0.7176346778869629, 0.7898635864257812, -2.367687702178955, 1.189746379852295, 0.984055757522583, -0.40885815024375916, 0.12238118052482605, 0.6666378378868103, -1.266461730003357, -0.9529404640197754], [1.6250520944595337, -2.467015266418457, -2.167768716812134, -0.9451210498809814, -0.28248074650764465, 0.09590786695480347, -2.0734777450561523, -0.42705458402633667, -0.062222931534051895, 1.3233296871185303, 0.1699582189321518, -0.3374409079551697, -0.34528130292892456, -0.06762860715389252, -0.4111084043979645, 0.3982298970222473, 1.5010302066802979, -0.2116706669330597, 0.09553265571594238, 1.8510804176330566, 0.5089043974876404, 0.9292162656784058, 1.1648368835449219, 0.02764555811882019, -0.7118472456932068, 0.6352178454399109, 0.6278464198112488, -1.0208439826965332, 1.189892292022705, -1.0267447233200073, -0.7547001242637634, 0.43378329277038574, 0.5636366009712219, 0.6232216358184814, -0.8950167894363403, 0.11350713670253754, -0.018903205171227455, -0.15707138180732727, 0.09610498696565628, 0.3541027307510376, -1.2878895998001099, -1.9111024141311646, -1.5271480083465576, -0.15890473127365112, 0.7515419125556946, -1.1341605186462402, -0.5301387906074524, -2.2673349380493164], [-0.2293500155210495, 0.3484174311161041, -0.07816202938556671, 0.5214697122573853, -0.7119527459144592, 0.5033046007156372, 1.0020346641540527, -0.039309628307819366, -1.8588663339614868, -1.581556797027588, -0.9043020009994507, -0.9083003401756287, 0.4952560365200043, 0.467507541179657, 0.6804110407829285, 0.09134926646947861, -0.23374947905540466, 0.6298298835754395, -0.6645513772964478, -0.47858622670173645, 0.8097280859947205, 0.15115925669670105, 1.6378655433654785, -0.7918314337730408, -2.023282051086426, -0.34660613536834717, 2.6243135929107666, -1.4503686428070068, 0.8643932342529297, 0.036512237042188644, 0.7709936499595642, -0.4720675051212311, 1.2093677520751953, 1.4890621900558472, 1.3021258115768433, 0.8691080212593079, 0.49707022309303284, 0.316885381937027, 0.04148854687809944, 0.06356503814458847, 0.385720431804657, 1.3582648038864136, -1.150040626525879, 0.5334911346435547, 0.2725769579410553, -0.2205176204442978, -1.24470055103302, -0.510202169418335], [0.7099438905715942, 0.6374759078025818, 0.5543227195739746, 1.4614617824554443, 2.45029354095459, 1.577956199645996, 1.6681095361709595, -1.2202165126800537, 0.07242853194475174, 0.14818479120731354, 1.117432951927185, -0.8270400166511536, -0.8021145462989807, 1.2805267572402954, -0.4145446717739105, -0.30955690145492554, 0.12000327557325363, -0.008482788689434528, -0.42708897590637207, -0.43646636605262756, 0.03691736236214638, -1.065980315208435, -0.401822566986084, 0.25589850544929504, -0.103873610496521, 1.2540363073349, -0.1149420291185379, 1.384848713874817, 0.6248911619186401, 0.01631433144211769, 0.5582399368286133, 1.0420161485671997, -2.3735039234161377, -0.5462411642074585, 1.0924572944641113, -0.6305049657821655, 0.25278207659721375, -1.1009252071380615, 0.727971076965332, 1.8999247550964355, 2.1625723838806152, 0.7660757899284363, 0.9350146651268005, -0.4629460275173187, 0.328845351934433, 0.37752801179885864, -0.289825439453125, 0.0347551666200161], [-2.071669816970825, 0.07515750080347061, -0.3285285234451294, -0.15466323494911194, 0.6208537220954895, -0.14001347124576569, -1.665097951889038, 0.12798871099948883, 0.1471828669309616, -1.1999597549438477, -1.2430157661437988, -0.3857015073299408, 0.01735660806298256, -0.8430129885673523, 0.8378881812095642, -0.7419652938842773, 0.8387546539306641, -0.571751594543457, 1.0561213493347168, -0.2693261206150055, 1.0075799226760864, -1.4916592836380005, -0.9872399568557739, 0.8863064646720886, -0.11406364291906357, -2.49564528465271, 0.7106439471244812, 1.9066773653030396, 1.615096092224121, 0.873353123664856, 1.1143289804458618, 0.4357626438140869, -0.10730631649494171, -1.4079982042312622, 0.3330431580543518, -0.7915242910385132, 0.46651509404182434, -0.24761448800563812, 1.4582666158676147, 0.37571007013320923, 0.22624193131923676, -0.7726773023605347, 0.735897958278656, -1.3746256828308105, 1.608534574508667, -0.7123808264732361, 0.3590719997882843, -1.0715426206588745], [0.3023967742919922, 0.0025006341747939587, -0.01879848912358284, -0.9259653687477112, -0.19779451191425323, -1.0778526067733765, -0.3887643814086914, 0.3418899178504944, -0.6646395921707153, -1.284420371055603, 1.6937114000320435, -2.2055423259735107, 0.5636095404624939, 1.3454207181930542, 0.6725084185600281, -0.4540102183818817, -1.634581208229065, 0.2754089832305908, 1.0487607717514038, 0.2705409824848175, -0.5303321480751038, -0.5025096535682678, 0.043337542563676834, -0.43588724732398987, 1.4024081230163574, -1.1656405925750732, -0.46644067764282227, -1.122241735458374, 1.4903043508529663, 0.6783208250999451, -2.4973855018615723, 1.2119297981262207, 0.3711586892604828, -0.4425358176231384, -0.4523075520992279, -1.0224741697311401, 0.10782467573881149, 0.8885493278503418, -0.062330976128578186, 0.48314914107322693, 0.8236916661262512, 0.5047079920768738, 1.0140947103500366, -0.3953564465045929, -1.906719446182251, -0.8876585364341736, 1.363999366760254, 0.7105095386505127], [0.4225591719150543, 1.3425365686416626, 0.14461436867713928, -0.3204641044139862, 0.150258406996727, -2.5728402137756348, -0.05091166868805885, -0.41711556911468506, -0.2128930538892746, -1.5019320249557495, -1.573637843132019, 0.7206003665924072, 0.8961735963821411, -0.2642133831977844, 1.1565426588058472, 0.18033172190189362, -0.47230780124664307, -0.2098279595375061, -0.5541474223136902, 1.8804985284805298, 0.4291781485080719, 0.02339467965066433, -0.6487085223197937, -0.7847160696983337, -2.37168025970459, 0.3899115025997162, 0.3837577700614929, 0.10981909930706024, -0.28167080879211426, -1.7124346494674683, -1.1311748027801514, -0.15846097469329834, 0.20041482150554657, 1.7622019052505493, -1.4060369729995728, -0.7319849729537964, -0.17873108386993408, -0.8992432355880737, -0.28263556957244873, -1.7266253232955933, -0.11528604477643967, 0.47686079144477844, 1.6772576570510864, 1.4949915409088135, -0.6425619125366211, 0.49063757061958313, -1.3272448778152466, 2.271528720855713], [-1.1150414943695068, -0.16537275910377502, 1.331164836883545, -1.441572904586792, -0.9034568667411804, -1.097862720489502, -1.0763367414474487, -0.20122459530830383, -1.4055794477462769, -0.9992811679840088, 0.11216918379068375, -0.1612250804901123, -0.18707311153411865, -0.9320897459983826, -0.9775558710098267, 0.4834630787372589, 0.8494554162025452, 1.7782131433486938, 0.13153626024723053, 0.7936208844184875, 0.1671421080827713, -0.10155734419822693, -0.4592463970184326, 0.670691967010498, -1.3076155185699463, 0.24196822941303253, 0.9156755805015564, 1.0863010883331299, -0.9676233530044556, -0.2015015035867691, -0.006061372812837362, 0.31404441595077515, -1.066721796989441, -1.2995105981826782, 0.49184733629226685, 1.9667282104492188, -0.5582597851753235, -0.07880112528800964, -0.08369481563568115, -1.127456784248352, 0.6320886611938477, -1.6469422578811646, -1.8963968753814697, 2.3301374912261963, 0.183977872133255, 0.7357082962989807, -0.4533745348453522, -0.5913152098655701], [0.1996079683303833, 0.027938228100538254, 1.9698143005371094, 1.118599772453308, -1.0476789474487305, -0.8312860727310181, 0.42194342613220215, 0.1805523931980133, -0.5966103076934814, -0.8469655513763428, -0.9083489179611206, -0.8173682689666748, 0.04351377114653587, 0.08004966378211975, -0.916316568851471, 0.45395368337631226, 0.45834094285964966, -1.248043179512024, -2.1939218044281006, 1.9192101955413818, -1.2438892126083374, 1.6845613718032837, -0.7889702320098877, -0.9593088626861572, 1.928966760635376, 1.2818812131881714, -0.5479032397270203, -0.5946925282478333, 0.2364872395992279, 0.9129944443702698, 1.2316285371780396, 0.7958283424377441, 0.2095821499824524, 0.44377192854881287, 0.7246333360671997, -0.18335303664207458, -1.0868182182312012, -0.6361326575279236, 0.47149378061294556, -2.5093507766723633, 0.2988192141056061, -1.4085952043533325, -1.9216861724853516, 0.4744953215122223, 0.7455469369888306, 0.3756130337715149, -1.7557899951934814, -1.2081944942474365], [-1.4040900468826294, 0.24201442301273346, -0.20903730392456055, -1.5287455320358276, -1.389275312423706, 0.4499300718307495, 1.0547256469726562, 0.19565564393997192, 0.49392661452293396, -0.8209494948387146, 0.64737868309021, -0.5129528045654297, -1.0273765325546265, -1.2976877689361572, -0.6891791820526123, 0.24034039676189423, 1.2200912237167358, 0.359859436750412, 1.3219069242477417, -0.3734656870365143, 2.2314043045043945, 0.43957027792930603, 0.06694162636995316, -0.47434625029563904, 0.1100180447101593, -0.590691328048706, -0.40172079205513, -1.0895836353302002, -0.2240092009305954, -0.7428416013717651, 1.270220398902893, 1.4394811391830444, 0.2710292935371399, -1.8892090320587158, 0.7116971611976624, -1.5131398439407349, 1.2992300987243652, -0.39937183260917664, 0.16935746371746063, 0.43242600560188293, 0.4768224358558655, -0.8566437363624573, -1.4689199924468994, 0.47584980726242065, 1.1753019094467163, 0.6467571258544922, -0.8731933832168579, 0.7256491780281067], [0.6893233060836792, 1.1236737966537476, -0.8485471606254578, 0.6957883834838867, 0.5710025429725647, -0.16736896336078644, -1.1819791793823242, 1.5147705078125, 0.9276435971260071, 0.21880748867988586, 0.5636913180351257, 0.4784857928752899, 0.9570580124855042, -0.08241767436265945, -1.5220720767974854, 0.03651852905750275, 0.1822124421596527, 0.653065025806427, 0.6844170689582825, 0.9407042860984802, 0.5040878057479858, 0.5724666714668274, -0.33570805191993713, 0.7532041072845459, -0.5620448589324951, 0.2783491611480713, 0.3513141870498657, 0.07742448896169662, 1.3856149911880493, 0.8334845304489136, -1.154815435409546, -1.1799030303955078, -0.3075234293937683, 0.17882414162158966, 0.8006520867347717, -0.6152735948562622, 0.13417218625545502, 0.6180950403213501, -0.5637153387069702, 0.17071540653705597, 0.2379876971244812, 1.8196158409118652, 0.4513496458530426, 0.03491370379924774, -3.6745846271514893, -0.5266491174697876, -0.3300863802433014, 0.3124854564666748], [0.09644132107496262, -1.6017605066299438, -0.8342980146408081, 0.12260839343070984, 1.4349623918533325, -1.6147582530975342, -0.3943023979663849, 0.4413068890571594, -0.0818723812699318, 0.1407950222492218, 0.9739969372749329, 0.4650156497955322, -2.0019686222076416, 1.0310032367706299, 0.08464012295007706, 0.40516772866249084, 1.3178298473358154, 1.1131467819213867, -0.745586633682251, 2.1593410968780518, 0.4701140224933624, -1.3541066646575928, -1.253636360168457, -0.8473615646362305, 0.5159394145011902, 1.446218729019165, -0.39256805181503296, 0.0437680259346962, 0.9839560389518738, -0.02305576764047146, 1.3469079732894897, 0.7952467203140259, -1.3689992427825928, 0.5777390599250793, -1.4430615901947021, 0.02242695726454258, -2.006789207458496, -0.3747725486755371, 0.6396048665046692, 0.2985272705554962, 0.41067373752593994, 0.06069924309849739, 0.07337432354688644, 0.8085152506828308, 0.25806665420532227, 2.2907679080963135, 0.8708881735801697, -0.2923074960708618], [0.5823330879211426, -0.22954939305782318, -0.12788920104503632, -0.07207706570625305, -1.2170393466949463, -1.2719632387161255, -0.896397590637207, -1.0220204591751099, 0.9402267932891846, 0.3551943600177765, -0.5616006255149841, -0.1522277593612671, 0.6341856718063354, -0.3492718040943146, 0.10432930290699005, -0.1528133898973465, -0.07389242947101593, -1.8490302562713623, 0.6731423139572144, -0.6494314670562744, -0.2218099981546402, -1.1331688165664673, -0.6463794112205505, -0.9218225479125977, -1.8237521648406982, 0.22471670806407928, -0.20292650163173676, -1.5015811920166016, 0.5348508954048157, -0.29660457372665405, -1.1474454402923584, 1.2981427907943726, -0.6971744298934937, 0.01740180514752865, -0.2861948013305664, 0.0590820349752903, -0.6517021656036377, 0.25408855080604553, -0.5700310468673706, 0.14371459186077118, -1.3410942554473877, 1.0543186664581299, 1.1793739795684814, -0.6616085171699524, -0.8724558353424072, -0.04343303292989731, 0.10616592317819595, 1.1605135202407837], [-0.45951035618782043, -0.5000690221786499, 0.04568593204021454, -0.9962860941886902, 0.3825879693031311, -0.3884938657283783, 0.3715515732765198, -0.994282066822052, 0.5964105725288391, 1.0553030967712402, -1.436227798461914, -0.45128366351127625, -0.14708352088928223, -2.408393383026123, -1.521113395690918, -1.163431167602539, 0.43619197607040405, 2.791328191757202, -1.1536756753921509, 0.22814103960990906, 0.264227956533432, 0.9740851521492004, 0.42998701333999634, 1.074265718460083, -2.4496898651123047, 0.6525598168373108, 0.12386426329612732, -2.188350200653076, 0.9883859157562256, 0.47791680693626404, -0.5303457379341125, -0.6384183764457703, -1.691801905632019, -1.8682948350906372, 0.5175291895866394, -0.37944185733795166, -0.41995519399642944, -0.42635592818260193, 1.2205640077590942, -0.3619299530982971, -0.2329212725162506, 0.20592822134494781, -0.8509573936462402, -0.0007044695666991174, -0.7545336484909058, -2.643406867980957, 1.3908430337905884, 1.4439595937728882], [0.7201232314109802, -1.178748607635498, -0.2692910134792328, -1.2444477081298828, -0.7458446621894836, 0.4772716462612152, 0.3215779662132263, 0.2158551961183548, 2.1953437328338623, -0.5323281288146973, 0.8199570775032043, 0.9741912484169006, 1.1191858053207397, 0.9893199801445007, -0.6631916761398315, -0.10252346098423004, -0.6871302723884583, 0.48495209217071533, -0.7586950063705444, -0.257035493850708, -2.3139142990112305, 1.0467108488082886, -0.8819230198860168, -1.7904590368270874, -1.5913379192352295, -0.2207215428352356, 1.7893662452697754, -0.2471269816160202, 0.7586243152618408, 0.6491246223449707, 1.0060492753982544, 0.4565884470939636, -0.21369968354701996, -0.44788438081741333, 0.409494549036026, -0.1757349669933319, -2.160632848739624, -0.6189202070236206, 0.9480952024459839, -0.00016496062744408846, -0.596889853477478, -1.3994545936584473, -2.094635009765625, -0.06091798469424248, -0.017308201640844345, 0.637273371219635, -0.8177294731140137, 0.2325223982334137], [0.6226163506507874, -1.8128690719604492, -0.8789075613021851, -1.0501108169555664, 1.3917449712753296, 0.4052067697048187, -0.7199410200119019, 0.6195334196090698, 0.9200972318649292, -0.5548057556152344, 0.323496550321579, -1.245023488998413, 0.75716632604599, -0.229519784450531, 0.7940715551376343, -0.17660364508628845, -2.1301305294036865, -0.4307901859283447, -0.6718463897705078, 0.13732700049877167, -0.12580281496047974, -1.6719884872436523, -1.211138129234314, 0.4700407087802887, 0.7451342940330505, -0.1625274121761322, -0.7006457448005676, 1.4364861249923706, -0.9089572429656982, -0.17061221599578857, 0.13868193328380585, 0.21783217787742615, 0.46891167759895325, 0.13158462941646576, 0.7380391955375671, -0.8157765865325928, 1.3735932111740112, -0.023169338703155518, 1.1829493045806885, 0.374255895614624, -0.3230406641960144, 0.20520448684692383, -0.2787006199359894, -0.027240049093961716, 2.53165602684021, 0.32643187046051025, -0.13309934735298157, 0.6540675163269043], [0.7618138790130615, -0.6123720407485962, -0.09946657717227936, 2.340766429901123, -0.7223783731460571, 0.1855701357126236, 0.5559169054031372, -0.3709211051464081, -1.1591217517852783, 1.1226239204406738, 0.8259462118148804, 0.15904657542705536, 0.8336822390556335, -0.3167688250541687, -1.4710971117019653, -3.1827211380004883, 0.6799901127815247, -0.8677128553390503, -0.8133667707443237, -1.0071797370910645, -1.265264868736267, 2.3102166652679443, 0.03860234469175339, 2.20072078704834, -1.4133487939834595, 1.364343285560608, -0.48275068402290344, 1.073253870010376, -1.1318403482437134, 1.0196452140808105, -0.5981494188308716, 0.718315601348877, 1.060067892074585, 1.9698262214660645, 0.9718409180641174, -1.43821120262146, 2.161177635192871, 0.29357412457466125, -0.08283347636461258, -0.5857970714569092, -0.5111933350563049, -1.9836686849594116, -0.14945930242538452, -1.014234185218811, 0.23524336516857147, -1.6958298683166504, 0.6715670824050903, -0.7736956477165222], [-2.1370291709899902, 0.11066512018442154, -0.0669671818614006, -0.7816555500030518, -0.972520112991333, 0.4403204917907715, 0.8295466303825378, 0.35854265093803406, -0.8536496758460999, -1.153616189956665, 0.6682882905006409, -0.9073415994644165, -0.14089111983776093, -0.5832177996635437, 0.3675081431865692, 1.4523100852966309, 1.1203012466430664, 0.04603691026568413, 1.5040619373321533, 0.0931394174695015, 1.057835578918457, 0.5096249580383301, 0.9418665766716003, -1.716943383216858, 0.01068346668034792, 3.547086477279663, -0.3635408878326416, 0.7194666266441345, -0.831270158290863, -0.43446415662765503, -1.869854211807251, 0.6748772263526917, 1.0995957851409912, 0.7549083828926086, 1.1628659963607788, -0.2837993800640106, -1.8929071426391602, 1.4036837816238403, 0.32765641808509827, 1.7105612754821777, -1.2740570306777954, 1.8796088695526123, 1.8091583251953125, 0.2604402005672455, 0.4972529411315918, 0.3463292121887207, 0.5419933199882507, 1.4124540090560913], [0.9687743186950684, -0.3658572733402252, -0.9834737777709961, 1.2545340061187744, -0.7951142191886902, 0.5455002188682556, -1.1466212272644043, 0.26726192235946655, -0.8939304947853088, 0.18962867558002472, 1.021986722946167, -0.06467384845018387, -0.9675483107566833, 0.7165278196334839, 0.5362623333930969, -0.03875618055462837, 1.1249767541885376, -1.5129450559616089, 0.7192348837852478, -1.3113269805908203, 0.6639871001243591, 0.7642291188240051, -1.757346749305725, -0.4462554156780243, 0.509476363658905, 0.7349181771278381, 0.24903225898742676, 0.7488579154014587, -0.349840372800827, 1.274577021598816, 1.28103768825531, 0.3198074400424957, 1.0623105764389038, 0.7604671120643616, 2.5451252460479736, 0.6614182591438293, 0.30930808186531067, -0.3314334452152252, -0.1680404543876648, 1.5984039306640625, 0.22068463265895844, -0.2830962538719177, -0.24417568743228912, -1.0743836164474487, 0.7016474008560181, -0.03471992909908295, 1.1080191135406494, 1.3814090490341187], [-2.1017582416534424, -0.18081845343112946, 0.5365697741508484, -1.210134744644165, -0.5510759949684143, 0.3527604043483734, -0.5601486563682556, 1.6135361194610596, -1.2127573490142822, -0.7751315832138062, -0.8696531057357788, -0.5101941227912903, -0.24603720009326935, 1.3278412818908691, -0.6143708229064941, -1.3859776258468628, 0.06673362106084824, -1.7185373306274414, -0.18419581651687622, -1.5123270750045776, 1.7474666833877563, 1.3248974084854126, -0.10213836282491684, -1.0689548254013062, 1.799751877784729, 0.5846239924430847, 0.6541666984558105, 0.1937251091003418, 1.8493025302886963, 0.4630233943462372, 0.1775699406862259, -0.7192872166633606, 0.3558734655380249, 0.024613168090581894, 1.228206753730774, -1.1224606037139893, 0.09296676516532898, 0.09174422174692154, -1.8660229444503784, -1.5766891241073608, 0.7184165120124817, 1.6800521612167358, 0.22608554363250732, 0.7206746339797974, -1.1574174165725708, -0.7076517343521118, -1.4659538269042969, 0.8365885019302368], [-0.6815012693405151, 1.1778620481491089, 1.5955373048782349, 0.24565649032592773, 0.26397910714149475, -0.8439972996711731, 3.3478777408599854, 0.37785983085632324, -0.7851803302764893, -1.2933359146118164, 0.8051919341087341, 0.26484760642051697, -1.079835295677185, 0.27396678924560547, -2.351323127746582, -1.1564871072769165, -0.43518808484077454, -0.22203175723552704, -1.9707136154174805, -0.13689225912094116, -0.9032351970672607, 0.8149209022521973, 1.3274567127227783, -0.3033829927444458, 1.1548880338668823, 0.7145061492919922, 0.19806182384490967, -0.979207456111908, 2.2342584133148193, 1.0623385906219482, -1.2911381721496582, -1.309504508972168, 0.8547552227973938, 0.5060067176818848, 0.010564806871116161, -1.0488417148590088, 1.53329598903656, 1.739414095878601, -0.28353849053382874, 0.5379345417022705, 2.485151529312134, 0.5588517785072327, 0.7631937861442566, -0.007236171513795853, -0.5244420766830444, -0.8247497081756592, -0.7655588388442993, 1.84154212474823], [0.20646117627620697, -1.4124428033828735, 1.346663475036621, -1.5691664218902588, 1.245935082435608, -0.10654877871274948, 0.6352317333221436, 1.6420538425445557, -0.40569958090782166, 2.651294708251953, -0.4111376404762268, -1.547194242477417, -0.6365416646003723, -0.29880890250205994, 0.48486194014549255, 0.22375760972499847, -1.144160270690918, -0.436962366104126, 0.015212171711027622, -0.07213189452886581, 0.1382591277360916, -1.1773207187652588, -0.254252165555954, -0.17916782200336456, 0.30024585127830505, 0.9143192768096924, -1.8224893808364868, -0.46556395292282104, -1.7792993783950806, 0.46348533034324646, -0.18851663172245026, -0.6218823790550232, 0.628323495388031, -1.6823991537094116, -1.426474690437317, -0.5980497598648071, -0.44722095131874084, -1.2145910263061523, 0.6248871088027954, -1.6781779527664185, -0.340917706489563, -0.1104465201497078, -0.10243456810712814, -2.065467119216919, -2.2521438598632812, -1.5570507049560547, 0.19156397879123688, 0.6935350894927979], [-1.8065871000289917, -1.0752770900726318, 0.6908960938453674, 0.11297120898962021, -1.375095248222351, 1.0900497436523438, 1.13936448097229, -0.7415313720703125, 1.452090859413147, 0.7948498725891113, 0.9531276226043701, -0.9844204783439636, 0.8597612977027893, -0.3439706861972809, 1.2223724126815796, 0.08361859619617462, -1.0802078247070312, 0.28174397349357605, 0.4874800443649292, 0.054923590272665024, 1.914339542388916, -1.1621171236038208, -0.0822296217083931, -0.020062778145074844, 0.5782806277275085, -1.1659516096115112, -1.0123395919799805, -0.4989454448223114, 0.22413630783557892, 0.5768210291862488, -0.09925481677055359, 0.23578715324401855, 1.5298227071762085, -0.5940380692481995, 0.08838234841823578, -0.06235677748918533, -2.182431221008301, -0.12478084862232208, 0.8569312691688538, -0.16437293589115143, 0.6004199981689453, 0.1572216898202896, -0.03393009677529335, 0.1948440819978714, 0.6588566899299622, -1.6945595741271973, 0.01140788197517395, -0.043468598276376724], [0.37943941354751587, 1.6300859451293945, -1.420055627822876, 0.23465010523796082, 0.053098246455192566, 1.45377516746521, 0.7350225448608398, -0.8527746200561523, 0.8693932890892029, -1.70957612991333, 0.48119261860847473, -0.9279163479804993, -0.5708004236221313, -0.12983480095863342, 1.1803126335144043, -0.3734218180179596, -1.3770354986190796, -0.8305293917655945, -0.7040239572525024, 1.2012847661972046, 0.6712320446968079, 0.26613566279411316, -0.05270318314433098, -2.3028695583343506, -0.6186448335647583, -0.8407690525054932, 0.6440213322639465, -0.5323320627212524, 0.8346309661865234, 0.15792399644851685, -1.5978248119354248, -2.738816261291504, 0.6394501328468323, 0.33872392773628235, 0.4121186435222626, -0.752225935459137, -0.24428552389144897, 0.8213235139846802, 0.48728957772254944, 0.7620257139205933, 1.844284176826477, -1.1982635259628296, -0.8247560858726501, 0.9751210808753967, 1.77095627784729, 0.9089474678039551, 0.7005495429039001, -0.9719077944755554], [2.0106165409088135, 1.3687456846237183, -0.07623939961194992, 0.15412312746047974, 0.41023707389831543, 0.8854984045028687, -0.1305760145187378, -1.053255558013916, -0.24026137590408325, 1.108983039855957, -0.4002288281917572, -0.2865687906742096, 1.552172064781189, -0.5669680237770081, 0.6399199962615967, 0.48690319061279297, -0.07799413800239563, 0.05822424963116646, 1.794629454612732, -1.7424184083938599, 0.2960304915904999, 0.29678645730018616, 1.1894408464431763, 0.6444091200828552, -1.6156096458435059, 0.21858470141887665, -0.5583403706550598, 0.44392409920692444, 0.20251120626926422, 0.20425842702388763, 0.3101963400840759, -0.0590345598757267, -0.7676640748977661, 1.0830974578857422, -0.3386281430721283, -1.4400044679641724, 0.17583011090755463, -0.17046470940113068, -0.043918076902627945, 1.579070806503296, -0.2818359136581421, 0.5958797335624695, -0.26107272505760193, 0.8925835490226746, -1.032694697380066, -0.14621376991271973, 2.6667027473449707, 1.9957762956619263], [2.1444942951202393, -0.1526055634021759, 0.3464096188545227, 1.384339690208435, -0.2709561288356781, 2.05011248588562, -0.19105671346187592, 0.9562250375747681, 0.5016380548477173, -0.5119940638542175, 1.308851957321167, -0.9615437984466553, 0.19743084907531738, 0.001420030021108687, 0.43050503730773926, -0.667285680770874, -0.8501861691474915, 0.6906697750091553, 0.6242346167564392, 1.1382793188095093, -0.26769396662712097, -0.7583880424499512, 0.7327179312705994, -0.4854431748390198, 1.6099486351013184, 0.738686203956604, -0.23825053870677948, -0.6537854075431824, 0.11254938691854477, -1.404718041419983, -0.3768061697483063, 0.03707736358046532, -1.0506709814071655, 0.7348477840423584, 1.5207065343856812, 0.8029777407646179, 0.6971965432167053, 0.9034853577613831, 2.262352466583252, 0.3513416647911072, -0.22798806428909302, -0.6017053723335266, -1.627611756324768, 0.5480300784111023, -0.4670145511627197, 0.2469349503517151, 0.23365142941474915, -1.8704273700714111], [0.1265127807855606, -0.10596293210983276, -0.26835787296295166, -1.1570106744766235, -0.7544442415237427, 0.4969707131385803, 0.021706033498048782, 0.6947084665298462, -1.1181775331497192, -0.6476526260375977, -0.9012219905853271, 2.1071577072143555, 0.1354784071445465, 1.0635467767715454, -0.789050817489624, -0.3871825039386749, -0.8486276268959045, -0.38958606123924255, 0.25131645798683167, 1.076322317123413, 0.539431095123291, -0.9164431691169739, 2.0163748264312744, 1.2820954322814941, -1.2926748991012573, 0.5047092437744141, -1.4208768606185913, 0.1505705714225769, 0.07237225025892258, -1.1132287979125977, 0.5722219944000244, 0.06802880018949509, 0.08736123889684677, 2.5125982761383057, -0.02979169227182865, 0.4676313102245331, 0.6936414837837219, 1.1183571815490723, 0.8492011427879333, 0.64594566822052, 1.635354995727539, -0.315182626247406, -0.5617683529853821, 1.8446528911590576, -1.9966487884521484, -1.3660426139831543, 0.5198724865913391, -0.8026320934295654], [-0.8615498542785645, -0.08807030320167542, 0.31694695353507996, -1.8811198472976685, -0.33181771636009216, -0.10095857083797455, 0.5481865406036377, -1.5623618364334106, -0.7982563972473145, -0.21881073713302612, 1.1592118740081787, -1.0980452299118042, 0.5951712131500244, 0.846919596195221, 0.7936364412307739, -0.05225467309355736, -0.6142011284828186, 0.9666932225227356, -0.08402151614427567, -1.0835758447647095, -0.08609941601753235, 1.1994774341583252, -1.0114071369171143, -1.2951198816299438, 0.5461826324462891, -0.41619884967803955, 0.31907713413238525, -0.24008148908615112, -0.5310056209564209, 0.42620542645454407, -0.6237815022468567, -0.7670696973800659, -0.31958264112472534, 0.6307390332221985, 0.23469975590705872, -1.0007888078689575, 1.2072556018829346, 0.9287772178649902, 0.38957512378692627, -0.5501097440719604, 0.35178688168525696, -0.2230249047279358, -1.774577021598816, -0.4901215732097626, 0.3099653422832489, 0.011933875270187855, -1.7959768772125244, -1.37632417678833]], "dtype": "float32"}, {"__nd__": [[-0.11633969843387604, -0.12751764059066772, 0.8991970419883728, -0.0014951976481825113, 1.1079388856887817, 0.4837525486946106, 0.38488608598709106, 0.5875586867332458, -0.29935193061828613, 0.5606343746185303, -0.024680249392986298, -2.2431116104125977, 1.011242389678955, 1.3175417184829712, -1.110406756401062, 0.7603110671043396, -1.5117355585098267, 1.5137073993682861, -1.1239805221557617, -0.6680135130882263, -0.5745437741279602, -1.971950888633728, -0.47966474294662476, -1.7864913940429688, -0.5786932110786438, -0.46172693371772766, 0.08731286227703094, 1.3052709102630615, -1.025569200515747, -0.35471677780151367, 0.38442063331604004, -1.3409039974212646], [0.0556710846722126, 0.1774643063545227, 0.9805100560188293, 1.6176743507385254, -0.2534412741661072, -0.7715389728546143, 1.0872650146484375, 0.0376557931303978, 0.02819606475532055, -0.04940705746412277, -0.308505117893219, 0.24934302270412445, -0.5076747536659241, 0.6903089880943298, 0.7810893058776855, -0.7387171983718872, -0.20370356738567352, -0.9325444102287292, -0.5498695373535156, -1.0830376148223877, -0.5926917195320129, 0.9236432313919067, 0.5814158320426941, 2.710008382797241, -0.5901788473129272, -2.027864456176758, -0.21794109046459198, -0.1550384759902954, 0.1868884414434433, 0.5216633677482605, 0.37737905979156494, -2.1686038970947266], [-0.14228804409503937, 0.6875975728034973, 1.1759321689605713, -0.789683997631073, -0.2035214751958847, -0.7688938975334167, -1.2311793565750122, -0.9238228797912598, 0.7793857455253601, -1.0546163320541382, 0.0250542052090168, 1.4855573177337646, -0.7710253596305847, 0.007801109459251165, 0.7295242547988892, 0.5734705328941345, -2.1935949325561523, -1.8381974697113037, 0.7947722673416138, 0.2623819410800934, -1.2064005136489868, 0.595139741897583, 1.7080819606781006, -0.6463825106620789, -0.1214856207370758, -0.518974244594574, 0.9517565369606018, 1.9563487768173218, 0.030477318912744522, 0.16448208689689636, 1.4317643642425537, -0.025765832513570786], [-1.2927261590957642, 0.7414723634719849, 1.2123218774795532, 0.38451430201530457, -1.1055893898010254, 0.12261585891246796, 0.3660297989845276, -2.1132898330688477, -0.38555440306663513, -0.28797397017478943, -1.4277007579803467, -1.0640442371368408, 0.437463641166687, -0.9759323596954346, 0.7445394992828369, 0.8761375546455383, -0.012651517055928707, -1.0930107831954956, 0.10142569243907928, -1.3928747177124023, 1.9366419315338135, -0.05143019184470177, 0.4924217760562897, 0.2445964515209198, -2.5488381385803223, 0.4419103264808655, -0.6990365982055664, 0.4095533490180969, -0.8677393198013306, -1.6238665580749512, -1.0549236536026, 0.9902921319007874], [-0.6852089762687683, 0.550258457660675, -0.004454384557902813, -0.9104132056236267, 0.01152038387954235, -0.03518088534474373, 0.055126164108514786, -0.18422533571720123, 1.6012996435165405, -1.174163579940796, 0.5432786345481873, -0.7504863142967224, -1.2375661134719849, -0.7965385317802429, -1.4662551879882812, -0.7918645143508911, -1.7594159841537476, -0.5964881181716919, -1.8892138004302979, 0.33605673909187317, -0.3207267224788666, 1.1806926727294922, 0.3438204824924469, 0.7483219504356384, 0.34826910495758057, -0.850784182548523, 0.5733973383903503, -0.14461301267147064, -1.4634026288986206, -0.9156538248062134, 0.20369333028793335, 0.1105840876698494], [-0.3190224766731262, -1.2342994213104248, -0.6065062284469604, -2.0413453578948975, 0.14315809309482574, -0.07475760579109192, 0.3144608438014984, -1.4239928722381592, -0.8672153949737549, 0.5157017707824707, -1.3717854022979736, 0.370291143655777, 1.4856700897216797, -0.4256797432899475, -1.07082200050354, -0.7653674483299255, -1.591869592666626, 0.3036736249923706, 1.3568252325057983, 1.5841506719589233, 1.4514933824539185, 0.560177206993103, 1.2317702770233154, 0.09168082475662231, 0.09074555337429047, -0.8770092725753784, -1.2379071712493896, -0.8108179569244385, -0.4042987525463104, -0.3533875346183777, 0.08376021683216095, 0.3898286521434784], [0.12108016759157181, 0.012693904340267181, -2.225803852081299, -1.0219026803970337, 0.11784886568784714, -1.3908897638320923, 0.35088998079299927, 0.5087823867797852, 0.669130265712738, -0.887937605381012, -0.7305781245231628, 0.32342329621315, -0.9503647089004517, 0.6495273113250732, 0.42590394616127014, 0.7211000919342041, 0.2696571350097656, 1.0088188648223877, 0.4410337805747986, -2.029283046722412, 0.3009032905101776, -0.16646708548069, -0.9090808629989624, 0.36218249797821045, -0.582420825958252, -0.20566871762275696, -0.9233859777450562, -0.017025666311383247, 0.7491623163223267, -2.0820019245147705, 0.28219276666641235, -0.1996600180864334], [-0.3661968410015106, -0.5233287811279297, -0.7650153040885925, -0.5269445776939392, 0.39702630043029785, 0.061671990901231766, -1.1323224306106567, -0.2602064609527588, 0.05095527321100235, 1.316166639328003, 0.2692820131778717, 0.9759503602981567, 0.4945027232170105, -1.9514672756195068, -0.34120333194732666, -1.3866833448410034, -0.6453176140785217, 0.06534693390130997, -0.3893830478191376, -0.8122485876083374, -1.3367806673049927, 0.7918499708175659, -0.0356159470975399, -1.0211089849472046, -0.9907326698303223, 0.19757422804832458, -1.762623906135559, -0.8125538229942322, 0.7739654183387756, 2.0609114170074463, -0.6451643705368042, -0.9285451173782349], [-0.9159837365150452, 0.9393765926361084, 0.48390161991119385, 1.572672963142395, -0.48491764068603516, 0.2665042281150818, 0.17644089460372925, -1.296654462814331, 0.1859695315361023, -1.565266489982605, -1.385408878326416, -2.0978760719299316, 0.3576522767543793, -2.343284845352173, -1.330100417137146, -0.6290490627288818, -0.9477626085281372, 0.6380214095115662, 0.2250266671180725, -2.5251431465148926, 0.1468394696712494, -0.9480963945388794, 0.22451914846897125, 0.4306207597255707, 0.900159478187561, 0.21224424242973328, 0.2371402531862259, 1.5833030939102173, -0.9846255779266357, 0.5183687210083008, 0.6188920736312866, -0.4198162853717804], [-0.1253846436738968, 0.39764708280563354, 0.7642645239830017, -0.24365362524986267, 0.7936556935310364, 1.6003581285476685, -1.623658299446106, -0.20039483904838562, 0.19636331498622894, 0.06441096216440201, 1.496514081954956, -0.5700610876083374, -1.9385656118392944, 0.6856744289398193, -2.054607629776001, 1.3404477834701538, 0.03141290694475174, 0.03366909921169281, -1.1005719900131226, -0.9014723300933838, 0.36690598726272583, -1.266408085823059, -0.5375194549560547, -0.6491401791572571, -2.1566641330718994, -1.381663203239441, 0.7666072845458984, -1.618784785270691, 0.42848333716392517, -0.6480607390403748, -0.187602698802948, 0.9540101885795593], [0.4575177729129791, 0.223592609167099, -0.33431124687194824, -1.9477653503417969, -0.03302142396569252, -0.07695026695728302, 0.9843473434448242, -0.9043303728103638, -0.7071465849876404, -0.30588480830192566, -1.5814012289047241, 0.4582723379135132, -1.2767479419708252, -0.03787289559841156, -1.7459050416946411, -0.5304059386253357, -1.3848917484283447, 0.041217103600502014, -0.6715531945228577, -0.36778804659843445, -0.09280640631914139, -0.831332266330719, 0.33103737235069275, 0.6492951512336731, 0.12317022681236267, 0.41863003373146057, 1.6313499212265015, 0.9889469146728516, 0.5390129089355469, -0.190634086728096, 0.053992755711078644, -1.165679931640625], [-1.0616955757141113, -0.7292212843894958, 1.0844719409942627, -0.752694308757782, -0.0007448674878105521, 0.5718072056770325, -0.5562214851379395, -0.6486843824386597, -2.342264175415039, 0.21597982943058014, 0.8026252388954163, 0.20120549201965332, 1.004173994064331, 0.8200783133506775, -0.8151248693466187, 0.45304420590400696, -1.1111118793487549, -0.37238994240760803, -0.07510847598314285, -0.9516725540161133, 1.0370862483978271, -0.768747866153717, -1.2625536918640137, -0.28621208667755127, 1.0750603675842285, 0.02373122051358223, -2.5430514812469482, 0.1640433818101883, 0.3736450672149658, 1.3408317565917969, 0.18799977004528046, 0.2987765371799469], [-0.45323771238327026, 0.7374553084373474, -0.5367925763130188, -2.0293920040130615, 0.9598062038421631, 0.41366761922836304, 0.2815074920654297, -0.9909146428108215, 0.7592639327049255, 0.36773207783699036, -1.166762351989746, 0.8029494881629944, 0.27861732244491577, -0.6146312355995178, -0.7496451139450073, 1.4633708000183105, 0.9356189370155334, -0.2793758809566498, 0.8534456491470337, -0.39940908551216125, 2.051561117172241, -0.7098560929298401, -0.07510112226009369, -0.04020783305168152, -0.15981842577457428, 0.5355185866355896, -0.06741020083427429, -0.26951661705970764, 1.1733410358428955, -0.4866010546684265, -0.5871139168739319, 0.3590022027492523], [0.20280557870864868, 0.08713017404079437, 2.5737931728363037, -0.20314429700374603, -0.6456308364868164, 0.05108172073960304, 1.302582025527954, -0.5810185074806213, -0.09574325382709503, 1.4309992790222168, -0.8179234266281128, -0.7531566023826599, -1.2830486297607422, 0.5156142115592957, 0.6272538304328918, -1.8367044925689697, -1.004655122756958, 0.43604597449302673, -2.0087838172912598, -0.7965567111968994, 0.7778804302215576, 0.14709481596946716, 0.2277553826570511, -0.7110991477966309, 0.8503966331481934, 0.8449262380599976, -1.213651180267334, 0.2955035865306854, -0.18042314052581787, 1.0773924589157104, 0.7995049357414246, 0.3920016288757324], [-0.3788638114929199, 0.2618580162525177, 0.6167540550231934, -1.95254647731781, 0.29487818479537964, 0.13416753709316254, 0.645561158657074, 0.8993092775344849, -0.20369644463062286, 0.4979338049888611, 2.4784066677093506, 0.1645280122756958, -0.5970378518104553, -0.5509659647941589, -0.055523522198200226, 0.7667087316513062, -0.6429256796836853, -0.4038279950618744, 0.6376104354858398, 0.32853269577026367, 1.4588741064071655, 0.7597063183784485, 0.20441441237926483, -0.3685084283351898, -0.20685626566410065, -0.9547168016433716, 0.5675915479660034, 0.3250430226325989, 0.2610636055469513, 0.1563684493303299, -0.2904418706893921, -2.9239323139190674], [0.010556023567914963, 1.252900242805481, -0.6942888498306274, 0.6423177123069763, 0.5708056688308716, -1.117163062095642, -0.14004650712013245, -1.1401011943817139, -0.7266077995300293, 1.4466215372085571, 1.3881691694259644, 0.39247918128967285, 1.1455405950546265, -1.2295551300048828, -0.6659594774246216, -0.4848445951938629, -1.1255855560302734, -0.7473539113998413, -0.4379497766494751, 0.09207142144441605, 0.07774810492992401, -0.005439160391688347, 0.1906474083662033, -1.0069211721420288, -1.1966791152954102, 0.003971989266574383, 0.4324246346950531, 0.14102432131767273, -1.9555164575576782, -1.0643165111541748, -0.09820561110973358, -1.1757444143295288], [0.5400282144546509, -1.347546100616455, 0.42735323309898376, 2.3645997047424316, -0.021523678675293922, -0.876228928565979, -0.15960632264614105, 0.7444517612457275, -2.027358055114746, -0.6893399357795715, 1.5111714601516724, 0.6761163473129272, -0.4603995978832245, 1.1569840908050537, 0.3776019811630249, -0.3556640148162842, 2.301082134246826, 1.447641372680664, 0.33487042784690857, 0.944589376449585, 0.7890565395355225, -1.3781981468200684, -1.2842403650283813, 1.6879792213439941, -0.6426885724067688, -2.5931284427642822, -0.5392575860023499, 0.5478420853614807, -0.6925951838493347, 2.685000419616699, 0.4392383098602295, 1.456939458847046], [-1.4787534475326538, 0.49675416946411133, 0.9870489239692688, 0.457009494304657, -0.7687510848045349, -0.4056985378265381, 0.16304290294647217, 0.7427881956100464, 0.6916679739952087, 0.21480490267276764, -0.8032078146934509, -1.1486115455627441, -0.6708012819290161, 2.0287704467773438, -0.05335264280438423, -2.057645559310913, -1.8662596940994263, -2.3340041637420654, -0.4846899211406708, -0.0016615240601822734, 0.718713104724884, 0.9504976868629456, -0.2611892819404602, 0.7726824879646301, -0.866791844367981, -0.36447417736053467, -1.1434457302093506, 0.24324433505535126, -0.16739489138126373, 0.4131709337234497, -0.19460082054138184, 2.0187363624572754], [-0.13909251987934113, 0.3837152123451233, 0.5530628561973572, -0.7567331194877625, -0.3875059187412262, 0.5812296271324158, 0.7336556315422058, 1.097383737564087, 2.177942991256714, 1.056379795074463, 0.4926024377346039, 0.973227322101593, -0.05100949481129646, -0.04367797076702118, -1.2896956205368042, 2.1057403087615967, 0.4468044936656952, 0.8100317716598511, -1.7387622594833374, 0.2609825134277344, 1.5785717964172363, -0.09104245901107788, 0.05724838376045227, -0.15095172822475433, 1.1188228130340576, -1.2191920280456543, 1.8958523273468018, -1.2578866481781006, 1.23232102394104, 0.39361804723739624, -0.9589065313339233, -0.23310694098472595], [-0.37463998794555664, 0.11773285269737244, 0.555381178855896, 2.507023811340332, -1.9159667491912842, -1.5851361751556396, 0.6230243444442749, 0.9668560028076172, 0.34486010670661926, -0.2498091757297516, -0.43158429861068726, -1.3560274839401245, 0.7059594988822937, 1.1356617212295532, 1.6994589567184448, -0.13911987841129303, -1.494328260421753, -1.6161781549453735, 0.26783260703086853, -0.036891963332891464, -0.88921719789505, -1.1246289014816284, 0.7090245485305786, -0.501483142375946, -1.2982968091964722, -0.5819753408432007, 0.45895421504974365, -0.7450371384620667, -0.592179000377655, 1.1653398275375366, 1.9494221210479736, 0.17286460101604462], [-1.5090278387069702, -0.7279869318008423, -0.07456070929765701, 0.5371668934822083, 1.4500457048416138, 0.2720523476600647, 0.10202833265066147, 0.03855104371905327, -0.43755051493644714, 2.0101966857910156, -0.45311230421066284, 0.30927228927612305, -0.2677818238735199, 2.7622127532958984, 1.2065898180007935, 0.7298503518104553, -0.4702369272708893, 1.022737741470337, -2.484506130218506, -0.4063103497028351, -0.9024650454521179, 1.0728836059570312, -0.7447695136070251, -1.8637797832489014, -0.42820143699645996, 0.8029932975769043, -0.9215019345283508, -0.6517073512077332, 0.49428144097328186, 0.6603038311004639, -0.7259435653686523, -0.28841522336006165], [0.573479413986206, 0.40567460656166077, -1.9404327869415283, -0.8246380686759949, 0.8680354356765747, -0.41463202238082886, 2.1362051963806152, -0.7498694658279419, 0.0068081216886639595, -1.1027874946594238, -0.5815296173095703, 1.7510323524475098, 0.04753843694925308, -0.3914277255535126, -0.5974352955818176, 0.8327252864837646, 0.5127053260803223, 1.8259000778198242, 0.9281127452850342, 0.4054279029369354, -0.611554741859436, 0.8882297873497009, 1.543426275253296, -0.6394025683403015, -2.311161756515503, 0.09505220502614975, 1.3031303882598877, -0.35947185754776, -1.8233972787857056, -1.0494955778121948, 0.17047815024852753, -0.6415103673934937], [-0.08089318871498108, -0.5468223690986633, 0.06875593215227127, -0.8792415261268616, -0.3414081931114197, -0.3879074156284332, -0.9638408422470093, -1.4932371377944946, -0.3486827313899994, 2.1749424934387207, -0.05686138570308685, -1.545083999633789, -1.9567129611968994, 0.3239412009716034, -1.8916739225387573, -1.9091418981552124, 0.059711914509534836, 1.321255087852478, 0.4671732485294342, -0.5065500736236572, 0.7315245866775513, 1.0353400707244873, -2.2027273178100586, 0.4009413421154022, 0.9872742891311646, -0.9179498553276062, 0.10257858037948608, -1.3701744079589844, -0.17657192051410675, -1.5336973667144775, 0.6129259467124939, -0.1051313504576683], [-0.7072453498840332, -0.06838557869195938, -0.7496488690376282, -0.1145923063158989, -0.175790473818779, -0.5393420457839966, -0.4551261067390442, -0.39784911274909973, 0.22016137838363647, -0.5613101720809937, 0.027635982260107994, 0.004016237799078226, 0.7285276055335999, 0.5318183898925781, -0.3900376558303833, 1.2652413845062256, 1.370136022567749, -0.4918271601200104, -1.3149160146713257, -0.06844182312488556, -0.4087190628051758, -1.987503170967102, -1.6036344766616821, 0.778983473777771, -0.42727741599082947, -1.94147789478302, -0.6759710907936096, 0.27767252922058105, 1.0113354921340942, 0.1820559948682785, -0.2754802703857422, 0.6877537369728088], [0.5079753398895264, -1.7493219375610352, -0.25329238176345825, 0.4541444182395935, -1.0925451517105103, 1.247257947921753, 0.6394129395484924, -1.4155747890472412, -1.2296712398529053, -0.8378282189369202, 0.10278891772031784, -0.12597401440143585, -1.1908432245254517, -0.1382603943347931, -0.19032278656959534, 1.3501598834991455, -1.3356318473815918, 0.14184793829917908, 0.25975117087364197, 0.3085533380508423, -0.5188159942626953, -2.0056545734405518, -0.6972229480743408, -0.7838599681854248, -0.6753873825073242, 0.6128109693527222, 1.430499792098999, -0.9333308935165405, 1.6502571105957031, -2.2339508533477783, -0.23431643843650818, -1.1801930665969849], [0.37905797362327576, -0.660163164138794, 0.46386197209358215, -0.7112237811088562, 1.3849797248840332, 0.015742460265755653, -0.3081227242946625, 0.9861792325973511, 1.493078351020813, 0.08844299614429474, -1.190117597579956, -0.6612745523452759, -0.11130917817354202, -0.7819087505340576, 1.2365397214889526, 0.4827110469341278, -0.8619222044944763, 0.18612989783287048, -0.6418784260749817, 0.24642470479011536, 1.3232852220535278, 1.779866337776184, 1.4530608654022217, 0.36360642313957214, 0.9594776630401611, 0.19137267768383026, 1.4922831058502197, 0.8584865927696228, -0.46095606684684753, 0.7604438066482544, 0.6336575150489807, -1.7976011037826538], [-0.3788876533508301, 1.5214565992355347, -0.3730604350566864, 0.2737553119659424, -0.2079962193965912, -1.3230072259902954, -0.44188499450683594, -2.030813694000244, 0.635991632938385, 1.033216118812561, -0.5954226851463318, -0.03565967455506325, -0.8149994611740112, 0.1845128834247589, -1.6240508556365967, -0.580354630947113, -0.35508644580841064, 0.5928244590759277, 0.8229975700378418, 0.9860242009162903, 0.7936933636665344, -0.4478275179862976, -0.43837404251098633, -1.2879801988601685, 0.2934795320034027, 1.1367883682250977, -1.0038899183273315, -0.2981412410736084, 1.176812767982483, 0.7514091730117798, 2.2378180027008057, -0.9873167276382446], [1.133019208908081, 1.6834580898284912, -0.044886164367198944, 0.11062721163034439, -1.4167813062667847, -0.8607754707336426, -0.032354533672332764, 0.053189925849437714, 1.371086835861206, -0.09532370418310165, 0.8218427300453186, 1.8500488996505737, 0.5977616310119629, -0.5132788419723511, -1.0218844413757324, 0.26063254475593567, 0.867772102355957, -1.2701027393341064, -1.2885794639587402, 1.0730570554733276, -1.2539293766021729, 0.005421164445579052, 0.26089170575141907, -0.9231292605400085, -1.837748408317566, 0.6033684015274048, 0.6560642123222351, -0.8556118011474609, -0.9651846289634705, -0.11344818770885468, -0.05175905302166939, 0.49751460552215576], [0.8268700838088989, 2.461364269256592, -0.7332891821861267, -0.8245304822921753, 1.9579130411148071, -1.6773146390914917, 1.0606513023376465, -0.2037871778011322, -0.29770874977111816, -0.13801692426204681, 0.286554217338562, 0.9647852182388306, 0.9848356246948242, 1.3578338623046875, 0.07716772705316544, -0.5747822523117065, 0.14208060503005981, -1.5799275636672974, -1.4746121168136597, -0.16838987171649933, -0.5009502172470093, -0.4223997890949249, -0.29557004570961, -0.8132187724113464, -0.40407443046569824, -0.9505219459533691, -1.7852280139923096, -0.036973197013139725, 1.5683928728103638, 0.8523329496383667, 0.5684190392494202, -1.899046540260315], [-0.09591901302337646, -0.43378597497940063, 0.07543281465768814, 0.162394717335701, -2.4032318592071533, -0.6259994506835938, -1.370558738708496, -0.894324004650116, -1.0828303098678589, 1.0036929845809937, 0.4235011041164398, 0.603215217590332, -0.5935748219490051, -1.7415053844451904, -0.2828293740749359, 2.361215829849243, 1.3034240007400513, -1.6484755277633667, -0.154318168759346, 0.614891529083252, 1.6949878931045532, -0.16374777257442474, 0.027902044355869293, -0.028281891718506813, 1.6306953430175781, -0.5646453499794006, 0.0014119939878582954, -1.1781052350997925, -0.2778986096382141, -2.0365560054779053, 0.22959427535533905, 0.19951003789901733], [-0.7119303941726685, -0.2780781388282776, -1.3271795511245728, -0.42339760065078735, -0.4157976508140564, -1.2541013956069946, -0.2732144594192505, 2.2965517044067383, -0.8743407726287842, 0.2915937006473541, 0.9303474426269531, 2.2035632133483887, -0.41226303577423096, 0.675582230091095, 0.4589172601699829, 0.037772487848997116, -0.9893873929977417, -1.818110466003418, 1.0843144655227661, 0.299609512090683, -0.8979009389877319, 0.28814610838890076, 0.9672142863273621, 1.1845625638961792, -0.21247686445713043, -0.2930947244167328, -0.5012776851654053, -0.2191275656223297, -1.263127326965332, -0.4421941041946411, -1.2737427949905396, -0.5727797150611877], [0.9462301135063171, 1.085307002067566, -1.683146357536316, -0.4727109372615814, 0.09115441143512726, -0.09503690898418427, -1.3731883764266968, 0.9081155061721802, 1.3734798431396484, -0.6386627554893494, -1.3740234375, 0.5822328329086304, -0.41258925199508667, -0.5294195413589478, -0.8504757881164551, 1.2698955535888672, -0.4325520694255829, 1.1612648963928223, -1.482643961906433, 0.32392618060112, -1.6602904796600342, 1.250085473060608, 0.8498926758766174, -0.8535822629928589, -0.4755423069000244, -0.034264031797647476, 1.1521672010421753, -1.0183829069137573, 0.8770524859428406, -0.10525510460138321, 0.24428759515285492, 1.3321714401245117], [-0.811094343662262, 0.5011335015296936, -0.613280713558197, -0.49016985297203064, -0.7038731575012207, 0.3752433657646179, 1.2488272190093994, 0.05322497338056564, 1.6230428218841553, 0.20305415987968445, 2.0881385803222656, 1.4645376205444336, -0.2368103414773941, -0.5052717924118042, -0.3852819800376892, 0.01515308115631342, -1.0311754941940308, 0.8443540334701538, 1.0429959297180176, 2.0842554569244385, 0.2516816258430481, 1.0474543571472168, 0.42192548513412476, -0.9181886911392212, -0.8870816230773926, 0.23503850400447845, -0.9512053728103638, 1.2566584348678589, -0.26306143403053284, -1.490297555923462, -0.3376157879829407, -0.6967437267303467], [-0.574139416217804, 2.453662395477295, -1.6077771186828613, -0.8984474539756775, 0.10766775906085968, 0.7880519032478333, -0.589568555355072, 0.07169578224420547, 1.2987111806869507, 1.3205829858779907, -0.32315531373023987, 0.514397919178009, 1.7062896490097046, -0.07899729162454605, 0.09954572468996048, -2.8325796127319336, 0.3441140949726105, -0.37699833512306213, -1.3225268125534058, -0.6298416256904602, 1.4256187677383423, 1.1704010963439941, 0.18383553624153137, -1.8000601530075073, 1.6303467750549316, -0.7630939483642578, -1.69486665725708, -1.5312435626983643, 0.5588215589523315, -2.9799482822418213, -0.5003279447555542, -1.1834627389907837], [-0.3645801246166229, 0.33838120102882385, 0.15895980596542358, -0.10757678747177124, -0.48366186022758484, 2.030900001525879, -1.9642555713653564, -1.615454077720642, -0.32527419924736023, 2.034595489501953, 0.12000928074121475, -0.3887355327606201, -0.4434056282043457, 0.5202061533927917, 0.7296760082244873, 0.5053451061248779, -0.4284498393535614, 0.07680489867925644, -1.102176547050476, 0.3117895722389221, 1.3504058122634888, -0.582911491394043, 0.7754387259483337, -0.06172740459442139, -0.5823753476142883, 0.5596877336502075, -0.4170951247215271, -0.5529710650444031, 0.9865680932998657, 1.2973499298095703, 0.9245200157165527, 0.4179720878601074], [-0.4130266308784485, -1.0361276865005493, 0.17694498598575592, -0.8555528521537781, 1.2472487688064575, -0.48681676387786865, 0.4710976481437683, -1.1264172792434692, 0.7103182077407837, 1.838128685951233, -0.07716411352157593, -0.7753161787986755, -0.6910890340805054, -1.74885094165802, 0.3942961096763611, -0.6544696688652039, -0.5391520261764526, -0.4227515161037445, -1.7788394689559937, 0.9429383277893066, -0.8492404818534851, 0.5681067109107971, 1.5467877388000488, -0.9569497108459473, 1.8348749876022339, -0.6086304187774658, -1.7514095306396484, 0.6511777639389038, 0.45277297496795654, 0.5940176248550415, -0.12861716747283936, -1.6742360591888428], [-1.7879815101623535, -1.1563079357147217, -0.3280155658721924, -0.3772120177745819, 1.3256173133850098, 2.0431816577911377, -1.4503259658813477, 0.8245552778244019, 1.311181902885437, 0.3937910497188568, -1.0649827718734741, -0.991484522819519, 1.5015287399291992, -0.2730044722557068, 1.6438746452331543, -0.8035056591033936, -0.5085627436637878, -0.26038461923599243, -1.082297682762146, 0.7159017324447632, 0.24578936398029327, 0.7723374366760254, -1.0160164833068848, 0.430162638425827, 0.6691747307777405, 0.07886946201324463, -0.6576972007751465, -1.0715487003326416, -0.6806802749633789, 0.869575023651123, 0.09052404761314392, 0.007049884181469679], [-0.9503561854362488, 1.0885087251663208, 1.8536516427993774, 0.07245831936597824, 1.5244449377059937, 0.07268576323986053, -2.3433678150177, -0.419734925031662, -0.4702504873275757, 0.02740592136979103, -0.8586763739585876, 0.6584892868995667, -0.8180452585220337, -0.0014511710032820702, -0.7627133131027222, -1.781476378440857, 0.11810331046581268, 0.11432746052742004, -0.45974722504615784, -0.7186701893806458, -0.4797029495239258, -0.14243271946907043, 0.08807900547981262, -0.9733503460884094, -0.18618565797805786, -0.49324992299079895, 0.8388981819152832, 0.5434315204620361, -0.6307855844497681, 0.3668288290500641, 1.6540300846099854, 0.06942703574895859], [2.995140552520752, 0.8145565986633301, 1.464847445487976, 0.49066823720932007, -0.07606004178524017, 1.4202181100845337, 2.077808380126953, -1.2764129638671875, 0.2671757936477661, 2.9846420288085938, 1.1372393369674683, 0.9312382936477661, 0.6303080320358276, -1.9542301893234253, -1.3140405416488647, -1.5095731019973755, -1.1686644554138184, -0.26983842253685, 1.0658576488494873, -0.6126819849014282, 2.2223284244537354, -1.0164809226989746, -0.12769129872322083, -1.4695544242858887, 0.4009208083152771, 1.4912009239196777, -0.02771146409213543, -0.21162718534469604, -0.5083771347999573, 0.7123346328735352, 0.249565988779068, -1.9998923540115356], [-0.20185798406600952, 0.8363472819328308, -1.22691810131073, 0.6018937230110168, -0.6629507541656494, 0.34879130125045776, -1.3202052116394043, 0.21209599077701569, 0.8541021943092346, 0.12972483038902283, 1.1031063795089722, 0.618743360042572, 0.9125607013702393, -0.7335010766983032, 1.2325661182403564, 0.49815478920936584, 1.5279426574707031, 0.2798710763454437, -0.35328805446624756, 1.5634573698043823, -0.08679318428039551, 0.34974798560142517, -1.0248990058898926, -1.6556594371795654, -0.196110337972641, 0.10355746001005173, 0.4297157824039459, -0.7898093461990356, 0.5530619025230408, -0.2516523003578186, -0.7118558883666992, 0.9022749066352844], [0.9233792424201965, 0.33648422360420227, -0.20306511223316193, -0.5899150967597961, -1.583599328994751, -0.6049516201019287, -0.03537364676594734, -0.17256233096122742, 0.2965044379234314, 0.3859955966472626, 0.09049787372350693, -0.6769859194755554, 0.21747373044490814, -0.7332823872566223, -0.8553593754768372, -0.9323434233665466, 2.123802900314331, 1.3398970365524292, 1.1575672626495361, 0.3210848867893219, -1.3968446254730225, 0.4519282579421997, -1.1192727088928223, -0.5430407524108887, -1.2183036804199219, -0.27334704995155334, -0.9347283840179443, 0.4212113916873932, -0.9288032650947571, -0.997783899307251, -0.31717970967292786, 0.7484700679779053], [0.1465970277786255, -1.134798526763916, -0.7532748579978943, -1.4143190383911133, -0.3687244653701782, -0.820858895778656, 0.7183498740196228, 1.6764675378799438, -0.4372166395187378, 0.6599562764167786, 0.2738111913204193, 1.6833562850952148, 1.0064787864685059, -1.9321268796920776, 0.9957238435745239, 0.5425209999084473, -0.5943399667739868, -1.3361544609069824, 0.9168185591697693, -1.2119970321655273, -1.3497661352157593, 1.0783790349960327, -1.7181851863861084, -1.5669139623641968, -0.13209757208824158, 2.3068313598632812, 0.40697237849235535, 1.7335519790649414, 0.19551976025104523, 0.18198993802070618, 0.6235752701759338, -0.41999557614326477], [0.5639786124229431, -0.6317747831344604, -0.9650527238845825, 0.12898707389831543, 0.6812499761581421, -2.4287426471710205, 0.542050302028656, -0.7765429615974426, -0.385623961687088, -0.38838812708854675, -0.2853124737739563, 1.240949273109436, -0.897977888584137, 1.3119548559188843, 2.2174479961395264, -1.4262927770614624, -2.067652702331543, -0.6738793253898621, 1.941743016242981, 1.2181333303451538, -0.9997787475585938, -0.3255690336227417, 0.8618587255477905, -0.4456687569618225, -0.5577219724655151, -1.4984608888626099, -0.8894231915473938, 0.9511304497718811, 0.2244097888469696, -1.0703638792037964, -0.9842942953109741, -0.4349289536476135], [1.4488193988800049, 0.7870330810546875, 0.7591296434402466, -0.6048772931098938, -0.8261520266532898, 0.07844094187021255, -0.8073725700378418, 1.1947898864746094, -0.9299328923225403, -1.0594513416290283, 0.5018804669380188, -0.48217418789863586, -0.25836801528930664, -1.3088996410369873, -1.2242721319198608, -0.048707615584135056, 0.40257197618484497, 0.8173394799232483, -2.0070266723632812, -0.0684848204255104, -0.38354355096817017, -0.5691659450531006, 0.5065147876739502, -1.145277976989746, 0.4728286862373352, 1.6790982484817505, -1.6353009939193726, -0.8277897238731384, 2.007723569869995, 0.7653829455375671, -0.12401386350393295, -0.025714056566357613], [1.2145936489105225, 1.2409658432006836, -0.6884391903877258, 1.2228871583938599, 1.3753716945648193, 1.8216872215270996, -1.680288553237915, -0.22500774264335632, -0.4024282395839691, 1.4582393169403076, 1.3302316665649414, -0.3402674198150635, -1.2278810739517212, 0.034463975578546524, 0.3107241094112396, -0.41614362597465515, -1.715554118156433, -0.6348498463630676, 1.5572582483291626, -1.1506277322769165, 0.8922731876373291, -0.27471500635147095, 0.9200380444526672, 0.7919225692749023, 0.23774461448192596, -1.394852876663208, -0.16507339477539062, 0.6528167128562927, 0.49702194333076477, -0.4381999969482422, -0.3659195005893707, 0.41186708211898804], [-0.1458175778388977, 0.7017612457275391, 0.2261463701725006, 2.354152202606201, 0.2659119963645935, 0.6189876198768616, -0.5240734815597534, 0.547961950302124, -0.14491891860961914, 0.7901250720024109, -0.7060232162475586, 0.011354311369359493, 0.11248990893363953, 0.7524216771125793, -1.2057383060455322, -0.08140327036380768, 0.8642041683197021, 0.5104483366012573, 0.12560950219631195, -0.4076598882675171, -0.20193447172641754, -0.21161402761936188, 1.743519902229309, -2.2031314373016357, -0.2406628131866455, -1.9442708492279053, -0.22676800191402435, 1.3309457302093506, -0.05540330335497856, -0.29973602294921875, 0.6751695275306702, -1.3693106174468994], [-1.1679574251174927, 0.805162250995636, 1.6046943664550781, 0.916613757610321, -0.8280498385429382, -0.8856745958328247, -1.1736441850662231, -1.2571390867233276, 0.8305137753486633, -0.6377527117729187, 0.5140644311904907, -0.26982250809669495, 2.1500232219696045, 1.8047994375228882, 0.7882305979728699, 0.7117539048194885, 1.5935851335525513, -0.9898024201393127, -0.7493394017219543, 0.7621563076972961, -0.18410198390483856, 0.734298050403595, 0.9903640151023865, 1.1261292695999146, -0.5853630900382996, -0.4449270963668823, -0.40401384234428406, 0.08560090512037277, -0.9100096821784973, -0.3560880422592163, 0.38860470056533813, 0.6907169818878174], [-0.8850512504577637, 0.6169518828392029, -1.3119279146194458, 1.0038334131240845, -1.5372366905212402, 0.0434306338429451, 0.08858737349510193, 0.09927191585302353, -0.37195736169815063, -1.007068395614624, 0.4871573746204376, 0.9645817279815674, 1.376544713973999, -0.532132625579834, -0.5804851651191711, 0.7782320976257324, -0.23861946165561676, -1.7819604873657227, -1.336653470993042, -0.975035548210144, -1.231650471687317, -1.3193222284317017, -1.4252479076385498, -0.12543806433677673, 0.8982192873954773, -0.9162585735321045, -0.5946196913719177, -1.0318630933761597, 0.9312285780906677, -0.19953471422195435, -0.8717377781867981, 0.1924654096364975]], "dtype": "float32"}], "e": {"__nd__": [[1.3729920387268066, 1.6154651641845703, -9.451895713806152, 3.220261812210083, -3.288147211074829, -3.9595935344696045, -2.8942694664001465, -12.863110542297363, -3.774618148803711, 6.313938617706299, -2.1996564865112305, 0.7568126916885376, 0.2606208622455597, -5.3106584548950195, 2.6133310794830322, -2.5797202587127686, 9.191701889038086, 3.5626988410949707, -2.342608690261841, -2.4388229846954346, 4.9753875732421875, -5.440825939178467, -4.843916893005371, -10.729670524597168, -1.7341989278793335, -3.7390642166137695, -21.5269775390625, -7.239187240600586, 15.006373405456543, -1.0902540683746338, 2.259563684463501, 2.6637024879455566], [3.838122844696045, 1.344510555267334, 2.1487741470336914, -11.872757911682129, -4.333494186401367, 3.077760934829712, -1.1760704517364502, 7.75171422958374, 2.3208675384521484, -1.300356149673462, 8.7556734085083, -1.3949307203292847, -0.9426664113998413, -4.344322681427002, 5.866335868835449, 6.806878566741943, -2.6901514530181885, -1.6341826915740967, 0.5638644695281982, 1.8081424236297607, 6.402277946472168, 12.353029251098633, 2.2002835273742676, 5.448212623596191, -7.439860820770264, 12.461774826049805, -0.28409063816070557, -2.791785955429077, -1.1798796653747559, -1.5127627849578857, -1.6540367603302002, -3.3168575763702393], [0.4862884283065796, -4.569661617279053, -1.7728888988494873, -11.314692497253418, 3.0623152256011963, -2.7784500122070312, -4.729100704193115, 3.6446473598480225, 6.019253730773926, -7.190601348876953, -5.445973873138428, 0.11501020193099976, -5.98023796081543, -7.822381973266602, 4.3104753494262695, 5.676305294036865, -0.051758140325546265, 4.507699012756348, 1.8586368560791016, 0.19345295429229736, -4.104775428771973, 10.969796180725098, -5.4570207595825195, -6.214121341705322, 4.959053039550781, 13.898399353027344, 9.220382690429688, 16.78662109375, 2.366690158843994, 8.382412910461426, 13.539819717407227, -0.3533882796764374], [-2.2291641235351562, -3.662946939468384, -12.664704322814941, -4.280389308929443, 2.6757102012634277, -1.8948315382003784, -2.488724946975708, 8.452376365661621, -3.0794196128845215, 2.8643970489501953, -5.6508097648620605, 6.138632774353027, 5.758545875549316, 13.080255508422852, 11.36804485321045, 0.07480019330978394, 7.037886142730713, -5.144717216491699, 6.351142406463623, -1.195777416229248, -1.9489461183547974, 4.369171142578125, -10.621227264404297, 2.271028995513916, 7.089829921722412, 0.4234752655029297, -9.018265724182129, -8.268598556518555, 12.659672737121582, 0.10553610324859619, -10.336907386779785, 8.328299522399902], [-5.152736663818359, -8.019922256469727, -10.590974807739258, -2.2972114086151123, 1.2224035263061523, -9.545538902282715, 0.8527328968048096, 5.595766544342041, -4.372236728668213, 0.5021027326583862, -5.294164657592773, 10.971871376037598, 13.454734802246094, 9.841168403625488, 12.013297080993652, -2.515864610671997, -0.9953635334968567, -9.660476684570312, -4.515662670135498, 8.644573211669922, -17.315153121948242, 7.461663722991943, 3.3564515113830566, -4.911952018737793, -0.8306992053985596, -3.9019339084625244, -13.596195220947266, -1.2274048328399658, 0.6243809461593628, 2.4943037033081055, -6.279669284820557, 3.6121251583099365], [2.6958389282226562, -8.982917785644531, 4.656001091003418, -1.1274874210357666, 0.06754732131958008, 6.974981307983398, -3.864203929901123, 1.7750977277755737, -5.420480251312256, 2.667740821838379, -0.25214502215385437, -9.408010482788086, -2.837740898132324, 0.5067254900932312, -11.589216232299805, 3.199331521987915, 8.342581748962402, 1.702556848526001, -6.984647750854492, -6.602828025817871, -3.8416199684143066, -6.990777969360352, -9.02879810333252, 4.6547040939331055, 4.301819801330566, -3.0328497886657715, 1.487733006477356, -0.5108419060707092, 4.436592102050781, 3.043769121170044, -0.39276832342147827, 0.12535783648490906], [2.4246413707733154, -1.9117143154144287, -7.287981033325195, -2.6796622276306152, -0.4517710208892822, -5.444186210632324, -7.248151779174805, -1.8314695358276367, -5.7215681076049805, -6.441849708557129, 10.75320053100586, 8.428570747375488, -2.584315776824951, 12.201321601867676, -8.712333679199219, 4.999696254730225, 10.269259452819824, 5.176114559173584, 11.064647674560547, 18.93838119506836, -0.12308961153030396, -1.6015875339508057, 0.37940776348114014, 4.440272331237793, 4.365408420562744, -9.532331466674805, 4.545653343200684, -2.193260431289673, -1.0637980699539185, -3.733233690261841, 3.520287036895752, 4.166600704193115], [1.1785061359405518, 0.365508496761322, -4.797932147979736, -0.6369680166244507, -13.960954666137695, -14.021307945251465, -5.329980373382568, -8.723662376403809, -0.36674419045448303, 12.014330863952637, -8.571573257446289, 11.453754425048828, -6.323654651641846, 1.0529567003250122, 1.00522780418396, 0.23950329422950745, 5.842083930969238, -7.7848076820373535, 1.4392220973968506, 5.4446611404418945, -11.306073188781738, 8.085850715637207, -1.5815398693084717, -0.7512526512145996, -0.2893241345882416, 11.475240707397461, 3.727815628051758, -5.728372097015381, 2.8354034423828125, -6.384150981903076, 3.69101619720459, 11.943472862243652], [-6.334447383880615, 3.347076177597046, -11.535802841186523, -10.985712051391602, -0.8890998363494873, 6.367185592651367, -16.533777236938477, -1.835373044013977, 1.4464607238769531, -10.60364818572998, 5.812016010284424, -3.550414562225342, 5.326834678649902, -6.586811065673828, -9.599432945251465, 10.558208465576172, 2.8987364768981934, 7.503979206085205, -0.9180266857147217, 4.277610778808594, -3.2108397483825684, -4.823308944702148, -5.380857467651367, -1.991192102432251, 2.3268942832946777, 4.761682510375977, -15.52611255645752, -5.845276832580566, 23.95459747314453, 5.435800075531006, -8.380851745605469, 11.293004989624023], [2.12014102935791, 5.159423351287842, 4.193903923034668, -5.979212760925293, -2.1326711177825928, -8.260778427124023, -1.7142139673233032, 7.097752094268799, -2.231592893600464, 0.4975171983242035, -5.615293979644775, 0.9005953669548035, -2.5331413745880127, 21.275175094604492, -6.112242221832275, 6.108676910400391, 9.10515022277832, 1.7731423377990723, -14.550995826721191, -17.550609588623047, -11.11027717590332, -1.484993815422058, -11.140186309814453, 5.057567119598389, -6.1187872886657715, -7.520079612731934, 1.1153260469436646, -5.156325340270996, 0.5565704703330994, 7.213200569152832, 8.707037925720215, 9.975706100463867], [3.2285287380218506, -0.30420929193496704, -5.682734489440918, -13.43429183959961, -3.1754097938537598, 3.3589491844177246, -11.694568634033203, 1.633508563041687, 5.561232089996338, 10.720090866088867, 9.748261451721191, 7.905902862548828, 3.268441677093506, -8.280396461486816, 10.054040908813477, 4.479120254516602, 17.527511596679688, -10.146904945373535, 5.035525321960449, 15.466936111450195, 2.084653377532959, 17.32971954345703, 2.9289770126342773, -1.1207444667816162, -0.34804725646972656, 12.812488555908203, -0.4721187353134155, -4.968513488769531, -7.452808856964111, -9.927192687988281, -7.917180061340332, 17.234378814697266], [0.9768046736717224, -2.5534987449645996, 4.393002986907959, -1.102594017982483, -9.02170467376709, -4.706551551818848, 5.074296951293945, -4.366991996765137, 12.904552459716797, 0.0020113587379455566, -6.532751560211182, -5.417893886566162, 0.5047324299812317, -11.77762222290039, 4.475037574768066, -4.114224433898926, -2.3240859508514404, 0.8660385608673096, -1.399850606918335, 3.740401029586792, 3.509918212890625, 5.285468578338623, 4.289449214935303, -8.323478698730469, 11.146353721618652, 9.122126579284668, -9.246355056762695, -3.6788880825042725, 6.103348255157471, 5.260679244995117, 1.2572611570358276, -3.113384485244751], [-5.384993553161621, -0.8680443167686462, -1.951092004776001, -6.18342399597168, 3.1787686347961426, -5.565507411956787, 5.306196212768555, 5.37841796875, 3.3798370361328125, 2.44230580329895, -2.456483840942383, 1.1733601093292236, -3.680765151977539, 1.46673583984375, 6.36387300491333, -3.448291301727295, -3.244476795196533, -7.659916400909424, 3.1560192108154297, -0.7564431428909302, -7.639026641845703, 10.75960636138916, 0.9238412380218506, 2.7500808238983154, -3.7760958671569824, 7.446905612945557, 5.173255920410156, 6.559334754943848, -11.565361976623535, -3.6113948822021484, -2.536675214767456, 4.072055339813232], [-0.06673890352249146, 5.088563442230225, 9.235536575317383, -2.9559292793273926, 1.1292186975479126, 5.721792221069336, 9.831053733825684, -0.7469087839126587, 1.4523810148239136, 10.053241729736328, -4.225503921508789, -3.1666314601898193, 10.314287185668945, 5.178078651428223, -10.937773704528809, -6.1702985763549805, -9.139381408691406, -5.634353160858154, -1.7149872779846191, 2.3494973182678223, 8.137980461120605, -6.743547439575195, 4.904420375823975, 3.986999034881592, 3.145416498184204, -2.918933868408203, -0.3515937030315399, -1.9658682346343994, 0.14651012420654297, 6.680732250213623, 3.0604233741760254, -14.903091430664062], [-2.464763641357422, -4.296154022216797, -9.497154235839844, -3.1283652782440186, -0.1560584008693695, -1.757395625114441, 9.244778633117676, -5.333160400390625, -1.3079190254211426, -0.14619481563568115, -3.625931739807129, 2.605466842651367, 9.159991264343262, -11.258462905883789, 7.009063243865967, -4.5207624435424805, 3.2407610416412354, 3.101832389831543, 10.196084976196289, 5.876884937286377, 2.930734395980835, 0.6583893895149231, 8.052157402038574, 12.186288833618164, 9.166342735290527, 4.246683597564697, -3.390690803527832, 8.416691780090332, -5.552569389343262, 2.7011756896972656, -4.510026454925537, -7.209810256958008], [16.66094207763672, -1.2532660961151123, -1.0034959316253662, -6.940519332885742, -6.647953510284424, -5.826786518096924, 15.415112495422363, -3.60318660736084, -2.236985206604004, 4.575824737548828, 0.5143405795097351, 5.38878870010376, -12.420342445373535, -11.941384315490723, -17.303327560424805, -3.610247850418091, -5.302571773529053, -4.3731913566589355, 13.992321014404297, -6.399127006530762, 1.5623525381088257, -1.3897311687469482, -9.586966514587402, 6.5246477127075195, -3.919724225997925, 9.892005920410156, 13.706693649291992, -11.526060104370117, -0.8358535766601562, -13.318696022033691, -6.971158027648926, 0.36114564538002014], [-6.002755641937256, -5.580550193786621, 9.444268226623535, 1.221555471420288, -1.5916777849197388, -7.768532752990723, -15.331002235412598, 1.8950586318969727, -0.4701993763446808, -2.8840956687927246, -2.881157398223877, -3.1394128799438477, -0.37296128273010254, 17.850536346435547, 7.600022792816162, -4.989346981048584, -6.765539169311523, -9.666132926940918, 0.3274550437927246, 7.247684955596924, -7.877704620361328, 8.757508277893066, 9.459729194641113, 5.738694667816162, -4.73692512512207, -9.406204223632812, -4.6474103927612305, -4.401219367980957, 2.254579782485962, 11.345699310302734, 8.104398727416992, 8.938232421875], [-3.259371280670166, 9.606539726257324, 2.2059154510498047, -8.809951782226562, 4.8147664070129395, -6.014261245727539, 1.3946539163589478, 6.731227874755859, 13.98439884185791, 9.908056259155273, 10.242655754089355, -1.082054853439331, 4.41995906829834, -8.48991584777832, -3.3502609729766846, 13.529534339904785, 0.5278137922286987, -0.6941637992858887, -14.418834686279297, 5.148254871368408, 3.505131721496582, 7.254561424255371, 7.04209041595459, -10.598345756530762, 2.9986958503723145, -0.8281108140945435, 4.4967265129089355, 6.0785746574401855, 1.5438624620437622, 4.142561435699463, 1.0475060939788818, -21.108369827270508], [-6.7139177322387695, -4.098071575164795, -4.576226234436035, -11.906510353088379, 6.462192058563232, -0.31147730350494385, 2.8343522548675537, 6.6328125, 5.981350421905518, 12.269848823547363, 1.1045680046081543, 9.846728324890137, -4.774991512298584, 5.529605388641357, 19.352828979492188, -7.7340593338012695, -13.317018508911133, -10.009117126464844, 0.6202510595321655, -2.6436142921447754, -5.628143787384033, 12.614615440368652, 4.532942771911621, 0.9618125557899475, 10.77104377746582, 7.0433478355407715, -4.987269401550293, 2.837176561355591, 12.739099502563477, -1.4581093788146973, -2.9202163219451904, -13.634507179260254], [-2.7637619972229004, 3.7304866313934326, 7.034074306488037, -7.389901638031006, 6.676314353942871, 3.9627997875213623, -4.4914164543151855, -18.72646141052246, 3.3980026245117188, 2.6849308013916016, -6.863729953765869, -8.229050636291504, 4.419187545776367, 3.6797845363616943, 6.2981672286987305, -6.649049758911133, -4.500523567199707, -1.2437288761138916, 3.8560309410095215, 0.7559259533882141, 2.875154972076416, 3.2813351154327393, 9.935517311096191, -1.2437729835510254, 1.8956502676010132, 3.154493570327759, -3.5204505920410156, 1.5512477159500122, -8.529983520507812, -7.314517021179199, 5.674163341522217, 4.483145713806152], [7.222628593444824, 2.116554021835327, -5.939969062805176, -1.487299919128418, 6.045105457305908, 2.4891293048858643, 6.167999744415283, 3.996028184890747, 10.02942180633545, -2.2605762481689453, -1.993276834487915, 3.739567756652832, -0.622287392616272, -3.373727321624756, -0.87067711353302, -10.213264465332031, -15.79635238647461, 6.2764081954956055, 11.153791427612305, 10.557759284973145, 1.6176950931549072, 7.482461452484131, 11.567742347717285, -6.0791096687316895, 0.7500192523002625, 1.5787513256072998, -0.6135784387588501, -2.0910563468933105, -4.185707092285156, -1.4172219038009644, -2.925633192062378, -0.497006893157959], [8.80463695526123, 10.415105819702148, 3.1279993057250977, -14.19810962677002, -6.455372333526611, -0.2565145492553711, 9.59107780456543, -11.992900848388672, 8.518256187438965, 12.486621856689453, 8.310952186584473, 3.19655442237854, 1.7511134147644043, -16.728557586669922, -15.507396697998047, -11.105621337890625, 3.0761971473693848, -1.6689046621322632, -2.8708598613739014, 2.0310869216918945, 18.04047966003418, 5.162886619567871, -7.813582897186279, 10.61189079284668, 7.8392839431762695, 8.818479537963867, -0.41675400733947754, -2.3325397968292236, -1.7364991903305054, -4.230137348175049, -8.125996589660645, -4.257883071899414], [-1.1243480443954468, -3.351612091064453, 7.983212471008301, 4.259699821472168, -14.464424133300781, -1.7785724401474, 1.5173801183700562, -0.5784435868263245, -2.6989376544952393, -3.1480441093444824, -7.408060073852539, -6.775907039642334, 10.154895782470703, -3.789351463317871, 5.724817276000977, 4.4799580574035645, 9.088191032409668, 4.162656784057617, -8.558233261108398, 5.164530277252197, -0.13130034506320953, -5.174697399139404, 0.5372010469436646, -3.157773494720459, 6.795772552490234, 4.390219211578369, -0.7617310881614685, -0.5955504179000854, 4.914212703704834, 7.863499164581299, -2.871229410171509, 11.456104278564453], [15.427748680114746, -6.564844608306885, 9.169859886169434, 0.41451841592788696, 8.399032592773438, -11.46434211730957, 12.826887130737305, 4.271346092224121, -0.8374323844909668, 4.543412685394287, 1.8914812803268433, -3.0183207988739014, -5.286131858825684, 7.537195205688477, 6.718954086303711, -5.6335039138793945, -8.120076179504395, 2.009748935699463, -3.2252116203308105, -3.734388828277588, -10.152749061584473, -2.6422643661499023, -0.35818421840667725, 3.4472899436950684, 3.058197498321533, -5.832115173339844, 1.2083367109298706, 2.617917537689209, 3.0209310054779053, 10.287338256835938, 3.315607786178589, -13.775288581848145], [-8.1322603225708, 4.7674241065979, 7.883249282836914, 3.7910444736480713, -3.6410348415374756, 8.529640197753906, 4.419973373413086, 6.37109899520874, 12.522233009338379, 11.856430053710938, -2.5533595085144043, -4.403904914855957, 4.898052215576172, 0.10809747129678726, 8.155442237854004, -6.262331008911133, -0.5212591290473938, -10.910420417785645, -3.9991402626037598, -1.3137630224227905, 18.02163314819336, 7.577303409576416, -2.399108648300171, 6.254217624664307, 0.05580247566103935, 2.89200758934021, 5.395984649658203, -6.730558395385742, -4.29958963394165, -1.2902302742004395, -3.2550930976867676, 1.9821898937225342], [6.473641395568848, -1.461623191833496, 2.204486846923828, -2.0284786224365234, 0.964105486869812, 5.39016580581665, 7.7608771324157715, 1.550417184829712, -2.7224490642547607, -8.345146179199219, -11.35078239440918, 7.005248546600342, 5.593403339385986, -0.25304511189460754, 0.7362316846847534, 2.119309663772583, -3.480515241622925, 3.353846788406372, -7.500093936920166, 0.8840775489807129, -11.868255615234375, -3.979548692703247, 4.928778171539307, -8.31054973602295, -2.064181089401245, 18.885793685913086, 2.2088074684143066, 0.9642930626869202, 3.4100677967071533, 8.005770683288574, -0.08342933654785156, 4.682765483856201], [8.695955276489258, -3.9755759239196777, 3.6746792793273926, -4.657937526702881, 1.9495588541030884, 15.332734107971191, 2.8933444023132324, -3.929375648498535, 3.2595300674438477, 5.5772576332092285, 2.429542064666748, 0.7425051927566528, -5.373140811920166, -13.628594398498535, 1.7537835836410522, 1.263545036315918, 12.174139976501465, 6.168421268463135, -9.11880111694336, -1.7877776622772217, 6.521416664123535, 0.6593623161315918, -3.37398624420166, 4.544672012329102, 4.885587215423584, 11.966259956359863, 2.068704128265381, -7.995305061340332, 6.077174186706543, -8.777470588684082, -12.502037048339844, -1.8948917388916016], [7.464483261108398, -5.609672546386719, -0.6874896883964539, 9.442546844482422, -13.275006294250488, -1.085737705230713, -5.7177958488464355, 4.753922462463379, 7.348148345947266, 4.815604209899902, 6.920891284942627, 3.3527309894561768, -6.949192523956299, 0.4327634572982788, -0.6421024799346924, -2.8452000617980957, 17.337017059326172, 7.6945414543151855, -2.142502784729004, 8.976156234741211, -9.89194393157959, 6.34724760055542, -7.561763763427734, 5.920833587646484, -4.223006725311279, -6.831121921539307, 5.708293914794922, -9.666858673095703, -2.7367238998413086, 1.7514086961746216, 1.565528154373169, 8.28450870513916], [3.097775459289551, -6.672283172607422, -3.336543560028076, 2.960092306137085, -10.095304489135742, -1.6995418071746826, 16.56561279296875, -5.696377277374268, -5.043837070465088, -6.7778754234313965, -1.3994262218475342, -1.2807221412658691, 1.6706537008285522, -1.3534467220306396, 1.582900881767273, 8.796843528747559, -11.866158485412598, -9.288098335266113, 3.883754014968872, 5.759500980377197, 1.3602721691131592, -14.547026634216309, -5.833191871643066, -2.9774365425109863, -0.5851262807846069, 4.277613162994385, 7.4494781494140625, -1.201744794845581, -1.0111134052276611, -11.178048133850098, -1.18051016330719, -1.9124467372894287], [1.4015650749206543, 0.9362627863883972, -1.1843397617340088, -0.45484888553619385, 1.426730751991272, 4.732324123382568, 4.474573135375977, 6.730011940002441, 0.45215678215026855, -1.4259214401245117, -3.2719242572784424, -0.06712901592254639, 3.015000820159912, -2.429765224456787, 6.045734405517578, -0.2749277353286743, -0.44357746839523315, -6.052596092224121, -7.1580281257629395, -8.877635955810547, 2.5974879264831543, 0.5928962230682373, -4.169122219085693, 10.006328582763672, 4.9077653884887695, -0.03851020336151123, 8.806268692016602, -5.161072254180908, 8.07746696472168, 6.220714092254639, -2.493807077407837, -9.775426864624023], [0.5000194907188416, -0.599738359451294, -14.450984001159668, -3.036566734313965, 6.960690021514893, 1.1735783815383911, 6.4671759605407715, -1.5683587789535522, 2.483966112136841, -10.687828063964844, 4.848343372344971, 7.437618732452393, 8.625733375549316, -7.7105512619018555, -3.673513174057007, -1.9288830757141113, 2.5627505779266357, 2.77738881111145, 15.995576858520508, 12.099740982055664, -2.6867167949676514, 4.561256408691406, 3.5581493377685547, 7.355936527252197, -5.763773441314697, 4.933701992034912, 4.572941303253174, 1.813138723373413, -14.008601188659668, -8.696626663208008, -10.606945037841797, 1.6234045028686523], [-4.040683746337891, 5.539701461791992, -2.027218818664551, -0.9299781322479248, -4.004001617431641, -1.13557767868042, -8.308207511901855, -9.473039627075195, 7.107470989227295, -0.02405226230621338, -5.819671630859375, -1.9347174167633057, 2.7038283348083496, -11.00005054473877, -10.029300689697266, 6.547238826751709, 5.349309921264648, -8.877546310424805, -4.261207103729248, -3.2827329635620117, -2.815502166748047, 1.5750353336334229, -4.046464920043945, -6.890470504760742, 2.0328409671783447, 0.12649697065353394, 3.555968999862671, -3.2611923217773438, 4.887711048126221, -11.937644004821777, 1.6027754545211792, -6.966263771057129], [-2.62677001953125, 7.346341609954834, 2.6085727214813232, -6.127164840698242, -5.643154621124268, 4.382394790649414, 6.679173946380615, 6.314521789550781, 1.6024155616760254, -6.84467887878418, -1.6434749364852905, 2.466320753097534, 6.579168796539307, 3.660919189453125, -7.685108184814453, 5.538288116455078, 4.172174453735352, -6.181722640991211, -8.541035652160645, -0.3964921236038208, 3.659219264984131, -6.935981273651123, 2.789760112762451, -10.089929580688477, 4.6881184577941895, -5.750804424285889, -5.4619951248168945, 6.285891056060791, 6.879819869995117, -14.190227508544922, -2.1866424083709717, -6.537668704986572], [3.0094480514526367, 3.6227495670318604, 4.52703332901001, -0.6440558433532715, 6.585850238800049, 3.2848422527313232, 4.8362202644348145, 0.6313989162445068, 2.411790370941162, -5.889632225036621, 2.825744867324829, 2.8497531414031982, 5.55839729309082, -3.038405179977417, -3.0102224349975586, 2.530163288116455, -8.973343849182129, -0.47020983695983887, 0.14703369140625, -0.4746532440185547, -2.3505663871765137, -5.210977554321289, 8.631482124328613, -0.07964763045310974, -9.81039810180664, 1.5805940628051758, 2.8070321083068848, -2.663421154022217, 1.1036951541900635, 16.160757064819336, 1.0372294187545776, -4.1297221183776855], [-9.866548538208008, -10.025341987609863, 2.289098024368286, -0.9735544919967651, -0.7920907735824585, -1.138326644897461, -3.3363823890686035, 5.819390296936035, -3.9674339294433594, -0.36524975299835205, -1.8232269287109375, -3.5286896228790283, 0.5602768659591675, 4.022363185882568, 9.119309425354004, 2.96775484085083, -2.8791117668151855, -4.299766540527344, -3.309375286102295, 1.3308148384094238, -2.2446718215942383, 4.891953468322754, -1.7543905973434448, -2.0807883739471436, 1.559598684310913, 2.8219847679138184, 2.218304395675659, -7.02096700668335, -6.8836259841918945, -6.3067803382873535, -4.095569133758545, -1.4842119216918945], [1.1381428241729736, 5.330000877380371, 6.172792434692383, 1.6881299018859863, -6.0946526527404785, -4.578777313232422, 2.898200511932373, 0.4357871413230896, 6.824563026428223, -6.4251203536987305, 1.5719776153564453, 1.1926405429840088, -4.74517297744751, 1.3396589756011963, 0.776238203048706, 3.1151046752929688, -8.185921669006348, -5.538957118988037, -2.2934041023254395, 4.1845245361328125, 0.4664401412010193, -0.3985684812068939, 6.424185276031494, 5.972507953643799, -4.064694881439209, -1.099806547164917, -1.3930509090423584, -5.175865650177002, 3.482072114944458, 10.31634521484375, 9.228874206542969, 2.657792806625366], [1.7815182209014893, 3.7481346130371094, 1.2158595323562622, 1.5848665237426758, 17.23725128173828, 2.7025299072265625, -7.0915703773498535, 1.947084665298462, -2.4528894424438477, 0.21973222494125366, -11.982887268066406, -7.713627338409424, -6.710360050201416, 3.273996353149414, 2.450415849685669, -10.435426712036133, -15.169811248779297, 1.4710720777511597, 5.381129264831543, -13.897320747375488, 0.16403567790985107, 4.703989028930664, 0.24695074558258057, -6.634984493255615, 3.289595603942871, 7.922067165374756, 5.942755699157715, 4.1284308433532715, -2.880765914916992, 4.110656261444092, 9.472119331359863, -4.422139644622803], [3.491818904876709, 2.084003210067749, 1.9676671028137207, 4.055298805236816, 16.444149017333984, 8.372596740722656, 0.962828516960144, 2.401013135910034, 0.43691134452819824, 7.708620071411133, 3.7197985649108887, -17.210296630859375, -0.4462251663208008, 10.266937255859375, -8.927871704101562, -6.123968124389648, 2.1256625652313232, 20.21939468383789, -6.120423793792725, 8.419731140136719, 9.965578079223633, -5.631363868713379, -2.4616236686706543, -5.647159099578857, 0.2826662063598633, -1.5870728492736816, 4.883028984069824, -8.052053451538086, -4.833789825439453, 10.196228981018066, 4.9992241859436035, 2.9965224266052246], [-4.463319778442383, 6.137567520141602, -8.025049209594727, -13.323193550109863, 6.082911491394043, -3.903611898422241, -0.32703638076782227, 0.19937759637832642, 2.5802688598632812, 23.54328155517578, -1.2946953773498535, 3.663849353790283, -1.7890653610229492, -0.12128853797912598, 4.734301567077637, -15.833888053894043, 0.6084058284759521, 2.900857925415039, 5.025641441345215, 2.795337677001953, 10.789566993713379, 17.20792579650879, -3.6088578701019287, -8.48608112335205, 6.662039279937744, 8.951871871948242, -19.426849365234375, -1.360623836517334, 5.241447925567627, -1.1835875511169434, 6.109160423278809, -8.562662124633789], [8.266213417053223, 3.478902578353882, -2.363382577896118, -3.6810388565063477, -6.110759735107422, -5.284949779510498, -0.9511590600013733, -1.9330183267593384, 4.695847511291504, -1.6060138940811157, -9.824292182922363, -4.128520488739014, -2.6962878704071045, -0.5642800331115723, 2.955155372619629, -0.987738311290741, -4.979460716247559, -4.598505973815918, -6.3853678703308105, -2.324985980987549, -1.6763101816177368, 1.581972360610962, 0.2671719491481781, 2.0709354877471924, -6.43543004989624, -3.0621399879455566, 3.2392578125, 2.287698984146118, -5.088494300842285, -5.273508071899414, 0.8196598291397095, 5.088274955749512], [5.202020645141602, 5.902146339416504, -0.3613920211791992, 10.120363235473633, -3.90520977973938, -1.960785150527954, -2.6053812503814697, -2.2076005935668945, 0.6654102206230164, 6.140463829040527, 14.95578384399414, 14.44481372833252, 1.8289799690246582, 2.863769054412842, 5.109264373779297, 3.658849000930786, 7.972231864929199, -11.866519927978516, 6.670925140380859, 12.424307823181152, 0.8428417444229126, -2.947669506072998, -2.3873682022094727, 8.179389953613281, -2.182365894317627, -4.8832926750183105, -2.2990164756774902, -5.747927665710449, 2.1941213607788086, 11.483528137207031, -5.565374374389648, 6.4981842041015625], [3.3074464797973633, 3.8577194213867188, 2.92740797996521, -8.13111686706543, -6.848934650421143, -5.712302207946777, 9.099919319152832, -3.1791634559631348, 9.413440704345703, -5.682092189788818, -7.882477283477783, 0.47194308042526245, 0.5963717103004456, 0.25902897119522095, 2.2691802978515625, 0.5989971160888672, 2.6072704792022705, 0.3317359685897827, -7.111324310302734, 4.039690017700195, -9.368780136108398, 0.5067249536514282, -6.004080772399902, -6.348495960235596, 1.8450658321380615, 7.361520767211914, 12.158639907836914, -1.2065770626068115, 10.743037223815918, -6.269328594207764, 2.3726284503936768, -3.5577354431152344], [0.2757091522216797, 10.190187454223633, -2.361304759979248, 8.089600563049316, 1.1950619220733643, -11.304363250732422, 13.026204109191895, 17.059799194335938, 5.866347312927246, -7.600598335266113, 0.7527893781661987, 0.2883707284927368, 9.585251808166504, 4.147726058959961, 10.390975952148438, -9.138786315917969, -4.249985218048096, -5.203943252563477, -0.8876101970672607, -9.969254493713379, -9.38776969909668, 4.4116106033325195, -0.34421730041503906, -5.842106342315674, 4.0737080574035645, 1.2777252197265625, -9.41495132446289, 5.675837516784668, 1.8729764223098755, 5.517886638641357, 0.5078773498535156, -11.035323143005371], [6.6991167068481445, 1.8696844577789307, 8.7545804977417, 9.248510360717773, -3.2165799140930176, -3.7912867069244385, -8.185104370117188, 6.555481433868408, 0.8912856578826904, 2.178903818130493, -0.033458709716796875, -2.3177361488342285, -5.847952365875244, 3.2681212425231934, -4.602447986602783, -0.7809256315231323, 9.01050090789795, -1.6433161497116089, -8.79333782196045, 9.431958198547363, -7.668882369995117, 0.01198577880859375, 9.812887191772461, 3.740551710128784, 3.199842691421509, 1.7563155889511108, -1.2549471855163574, 0.324240118265152, 3.3615782260894775, 22.111614227294922, 7.095783710479736, 8.910932540893555], [14.418896675109863, 0.06664055585861206, -1.5997856855392456, 8.125386238098145, -3.8137993812561035, 0.639371931552887, 4.5626068115234375, -1.157914400100708, -8.363988876342773, -4.955262660980225, -4.731667518615723, -0.37431633472442627, -7.143039703369141, -7.1355390548706055, 5.22233772277832, 5.296075820922852, -5.464937686920166, 0.9314618110656738, 13.788944244384766, -4.572694778442383, -0.6805236339569092, -4.250083923339844, 19.23163414001465, 0.7605127692222595, -9.358407020568848, 5.755695343017578, 6.185346603393555, 3.827686309814453, -6.0015869140625, -6.199367046356201, 9.054217338562012, -7.40265417098999], [1.9891860485076904, -6.314565181732178, -13.213421821594238, 10.064443588256836, 2.207740545272827, 7.975974082946777, -4.503752708435059, 12.92137622833252, -4.601823806762695, 0.1252685785293579, 0.03687921166419983, 6.854736328125, -3.1399378776550293, 7.692061424255371, -0.5405200123786926, 6.675504207611084, 3.6914279460906982, 13.714900970458984, 2.1243128776550293, -0.06897342205047607, -6.251442909240723, 1.2976388931274414, -4.468625545501709, 7.805362224578857, -1.3885438442230225, -1.0935794115066528, 5.325346946716309, -6.161396026611328, 5.041455268859863, 12.183686256408691, -4.1443376541137695, 6.950790882110596], [-10.466452598571777, -1.0090144872665405, 4.613480091094971, -1.8768620491027832, -4.92769193649292, -9.415328025817871, 6.9766082763671875, -4.108838081359863, -0.42106661200523376, -2.7382051944732666, -11.688141822814941, 0.027873635292053223, 12.016242980957031, -2.260974884033203, -1.1337114572525024, -0.5801178216934204, 5.8813934326171875, -4.517026424407959, -14.380485534667969, -3.9686036109924316, -3.6000659465789795, -3.2477951049804688, -10.667184829711914, -4.989943504333496, -5.837889671325684, 5.372507095336914, -4.580691337585449, -2.105292320251465, 0.1421375572681427, 10.540739059448242, 4.501462936401367, -0.7429224848747253], [7.39045524597168, 5.91629695892334, 8.122879981994629, 18.242979049682617, -5.771488666534424, -9.744059562683105, 0.9778956174850464, 14.610330581665039, -6.078333854675293, -2.1228556632995605, 4.3391876220703125, -5.524580955505371, -1.8058991432189941, 7.9952545166015625, 1.6104702949523926, -8.2831449508667, -4.714259147644043, -8.546378135681152, -10.593160629272461, -4.3806471824646, -8.777922630310059, -3.055601119995117, 6.905520915985107, -9.435531616210938, -1.358579397201538, -2.546861171722412, -1.3535913228988647, 5.918336868286133, -4.482330799102783, 7.37462854385376, 2.757258892059326, -2.24558687210083], [-0.3396087884902954, 2.3061280250549316, 1.05879807472229, 2.770883798599243, 9.214550018310547, -1.199286937713623, -1.6490747928619385, 5.219968795776367, 0.7542375922203064, -5.286343574523926, -2.954389810562134, -0.3512842655181885, 6.409749984741211, -1.6282129287719727, 6.299642562866211, 9.569684982299805, 6.430513858795166, 2.648616313934326, -1.8733255863189697, -6.334866523742676, -1.9336439371109009, -2.334078073501587, -1.180456280708313, -3.288846254348755, 10.882683753967285, 0.7647891044616699, 6.858766555786133, 6.722436428070068, 6.102039337158203, 6.475946426391602, -0.2843031883239746, -1.1225038766860962], [-3.3681774139404297, -3.098945379257202, 1.3631404638290405, 5.384585380554199, 1.3646564483642578, -6.420278072357178, -2.945960760116577, 1.421766996383667, -0.6831467151641846, -11.080753326416016, -2.229135036468506, -10.154975891113281, 4.196877479553223, 9.507107734680176, 3.088998794555664, 4.330608367919922, 8.357532501220703, -8.587319374084473, 1.533194661140442, -4.89637565612793, 0.6825000047683716, -1.7622780799865723, -11.850761413574219, 19.8590030670166, 1.1781738996505737, 0.828088641166687, -1.9330188035964966, -2.6180672645568848, -2.7189712524414062, 14.061739921569824, 4.644027233123779, 19.41519546508789], [8.384989738464355, 14.297104835510254, -0.8512135744094849, -1.856964111328125, -2.710594892501831, 0.1849285513162613, 6.242806434631348, -5.67767333984375, -3.815704584121704, -0.7541888356208801, -5.971986293792725, -1.1797029972076416, 0.6027321219444275, -11.069168090820312, -20.897228240966797, -2.4155683517456055, -2.5624942779541016, 4.4576416015625, 9.121615409851074, -6.866217613220215, 10.228816986083984, -7.532707691192627, 10.595732688903809, -2.8975226879119873, 8.725822448730469, 11.235760688781738, -2.5304269790649414, 4.6834917068481445, -2.0241494178771973, 7.551450729370117, 8.93980884552002, -2.123549699783325], [2.8084096908569336, 5.742249011993408, -5.574118614196777, -1.8343976736068726, 0.27077925205230713, 19.248579025268555, -7.785497188568115, 3.0959739685058594, 9.482267379760742, 5.538668155670166, 4.421805381774902, -3.0673060417175293, 6.350257873535156, -13.677445411682129, -3.30155611038208, -0.46789389848709106, -12.915922164916992, -6.5977582931518555, -0.2777950167655945, -0.08152502775192261, 0.3915413022041321, -4.252043724060059, 2.4754679203033447, -7.858132362365723, 4.956254005432129, 9.468894958496094, 6.011539459228516, -0.7775444984436035, 0.9274137616157532, -4.158837795257568, -5.011577129364014, -1.553143858909607], [-4.105770587921143, 1.9342442750930786, -4.985607624053955, -11.428487777709961, 2.5691967010498047, 13.623909950256348, -9.695212364196777, -8.176891326904297, 10.58612060546875, -8.553604125976562, -13.052905082702637, 0.2809518575668335, 3.676685333251953, 4.629276752471924, 1.9092001914978027, 8.417069435119629, 15.813629150390625, 9.652276992797852, 0.016970396041870117, 4.092045783996582, 17.152334213256836, -0.32299232482910156, 1.4018511772155762, 13.317699432373047, -8.233842849731445, -4.995221138000488, 1.6162348985671997, -8.550634384155273, -5.317446708679199, -11.797139167785645, -2.820009231567383, 17.931804656982422], [7.762674331665039, 2.7285990715026855, -8.673847198486328, 0.06933009624481201, 2.138570785522461, -5.998143196105957, -1.3804335594177246, 1.6611888408660889, 6.9597601890563965, 8.956527709960938, 2.390094041824341, 19.726665496826172, -1.6548995971679688, -6.027478218078613, 10.091670989990234, -2.0836029052734375, -2.492155075073242, 3.4551353454589844, 1.5678848028182983, 6.892557621002197, 4.347962856292725, 16.51576042175293, 6.7173357009887695, -10.464279174804688, 1.8089563846588135, 2.2125234603881836, 13.700278282165527, 0.6252524852752686, 6.196326732635498, -0.6780374050140381, 1.8430041074752808, -8.051956176757812], [-3.1234323978424072, 0.615286648273468, -4.75563383102417, 0.5427011251449585, -3.734797716140747, 11.41097354888916, -5.159219264984131, -4.809199333190918, -2.2725765705108643, 11.477374076843262, 9.117025375366211, 11.243748664855957, 7.406777381896973, -2.3900134563446045, 5.229870796203613, 13.302521705627441, 5.9665632247924805, 2.214823007583618, -9.154821395874023, 12.298617362976074, 6.709432601928711, -1.153282642364502, 6.752188682556152, -6.327145576477051, -3.270108461380005, -1.6114176511764526, -0.16763770580291748, -4.656411647796631, -1.607354998588562, -1.6593830585479736, -8.122562408447266, -3.3146579265594482], [3.0607686042785645, -10.47203254699707, -10.527461051940918, -13.63345718383789, 1.6554946899414062, -3.1040396690368652, 3.2202298641204834, 2.0575358867645264, -8.37018871307373, -1.1284339427947998, -2.8886659145355225, 20.041061401367188, -5.1045989990234375, -0.9054964780807495, 8.415207862854004, 1.8421134948730469, 5.154263019561768, 5.645016193389893, -1.3312925100326538, 5.112348556518555, -12.56261157989502, 9.323149681091309, -4.234724521636963, -6.809056758880615, 2.209704875946045, 14.289064407348633, -5.45778751373291, -6.731382369995117, 13.715362548828125, 5.222415924072266, 0.7179542183876038, -4.907623291015625], [1.4400160312652588, 0.732787013053894, -10.637920379638672, -4.748771667480469, -5.210298538208008, -10.917080879211426, -0.7566601037979126, -8.360002517700195, -2.0241456031799316, -11.080492973327637, -12.11868953704834, 4.216872215270996, -3.4058923721313477, -0.5921193957328796, 9.557909965515137, -14.466276168823242, 1.6831026077270508, -2.081242561340332, 6.241054534912109, -7.9388556480407715, -11.516891479492188, 2.781202793121338, -14.927407264709473, 1.427583932876587, 0.45560991764068604, 1.3557660579681396, -10.63218879699707, 2.7340750694274902, 4.605287551879883, -14.739306449890137, 9.779937744140625, -2.4070494174957275], [-1.4002723693847656, -18.152067184448242, 3.1372296810150146, -8.817411422729492, -2.4454197883605957, 0.9584373235702515, 4.807121753692627, 6.372288227081299, 8.869357109069824, -7.275496959686279, 8.826794624328613, -0.4893306493759155, -6.72680139541626, -5.17962646484375, -1.758143424987793, 12.345433235168457, -8.867204666137695, -2.527676582336426, 3.068918228149414, 1.013893723487854, -8.262449264526367, 2.1250009536743164, -3.4602210521698, 5.280257701873779, -2.0220046043395996, 3.123516798019409, 17.877042770385742, -0.8937610387802124, -8.612143516540527, -4.431396484375, -5.609564304351807, 1.1732066869735718], [3.138228416442871, 2.6331942081451416, -2.374919891357422, -9.967765808105469, -6.546349048614502, -0.13116608560085297, 5.8025712966918945, -7.45003604888916, -1.5283865928649902, 2.4123806953430176, 4.853160381317139, 6.178127765655518, -8.304804801940918, -4.296175003051758, -3.467434883117676, 9.390320777893066, -5.049081325531006, -2.2055506706237793, 8.788002014160156, -3.6792895793914795, 5.686054706573486, -2.239131212234497, -0.8113863468170166, -1.9825475215911865, -2.219712972640991, 7.940346717834473, 2.8799092769622803, 0.8699887990951538, 9.932751655578613, -10.621397972106934, -6.967278480529785, 3.008063793182373], [3.8415215015411377, 11.107589721679688, 7.128352642059326, 7.532066345214844, -1.4816709756851196, 3.5755796432495117, 7.70456600189209, -9.228796005249023, -2.5396180152893066, 7.308856964111328, -1.3570830821990967, -13.39881420135498, 4.026447772979736, 0.8236533999443054, -0.08360284566879272, -15.12436580657959, 2.5012571811676025, 8.703678131103516, 10.716009140014648, -2.9986038208007812, 10.200053215026855, 3.6462316513061523, 7.481165409088135, -1.899173378944397, 0.14839601516723633, 2.4770193099975586, -6.978428363800049, 3.0507893562316895, -4.80810546875, -6.390602111816406, 7.149423599243164, -14.506474494934082], [-7.123664855957031, 11.242500305175781, 0.34859514236450195, -7.993601322174072, -0.6195495128631592, -0.1922738254070282, -4.840623378753662, 3.5468945503234863, 7.71168327331543, -0.7748172283172607, 4.837527751922607, 4.063384056091309, 15.81214714050293, 5.5717620849609375, -9.73097038269043, 11.37861156463623, 14.013971328735352, -1.9872794151306152, -14.881044387817383, -1.6856662034988403, 7.159295558929443, 2.6900689601898193, -8.051931381225586, 1.3066906929016113, 0.14539575576782227, -9.772232055664062, 2.6968321800231934, -9.698087692260742, -1.2439550161361694, -9.401761054992676, -7.6199846267700195, 0.983349621295929], [1.7651581764221191, 0.07861161231994629, 14.685529708862305, -7.432356834411621, 4.660836219787598, 15.437844276428223, 0.9200353622436523, -10.011347770690918, 4.385721683502197, 14.63381290435791, -14.420344352722168, -17.644088745117188, 3.5454046726226807, -5.0883097648620605, -6.733248233795166, -10.707084655761719, -15.111861228942871, 5.760441303253174, -6.2542524337768555, -4.503610134124756, 13.68043041229248, -3.295996904373169, 8.033283233642578, -9.250454902648926, -0.26661109924316406, 7.603117942810059, 3.56807279586792, -1.68480384349823, 2.148143768310547, 4.545334815979004, 7.404476642608643, -12.304411888122559], [-2.0264291763305664, 0.3627438545227051, 5.472338676452637, -13.901164054870605, -0.8502993583679199, -0.5668337345123291, -6.0665388107299805, 6.851467132568359, 4.310727119445801, 12.917590141296387, -1.3043487071990967, -2.489053249359131, 13.684322357177734, 3.164445400238037, 1.277552843093872, -18.185754776000977, 7.9356794357299805, 5.6965179443359375, -15.044720649719238, 0.8853092193603516, -1.0936205387115479, 8.223896980285645, -10.772139549255371, -8.915102005004883, 11.295862197875977, 5.110322952270508, -19.27755355834961, -12.44455337524414, 2.897681713104248, 5.130974292755127, 0.11882239580154419, 1.7912195920944214], [5.605973243713379, 1.331519603729248, 0.011032938957214355, -10.414969444274902, 6.043752193450928, 12.41592025756836, 4.84418249130249, 3.678800106048584, 2.648104190826416, 3.617042064666748, -4.77022647857666, 2.925015449523926, -10.072990417480469, -0.625801146030426, -3.9115304946899414, -4.952666759490967, 0.5718855857849121, 6.61590576171875, 5.657486915588379, 1.0631431341171265, 9.863870620727539, 4.967236042022705, 1.6988484859466553, -0.0819384902715683, 4.994940280914307, 9.98106575012207, 12.550721168518066, -0.9545066356658936, -0.6377196311950684, -4.576788425445557, 5.143705368041992, -3.2734334468841553]], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(tiled_matmul, json.loads(_TESTS), 'cuda', *[0.0001, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")